## Phase 0 — Already Done

- data collection
  - NERRS (1995–2025, most regions minus hudson river and 2x great lakes, all stations)
    - originally 15min resolution
    - mostly intact water conditions
    - very sparse nutrient data
      - nutrient data extended ("as-of" forward-fill, 7-day cap)
      - this just in case a record at 945am was skipped over by picking the 1000am record instead to collate at 1hr resolution
    - entirely blank meteorlogical data?
  - ERA5 atmospherics
    - get that atmospheric data back
    - 1hr resolution
    - 0.25 degree resolution
      - not a perfect match to each station in nerrs
      - median centroid was calculated for all stations in a region
      - though most times that centroid ended up nearby but over land (so switched to skin temp instead of water surface temp)
- data cleaning and collation
- split into two datasets
  - full (with nutrients, less complete timeline)
  - and core (without nutrients, more complete)
  - script can be rerun for 1hr resolution
  - but 12hr resolution is included in this repo (zipped)
- various metrics analysis, exploration

## Logic

### Modular Chain
- Model A: ERA5 drivers -> water_temp
- Model B: ERA5 drivers + predicted water_temp -> water properties (salinity, oxygen, pH, etc.)
- Model C: ERA5 drivers + predicted water state -> nutrients (only where nutrient data exists)
  - C can be sacrificed if we run short on time, but it would be a shame

### Why?
- Logical process, re: forcing -> state -> chemistry (deltas from local mean)
- The super sparse nutrient data, if we can keep it, won't impact this as badly as it would one gigantic model
- Easier to read
- Data 'leakage' comes up a lot in reading about this, and this should help reduce it

### To consider...
- When training model B, feed **out-of-fold** predictions (deltas, etc, rather than raw original)
- Then again, if we have time, from B (state) to C (nutrients)

In [ ]:
# first... dependencies
import numpy as np                  # for arrays and math
import pandas as pd                 # for dataframes and csv I/O
import matplotlib.pyplot as plt     # basic visualizations
import seaborn as sns               # for quick readable charts
from sklearn.cluster import KMeans      # simple clustering
from sklearn.decomposition import PCA   # quick 2d display of clusters
from sklearn.preprocessing import StandardScaler  # normalize features
from sklearn.metrics import silhouette_score  # cluster quality check

# keep charts easy to read
sns.set_theme( style = 'whitegrid' )

# the files (thee have been collated and cleaned already)
res = 1 # hours - alternatives 4 and 12

# https://www.youtube.com/watch?v=_W7K79FjI58
# mandatory listening while working on this project

In [ ]:
water = pd.read_csv( f'../data/{res}hr/t4d.{res}hr.water.history.csv' )

water[ 'region' ] = water[ 'region' ].astype( str ).str.strip( ).str.lower( )
water[ 'station' ] = water[ 'station' ].astype( str ).str.strip( ).str.lower( )

# HEE is too sparse for this analysis, so drop it globally here
water = water.loc[ water[ 'region' ] != 'hee' ].copy( )

In [ ]:
# lightweight station and region lookup for labels only
station_lookup = pd.read_csv( '../data/reference/nerrs_station_index.csv' )

station_lookup[ 'region' ] = station_lookup[ 'region_code' ].astype( str ).str.strip( ).str.lower( )
station_lookup[ 'station_full' ] = station_lookup[ 'station' ].astype( str ).str.strip( ).str.lower( )
station_lookup[ 'station' ] = station_lookup[ 'station_full' ].str[ -2: ]

station_lookup = station_lookup[ [ 
    'region',
    'station',
    'station_full',
    'region_name',
    'station_name',
    'latitude',
    'longitude',
    'in_t4d_1hr_water_history',
] ].drop_duplicates( subset = [ 'region', 'station' ] )

region_name_lookup = ( 
    station_lookup
    .dropna( subset = [ 'region_name' ] )
    .drop_duplicates( subset = [ 'region' ] )
    .set_index( 'region' )[ 'region_name' ]
    .to_dict( )
)

station_name_lookup = ( 
    station_lookup
    .dropna( subset = [ 'station_name' ] )
    .set_index( [ 'region', 'station' ] )[ 'station_name' ]
    .to_dict( )
)


def station_label( region, station ):
    region_key = str( region ).strip( ).lower( )
    station_key = str( station ).strip( ).lower( )[ -2: ]

    name = station_name_lookup.get( ( region_key, station_key ) )

    if pd.isna( name ) or name is None or str( name ).strip( ) == '':
        return station_key

    return f'{station_key} - {name}'

lookup_missing = ( 
    water[ [ 'region', 'station' ] ]
    .drop_duplicates( )
    .merge( station_lookup[ [ 'region', 'station', 'station_name' ] ], on = [ 'region', 'station' ], how = 'left' )
    [ 'station_name' ]
    .isna( )
    .sum( )
)

print( f'lookup missing station names: {int( lookup_missing )}' )

## Phase 1 — Characterization & Classification
Goal: understand what we have before modeling anything

- 1.1 compute per-station summary statistics over a defined baseline period (suggest 1995–2005)
  - mean annual water temp, seasonal amplitude, mean salinity, mean DO
- 1.2 cluster stations in (mean temp × seasonal amplitude) space 
  - k-means, try k=3 or 4, use elbow/silhouette to let the data suggest the right number of groups
  - example in nutrient analysis work
- 1.3 run a parallel k-means pass on atmospheric/climate forcing signatures
  - air temp, wind, solar radiation, precipitation (baseline means + seasonal amplitudes)
- 1.4 enrich clusters with any available station metadata (estuary type, distance from mouth, watershed area)
  - confirm clusters are physically meaningful, not just statistical artifacts
- 1.5 assign each station a baseline regime label
  - see if kmeans self-identify... 
  - otherwise probably get a rolling means (temp) per station for 1995-2000 to classify
- 1.6 visualize stations on a map colored by regime
  - sanity check that PR/HI are warm, alaska/maine are cold, etc.
- 1.7 document baseline period statistics per regime as a reference table
  - this will be needed for the paper/poster later, too


In [ ]:
# lets make this more readable
water = water.rename( columns = { 
    'w_temp_c': 'water_temp',
    'w_sal_psu': 'salinity',
    'w_do_mg_l': 'oxygen',
    'w_do_pct': 'oxy_saturation',
    'depth_m': 'depth',
    'w_ph': 'ph',
    'm_wind_ms': 'wind_speed',
    'm_ssrd_kwh_m2': 'solar_radiation',
    'm_precip_mmh': 'precipitation',
    'm_temp_c': 'air_temp'
} )

# 1.0 - a description
water.describe( ).round( 3 ).T

In [ ]:
# 1.1 - station character baseline (first valid years, not fixed 1995-2000)
# this keeps the idea simple: each station gets its own early baseline window

base_all = water.copy( )
base_all[ 'datetime' ] = pd.to_datetime( base_all[ 'datetime' ], errors = 'coerce' )

base_all = base_all.loc[ 
    :,
    [ 
        'region',
        'station',
        'datetime',
        'water_temp',
        'salinity',
        'oxygen',
        'oxy_saturation',
        'ph',
        'depth',
    ],
].dropna( subset = [ 'datetime' ] )

base_all[ 'year' ] = base_all[ 'datetime' ].dt.year

In [ ]:
#base_all.describe( ).round( 3 ).T

In [ ]:
# annual coverage check so thin years do not define the baseline
year_obs = ( 
    base_all
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( n_obs = ( 'datetime', 'size' ) )
)

# leap years and all that jazz
year_obs[ 'expected_obs' ] = np.where( 
    year_obs[ 'year' ] % 4 == 0,
    366 * 24,
    365 * 24,
)

In [ ]:
year_obs.describe( ).round( 2 ).T

In [ ]:
# try to identify years with enough coverage to be considered valid for defining the baseline
year_obs[ 'coverage_frac' ] = year_obs[ 'n_obs' ] / year_obs[ 'expected_obs' ]
year_obs[ 'year_is_valid' ] = year_obs[ 'coverage_frac' ] >= 0.70 

valid_years = year_obs.loc[ year_obs[ 'year_is_valid' ] ].copy( )
valid_years = valid_years.sort_values( [ 'region', 'station', 'year' ] ).reset_index( drop = True )
valid_years[ 'valid_year_rank' ] = valid_years.groupby( [ 'region', 'station' ] ).cumcount( ) + 1

# first 5 valid years per station
# turned out too many didn't even start operating between 95 and 00
baseline_years = valid_years.loc[ valid_years[ 'valid_year_rank' ] <= 5 ].copy( )

baseline_window = ( 
    baseline_years
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        baseline_start_year = ( 'year', 'min' ),
        baseline_end_year = ( 'year', 'max' ),
        n_valid_years = ( 'year', 'size' ),
        mean_year_coverage = ( 'coverage_frac', 'mean' ),
    )
)

baseline_window[ 'is_partial_baseline' ] = baseline_window[ 'n_valid_years' ] < 5

In [ ]:
baseline_window.sample( 15 ).round( 3 ).T

In [ ]:
# keep stations with at least 3 valid years
eligible_stations = baseline_window.loc[ 
    baseline_window[ 'n_valid_years' ] >= 3,
    [ 'region', 'station' ],
]

# pull only rows from each station's selected baseline years
base = base_all.merge( 
    baseline_years[ [ 'region', 'station', 'year' ] ],
    on = [ 'region', 'station', 'year' ],
    how = 'inner',
)

# then keep only stations that passed the >=3-year minimum
base = base.merge( eligible_stations, on = [ 'region', 'station' ], how = 'inner' )

In [ ]:
#base.describe( ).round( 3 ).T

In [ ]:
# okay, now build station character features from the selected baseline window
base[ 'month' ] = base[ 'datetime' ].dt.month
# get mean of each year per station, then the mean of those five years per station
# yearly means first ( equal-year weighting )
annual = ( 
    base
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        water_temp_ymean = ( 'water_temp', 'mean' ),
        salinity_ymean = ( 'salinity', 'mean' ),
        oxygen_ymean = ( 'oxygen', 'mean' ),
        saturation_ymean = ( 'oxy_saturation', 'mean' ),
        ph_ymean = ( 'ph', 'mean' ),
        depth_ymean = ( 'depth', 'mean' ),
    )
)

# each station gets its own, yo
annual_means = ( 
    annual
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        mean_annual_water_temp = ( 'water_temp_ymean', 'mean' ),
        mean_annual_salinity = ( 'salinity_ymean', 'mean' ),
        mean_annual_oxygen = ( 'oxygen_ymean', 'mean' ),
        mean_annual_saturation = ( 'saturation_ymean', 'mean' ),
        mean_annual_ph = ( 'ph_ymean', 'mean' ),
        mean_annual_depth = ( 'depth_ymean', 'mean' ),
        n_years_water_temp = ( 'water_temp_ymean', 'count' ),
        n_years_salinity = ( 'salinity_ymean', 'count' ),
        n_years_oxygen = ( 'oxygen_ymean', 'count' ),
        n_years_saturation = ( 'saturation_ymean', 'count' ),
        n_years_ph = ( 'ph_ymean', 'count' ),
        n_years_depth = ( 'depth_ymean', 'count' ),
    )
)

In [ ]:
annual_means.describe( ).round( 2 ).T

In [ ]:
# seasonal amplitudes from monthly climatology
monthly = ( 
    base
    .groupby( [ 'region', 'station', 'month' ], as_index = False )
    .agg( 
        water_temp_mmean = ( 'water_temp', 'mean' ),
        salinity_mmean = ( 'salinity', 'mean' ),
        oxygen_mmean = ( 'oxygen', 'mean' ),
        saturation_mmean = ( 'oxy_saturation', 'mean' ),
        ph_mmean = ( 'ph', 'mean' ),
        depth_mmean = ( 'depth', 'mean' ),
    )
)

In [ ]:
monthly.describe( ).round( 2 ).T

In [ ]:
monthly.sample( 10 ).round( 3 ).T

In [ ]:
# building to a new feature, the 'swing' the mean swing of seasonal properties
seasonal_amp = ( 
    monthly
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        seasonal_amp_water_temp = ( 'water_temp_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_salinity = ( 'salinity_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_oxygen = ( 'oxygen_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_saturation = ( 'saturation_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_ph = ( 'ph_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_depth = ( 'depth_mmean', lambda s: s.max( ) - s.min( ) ),
    )
)

In [ ]:
seasonal_amp.describe( ).round( 2 ).T

In [ ]:
seasonal_amp.sample( 10 ).round( 3 ).T

In [ ]:
# depth summary ( useful if sensor environment differs by station )
# let's see if the recorders are variable depth per station, or predictable?
depth_stats = ( 
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        median_depth = ( 'depth', 'median' ),
        p10_depth = ( 'depth', lambda s: s.quantile( 0.10 ) ),
        p90_depth = ( 'depth', lambda s: s.quantile( 0.90 ) ),
    )
)
depth_stats[ 'iqr_depth' ] = depth_stats[ 'p90_depth' ] - depth_stats[ 'p10_depth' ]

In [ ]:
depth_stats.describe( ).round( 2 ).T

In [ ]:
coverage = ( 
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_obs_total = ( 'datetime', 'size' ),
        n_years_present = ( 'year', 'nunique' ),
    )
)

# lets snap these together into a baseline ...
station_baseline = ( 
    annual_means
    .merge( seasonal_amp, on = [ 'region', 'station' ], how = 'left' )
    .merge( depth_stats, on = [ 'region', 'station' ], how = 'left' )
    .merge( coverage, on = [ 'region', 'station' ], how = 'left' )
    .merge( 
        baseline_window[ [ 
            'region',
            'station',
            'baseline_start_year',
            'baseline_end_year',
            'n_valid_years',
            'mean_year_coverage',
            'is_partial_baseline',
        ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

# placeholder flag in case we later add nearest-neighbor feature fallback
station_baseline[ 'character_imputed' ] = False
station_baseline_display = ( 
    station_baseline
    .merge( 
        station_lookup[ [ 'region', 'station', 'region_name', 'station_name', 'latitude', 'longitude' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

In [ ]:
station_baseline.describe( ).round( 2 ).T

In [ ]:
# quick baseline summary
n_total_stations = baseline_window[ [ 'region', 'station' ] ].drop_duplicates( ).shape[ 0 ]
n_eligible_stations = eligible_stations.shape[ 0 ]
n_full_five_year = int( ( baseline_window[ 'n_valid_years' ] >= 5 ).sum( ) )

print( f'stations with >=3 valid baseline years: {n_eligible_stations} of {n_total_stations}' )
print( f'stations with full 5-year baseline: {n_full_five_year} of {n_total_stations}' )
del ( 
    n_total_stations,
    n_eligible_stations,
    n_full_five_year
)

#eligible_stations.sample( 10 ).round( 3 ).T

In [ ]:
# tall small-multiples: one subplot per region, one line per station

water[ 'datetime' ] = pd.to_datetime( water[ 'datetime' ], errors = 'coerce' )
water[ 'year' ] = water[ 'datetime' ].dt.year

annual_station = ( 
    water
    .groupby( [ 'region', 'station', 'year' ], as_index = False )[ 'water_temp' ]
    .mean( )
)

regions = sorted( annual_station[ 'region' ].dropna( ).unique( ) )
n_regions = len( regions ) # need for number o'plots

fig, axes = plt.subplots( 
    n_regions,
    1,
    figsize = ( 16, 2 * n_regions ),
    sharex = True,
    constrained_layout = True
)

for ax, region in zip( axes, regions ):
    sub = annual_station.loc[ annual_station[ 'region' ] == region ].sort_values( [ 'station', 'year' ] )

    for station, g in sub.groupby( 'station' ):
        ax.plot( 
            g[ 'year' ],
            g[ 'water_temp' ],
            linewidth = 1.5,
            alpha = 0.85,
            label = station_label( region, station )
        )

    region_title = region_name_lookup.get( region, region )
    ax.set_title( f'{region.upper( )} ( {region_title} ): Mean Annual Water Temp by Station' )
    ax.set_ylabel( 'Temp ( C )' )

    # keep legends readable
    n_stations = sub[ 'station' ].nunique( )

    ax.legend( 
        ncol = 1,
        fontsize = 7,
        frameon = False,
        loc = 'upper left',
        bbox_to_anchor = ( 1.01, 1.0 ),
        borderaxespad = 0
    )

axes[ -1 ].set_xlabel( 'Year' )
plt.show( )

del ( # lil bit of cleanup, for ememory
    annual_station,
    regions,
    n_regions
)

### 1.2 - Domain-Driven KMeans (Temperature, Salinity, Oxygen, Depth)

single slim clustering pass with interpretable station-character features


In [ ]:
# single, slim KMeans pass with a domain-driven feature subset
# selected features:
# - mean and seasonal amplitude for temperature
# - mean and seasonal amplitude for salinity
# - mean and seasonal amplitude for oxygen
# - mean annual depth
domain_feature_cols = [ 
    'mean_annual_water_temp',
    'seasonal_amp_water_temp',
    'mean_annual_salinity',
    'seasonal_amp_salinity',
    'mean_annual_oxygen',
    'seasonal_amp_oxygen',
    'mean_annual_depth',
]

print( 'domain features:', domain_feature_cols )

domain_input = station_baseline[ [ 'region', 'station' ] + domain_feature_cols ].copy( )

# simple missing-value strategy: region median, then global median
for col in domain_feature_cols:
    domain_input[ col ] = domain_input.groupby( 'region' )[ col ].transform( lambda s: s.fillna( s.median( ) ) )
    domain_input[ col ] = domain_input[ col ].fillna( domain_input[ col ].median( ) )

scaler_domain = StandardScaler( )
X_scaled_domain = scaler_domain.fit_transform( domain_input[ domain_feature_cols ] )

# k scan for elbow + silhouette
k_scan_rows = [ ]

for k in range( 2, 11 ):
    km_scan = KMeans( n_clusters = k, random_state = 42, n_init = 20 )
    labels_scan = km_scan.fit_predict( X_scaled_domain )

    k_scan_rows.append( { 
        'k': k,
        'inertia': float( km_scan.inertia_ ),
        'silhouette': float( silhouette_score( X_scaled_domain, labels_scan ) ),
    } )

k_scan_domain = pd.DataFrame( k_scan_rows )
k_scan_domain[ 'inertia_drop' ] = k_scan_domain[ 'inertia' ].shift( 1 ) - k_scan_domain[ 'inertia' ]
k_scan_domain[ 'inertia_drop_pct' ] = k_scan_domain[ 'inertia_drop' ] / k_scan_domain[ 'inertia' ].shift( 1 )

recommended_k = int( k_scan_domain.loc[ k_scan_domain[ 'silhouette' ].idxmax( ), 'k' ] )

# keep the final pick interpretable for section 1.2
k_min_interpretable = 3
k_max_interpretable = 6
k_scan_interpretable = k_scan_domain.loc[ k_scan_domain[ 'k' ].between( k_min_interpretable, k_max_interpretable ) ].copy( )

if k_scan_interpretable.empty:
    recommended_k_interpretable = 4

else:
    recommended_k_interpretable = int( 
        k_scan_interpretable.loc[ k_scan_interpretable[ 'silhouette' ].idxmax( ), 'k' ]
    )

print( '\ndomain-feature k scan:' )
print( k_scan_domain.round( 4 ) )
print( f'unbounded silhouette max K: { recommended_k }' )
print( f'interpretable-window K ( { k_min_interpretable }-{ k_max_interpretable } ): { recommended_k_interpretable }' )

fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )

axes[ 0 ].plot( 
    k_scan_domain[ 'k' ],
    k_scan_domain[ 'inertia' ],
    marker = 'o'
)
axes[ 0 ].set_title( 'Domain Features: K vs Inertia' )
axes[ 0 ].set_xlabel( 'K' )
axes[ 0 ].set_ylabel( 'Inertia' )

axes[ 1 ].plot( 
    k_scan_domain[ 'k' ],
    k_scan_domain[ 'silhouette' ],
    marker = 'o'
)
axes[ 1 ].set_title( 'Domain Features: K vs Silhouette Score' )
axes[ 1 ].set_xlabel( 'K' )
axes[ 1 ].set_ylabel( 'Silhouette Score' )

plt.tight_layout( )
plt.show( )


In [ ]:
# not a really DISTINCT elbow, but a clear flattening after 4 or 5 clusters, and a silhouette max at 5

# use the bounded recommendation by default; override manually if needed
k_clusters = recommended_k_interpretable
kmeans_model = KMeans( n_clusters = k_clusters, random_state = 42, n_init = 20 )
domain_input[ 'cluster' ] = kmeans_model.fit_predict( X_scaled_domain )

# keep this alias so downstream sections can still use the same naming
domain_input[ 'cluster_domain' ] = domain_input[ 'cluster' ]

domain_silhouette = float( silhouette_score( X_scaled_domain, domain_input[ 'cluster' ] ) )

print( f'\nchosen K: { k_clusters }' )
print( 'domain-feature silhouette:', round( domain_silhouette, 4 ) )

print( '\ncluster sizes:' )
print( domain_input[ 'cluster' ].value_counts( ).sort_index( ) )

cluster_profile_raw = domain_input.groupby( 'cluster' )[ domain_feature_cols ].mean( )
cluster_profile = cluster_profile_raw.round( 3 )

cluster_profile_mean = cluster_profile_raw.mean( )
cluster_profile_std = cluster_profile_raw.std( ddof = 0 ).replace( 0, np.nan )
cluster_profile_z = ( cluster_profile_raw - cluster_profile_mean ) / cluster_profile_std


def bucket_tag( value, low_label, mid_label, high_label, threshold = 0.35 ):
    if pd.isna( value ):
        return mid_label

    if value >= threshold:
        return high_label

    if value <= -threshold:
        return low_label

    return mid_label


def build_cluster_name( z_row ):
    temp_tag = bucket_tag( z_row[ 'mean_annual_water_temp' ], 'Cool', 'Temp', 'Warm' )
    sal_tag = bucket_tag( z_row[ 'mean_annual_salinity' ], 'Fresh', 'Brackish', 'Saline' )
    oxy_tag = bucket_tag( z_row[ 'mean_annual_oxygen' ], 'Low O2', 'Mid O2', 'High O2' )

    amp_tag = bucket_tag( z_row[ 'seasonal_amp_water_temp' ], 'Stable', 'Mixed', 'Seasonal' )
    depth_tag = bucket_tag( z_row[ 'mean_annual_depth' ], 'Shallow', 'Mid', 'Deep' )

    name = f'{ temp_tag } / { sal_tag } / { oxy_tag }'
    short_name = f'{ temp_tag }-{ sal_tag }'
    note = f'{ amp_tag }, { depth_tag }'

    return pd.Series( { 'cluster_name': name, 'cluster_label': short_name, 'cluster_note': note } )


cluster_name_map = cluster_profile_z.apply( build_cluster_name, axis = 1 ).reset_index( )

print( '\ncluster names:' )
print( cluster_name_map )

In [ ]:
# keep labels categorical and ordered by cluster id for readability + compact memory
cluster_order = sorted( cluster_name_map[ 'cluster' ].astype( int ).tolist( ) )
cluster_name_order = cluster_name_map.sort_values( 'cluster' )[ 'cluster_name' ].tolist( )
cluster_label_order = cluster_name_map.sort_values( 'cluster' )[ 'cluster_label' ].tolist( )

domain_input = domain_input.merge( cluster_name_map, on = 'cluster', how = 'left' )

# write cluster labels back to the station tables
for cluster_col in [ 'cluster', 'cluster_domain', 'cluster_name', 'cluster_label', 'cluster_note' ]:
    if cluster_col in station_baseline.columns:
        station_baseline = station_baseline.drop( columns = [ cluster_col ] )

station_baseline = station_baseline.merge( 
    domain_input[ [ 'region', 'station', 'cluster', 'cluster_domain', 'cluster_name', 'cluster_label', 'cluster_note' ] ],
    on = [ 'region', 'station' ],
    how = 'left',
)

station_baseline[ 'cluster' ] = pd.Categorical( 
    station_baseline[ 'cluster' ],
    categories = cluster_order,
    ordered = True,
)
station_baseline[ 'cluster_name' ] = pd.Categorical( 
    station_baseline[ 'cluster_name' ],
    categories = cluster_name_order,
    ordered = True,
)
station_baseline[ 'cluster_label' ] = pd.Categorical( 
    station_baseline[ 'cluster_label' ],
    categories = cluster_label_order,
    ordered = True,
)

for cluster_col in [ 'cluster', 'cluster_domain', 'cluster_name', 'cluster_label', 'cluster_note' ]:
    if cluster_col in station_baseline_display.columns:
        station_baseline_display = station_baseline_display.drop( columns = [ cluster_col ] )

station_baseline_display = station_baseline_display.merge( 
    domain_input[ [ 'region', 'station', 'cluster', 'cluster_domain', 'cluster_name', 'cluster_label', 'cluster_note' ] ],
    on = [ 'region', 'station' ],
    how = 'left',
)

# cluster mean feature profiles
plt.figure( figsize = ( 12, 5 ) )
sns.heatmap( cluster_profile, cmap = 'YlGnBu', annot = True, fmt = '.2f' )
plt.title( 'Cluster Mean Station-Character Features ( Domain Set )' )
plt.xlabel( 'Features' )
plt.ylabel( 'Cluster' )
plt.tight_layout( )
plt.show( )

In [ ]:
# correlation matrix for the domain feature set
feature_corr_domain = domain_input[ domain_feature_cols ].corr( )
tri_mask = np.triu( np.ones_like( feature_corr_domain, dtype = bool ), k = 1 )

plt.figure( figsize = ( 10, 8 ) )
sns.heatmap( 
    feature_corr_domain,
    mask = tri_mask,
    annot = True,
    fmt = '.2f',
    cmap = 'coolwarm',
    center = 0,
    vmin = -1,
    vmax = 1,
    square = True,
    linewidths = 0.35,
    annot_kws = { 'size': 8 },
    cbar_kws = { 'label': 'Pearson r', 'shrink': 0.85 },
)
plt.title( 'Domain Feature Correlation Matrix ( Triangle + Labels )' )
plt.tight_layout( )
plt.show( )

In [ ]:
corr_pairs_domain = ( 
    feature_corr_domain
    .where( np.triu( np.ones( feature_corr_domain.shape ), k = 1 ).astype( bool ) )
    .stack( )
    .reset_index( )
    .rename( columns = { 'level_0': 'feature_a', 'level_1': 'feature_b', 0: 'corr' } )
)

corr_pairs_domain[ 'abs_corr' ] = corr_pairs_domain[ 'corr' ].abs( )
corr_pairs_domain = corr_pairs_domain.sort_values( 'abs_corr', ascending = False )

print( '\ntop correlated domain-feature pairs:' )
corr_pairs_domain.head( 12 )

# PCA scatter of resulting clusters
pca_domain = PCA( n_components = 3 )
pcs_domain = pca_domain.fit_transform( X_scaled_domain )

pca_loadings_domain = pd.DataFrame( 
    pca_domain.components_.T,
    index = domain_feature_cols,
    columns = [ 'PC1', 'PC2', 'PC3' ],
)

# for eventual readability 
feature_phrase_map = { 
    'mean_annual_water_temp': ( 'warmer mean temp', 'cooler mean temp' ),
    'seasonal_amp_water_temp': ( 'larger temp seasonality', 'smaller temp seasonality' ),
    'mean_annual_salinity': ( 'saltier water', 'fresher water' ),
    'seasonal_amp_salinity': ( 'larger salinity swings', 'smaller salinity swings' ),
    'mean_annual_oxygen': ( 'higher mean oxygen', 'lower mean oxygen' ),
    'seasonal_amp_oxygen': ( 'larger oxygen swings', 'smaller oxygen swings' ),
    'mean_annual_depth': ( 'deeper stations', 'shallower stations' ),
}

# automate, if at all possible, interpreting the PCAs


def pc_axis_interpretation( component_name ):
    top_features = pca_loadings_domain[ component_name ].abs( ).sort_values( ascending = False ).head( 2 ).index
    text_parts = [ ]

    for feat in top_features:
        pos_text, neg_text = feature_phrase_map[ feat ]
        loading = pca_loadings_domain.loc[ feat, component_name ]
        text_parts.append( pos_text if loading >= 0 else neg_text )

    return ' + '.join( text_parts )


pc1_var_pct = round( float( pca_domain.explained_variance_ratio_[ 0 ] ) * 100, 1 )
pc2_var_pct = round( float( pca_domain.explained_variance_ratio_[ 1 ] ) * 100, 1 )
pc3_var_pct = round( float( pca_domain.explained_variance_ratio_[ 2 ] ) * 100, 1 )
domain_pca_3d_var_pct = round( pc1_var_pct + pc2_var_pct + pc3_var_pct, 1 )
pc1_label_text = pc_axis_interpretation( 'PC1' )
pc2_label_text = pc_axis_interpretation( 'PC2' )

print( f'domain PCA variance: PC1={ pc1_var_pct }%, PC2={ pc2_var_pct }%, PC3={ pc3_var_pct }%, cumulative={ domain_pca_3d_var_pct }%' )

domain_plot = domain_input[ [ 'region', 'station', 'cluster', 'cluster_name', 'cluster_label', 'cluster_note' ] ].copy( )
domain_plot[ 'pc1' ] = pcs_domain[ :, 0 ]
domain_plot[ 'pc2' ] = pcs_domain[ :, 1 ]
domain_plot[ 'pc3' ] = pcs_domain[ :, 2 ]

cluster_centers = ( 
    domain_plot
    .groupby( [ 'cluster', 'cluster_name', 'cluster_label', 'cluster_note' ], as_index = False )[ [ 'pc1', 'pc2' ] ]
    .mean( )
)

plt.figure( figsize = ( 12, 8 ) )
sns.scatterplot( 
    data = domain_plot,
    x = 'pc1',
    y = 'pc2',
    hue = 'cluster',
    palette = 'tab10',
    s = 70,
    alpha = 0.88,
)
plt.title( f'Domain KMeans Clusters in PCA Space (PC1/PC2, K = { k_clusters })' )
plt.xlabel( f'PC1 ( { pc1_var_pct }% var )' )
plt.ylabel( f'PC2 ( { pc2_var_pct }% var )' )
plt.legend( title = 'cluster', bbox_to_anchor = ( 1.03, 1 ), loc = 'upper left' )
plt.tight_layout( )
plt.show( )


In [ ]:
# interactive 3d PCA view (plotly)
# will help with understanding, but screenshots will be lacking just beacuse
import plotly.express as px

domain_plot_interactive = domain_plot.copy( )
domain_plot_interactive[ 'cluster' ] = domain_plot_interactive[ 'cluster' ].astype( str )

fig_domain_3d = px.scatter_3d( 
    domain_plot_interactive,
    x = 'pc1',
    y = 'pc2',
    z = 'pc3',
    color = 'cluster',
    hover_data = [ 'region', 'station', 'cluster_name', 'cluster_label', 'cluster_note' ],
    title = f'Domain KMeans 3D PCA (Interactive) | K = { k_clusters }',
    opacity = 0.85,
)
fig_domain_3d.update_traces( marker = dict( size = 4 ) )
fig_domain_3d.update_layout( height = 700 )
fig_domain_3d.show( )


In [ ]:
# just wanna see soem examples
station_baseline_display[ [ 
    'region',
    'station',
    'station_name',
    'cluster',
    'cluster_name',
    'cluster_label',
    'cluster_note',
    'baseline_start_year',
    'baseline_end_year',
    'n_valid_years',
] ].sort_values( [ 'cluster', 'region', 'station' ] ).sample( 5 ).T

#### PCA Interpretation (Domain Feature Set)

- **PC1** and **PC2** are weighted blends of the selected domain features
- gonna play with PC3, too, just to see if an interactive 3D space helps aplexain some ... odd overlaps
- use **explained variance ratio** to see how much structure each component captures
- use **absolute loadings** to identify which features dominate each component
- nearby points in PCA space still represent similar station character


In [ ]:
# quick PCA interpretation table for the domain feature run
pca_variance = pd.Series( 
    pca_domain.explained_variance_ratio_,
    index = [ f'PC{i+1}' for i in range( len( pca_domain.explained_variance_ratio_ ) ) ],
    name = 'explained_variance_ratio',
)

pca_loadings = pd.DataFrame( 
    pca_domain.components_.T,
    index = domain_feature_cols,
    columns = [ f'PC{i+1}' for i in range( pca_domain.n_components_ ) ],
)

print( 'explained variance ratio:' )
print( pca_variance.round( 3 ) )

pca_loadings.round( 3 )

### 1.2b - Quick DBSCAN Trial

quick side-by-side pass against the phase 1.2 k-means setup using the same domain feature space

by request

In [ ]:
# 1.2b - quick DBSCAN scan on the same standardized domain feature space
# see https://www.youtube.com/watch?v=RDZUdRSDOok
from sklearn.cluster import DBSCAN

dbscan_min_samples = 5
eps_values = np.round( np.arange( 0.60, 2.01, 0.10 ), 2 )
dbscan_eps_override = 1.50  # set None to use auto-pick from scan
dbscan_scan_rows = [ ]

for eps in eps_values:
    labels_eps = DBSCAN( eps = float( eps ), min_samples = dbscan_min_samples ).fit_predict( X_scaled_domain )
    n_clusters_eps = len( set( labels_eps ) - { -1 } )
    noise_pct_eps = float( ( labels_eps == -1 ).mean( ) * 100 )
    core_mask_eps = labels_eps != -1

    if n_clusters_eps >= 2 and core_mask_eps.sum( ) >= 3:
        sil_eps = float( silhouette_score( X_scaled_domain[ core_mask_eps ], labels_eps[ core_mask_eps ] ) )

    else:
        sil_eps = np.nan

    dbscan_scan_rows.append( { 
        'eps': float( eps ),
        'min_samples': dbscan_min_samples,
        'n_clusters': int( n_clusters_eps ),
        'noise_pct': noise_pct_eps,
        'silhouette_core_only': sil_eps,
    } )

dbscan_scan_by_eps = pd.DataFrame( dbscan_scan_rows ).sort_values( 'eps' ).reset_index( drop = True )
dbscan_scan = dbscan_scan_by_eps.sort_values( 
    [ 'silhouette_core_only', 'n_clusters', 'eps' ],
    ascending = [ False, False, True ],
)

if dbscan_eps_override is None:
    dbscan_eps = float( dbscan_scan.iloc[ 0 ][ 'eps' ] )
    dbscan_eps_source = 'auto'

else:
    dbscan_eps = float( dbscan_eps_override )
    dbscan_eps_source = 'manual override'

dbscan_model = DBSCAN( eps = dbscan_eps, min_samples = dbscan_min_samples )
domain_input[ 'cluster_dbscan' ] = dbscan_model.fit_predict( X_scaled_domain )

dbscan_n_clusters = int( len( set( domain_input[ 'cluster_dbscan' ] ) - { -1 } ) )
dbscan_noise_pct = float( ( domain_input[ 'cluster_dbscan' ] == -1 ).mean( ) * 100 )
dbscan_core_mask = domain_input[ 'cluster_dbscan' ] != -1

if dbscan_n_clusters >= 2 and dbscan_core_mask.sum( ) >= 3:
    dbscan_silhouette = float( 
        silhouette_score( 
            X_scaled_domain[ dbscan_core_mask ],
            domain_input.loc[ dbscan_core_mask, 'cluster_dbscan' ],
        )
    )

else:
    dbscan_silhouette = np.nan

comparison_rows = [ { 
    'method': 'kmeans',
    'n_clusters': int( k_clusters ),
    'noise_pct': 0.0,
    'silhouette': float( domain_silhouette ),
}, { 
    'method': 'dbscan',
    'n_clusters': dbscan_n_clusters,
    'noise_pct': dbscan_noise_pct,
    'silhouette': dbscan_silhouette,
} ]

comparison_phase_1_2b = pd.DataFrame( comparison_rows )

scan_match = dbscan_scan_by_eps.loc[ np.isclose( dbscan_scan_by_eps[ 'eps' ], dbscan_eps ) ]
if scan_match.empty:
    selected_scan_row = dbscan_scan_by_eps.iloc[ [ ( dbscan_scan_by_eps[ 'eps' ] - dbscan_eps ).abs( ).idxmin( ) ] ]

else:
    selected_scan_row = scan_match

print( f'dbscan selected eps: { dbscan_eps } ( { dbscan_eps_source }, min_samples = { dbscan_min_samples } )' )
print( 'selected eps row from scan table:' )
display( selected_scan_row )
print( 'top DBSCAN scan rows:' )
display( dbscan_scan.head( 10 ) )
print( )
print( 'quick kmeans vs dbscan comparison:' )
comparison_phase_1_2b

In [ ]:
# 1.2b - pca view with DBSCAN labels ( noise = -1 )
dbscan_plot = domain_plot[ [ 'region', 'station', 'pc1', 'pc2' ] ].copy( )
dbscan_plot[ 'cluster_dbscan' ] = domain_input[ 'cluster_dbscan' ].to_numpy( )

dbscan_clusters = sorted( cluster for cluster in dbscan_plot[ 'cluster_dbscan' ].unique( ) if cluster != -1 )
dbscan_palette = { -1: '#9e9e9e' }
for idx, cluster_id in enumerate( dbscan_clusters ):
    dbscan_palette[ int( cluster_id ) ] = sns.color_palette( 'tab10', 10 )[ idx % 10 ]

plt.figure( figsize = ( 12, 8 ) )
sns.scatterplot( 
    data = dbscan_plot,
    x = 'pc1',
    y = 'pc2',
    hue = 'cluster_dbscan',
    palette = dbscan_palette,
    s = 70,
    alpha = 0.90,
)
plt.title( f'DBSCAN Clusters in PCA Space ( eps = { dbscan_eps }, min_samples = { dbscan_min_samples } )' )
plt.xlabel( f'PC1 ( { pc1_var_pct }% var )' )
plt.ylabel( f'PC2 ( { pc2_var_pct }% var )' )
plt.legend( title = 'dbscan cluster', bbox_to_anchor = ( 1.03, 1 ), loc = 'upper left' )
plt.tight_layout( )
plt.show( )


In [ ]:
# 1.2b - quick interpretation from DBSCAN core clusters
dbscan_profile_raw = ( 
    domain_input
    .loc[ domain_input[ 'cluster_dbscan' ] != -1 ]
    .groupby( 'cluster_dbscan' )[ domain_feature_cols ]
    .mean( )
)


dbscan_profile = dbscan_profile_raw.round( 3 )
dbscan_profile_z = ( dbscan_profile_raw - cluster_profile_mean ) / cluster_profile_std
dbscan_name_map = dbscan_profile_z.apply( build_cluster_name, axis = 1 ).reset_index( )

dbscan_counts = ( 
    domain_input[ 'cluster_dbscan' ]
    .value_counts( )
    .rename_axis( 'cluster_dbscan' )
    .reset_index( name = 'n_stations' )
    .sort_values( 'cluster_dbscan' )
)

dbscan_interpretation = dbscan_name_map.merge( dbscan_counts, on = 'cluster_dbscan', how = 'left' )
print( 'DBSCAN cluster interpretation ( core clusters only ):' )
display( dbscan_interpretation.sort_values( 'cluster_dbscan' ) )

plt.figure( figsize = ( 12, 5 ) )
sns.heatmap( dbscan_profile, cmap = 'YlOrBr', annot = True, fmt = '.2f' )
plt.title( 'DBSCAN Mean Station-Character Features ( Core Clusters )' )
plt.xlabel( 'Features' )
plt.ylabel( 'DBSCAN Cluster' )
plt.tight_layout( )
plt.show( )

### 1.2c - Barebones HDBSCAN Trial

quick hierarchical density-clustering pass on the same domain feature space for comparison


In [ ]:
# 1.2c - barebones HDBSCAN fit on the same standardized domain feature space
hdbscan_min_cluster_size = 6
hdbscan_min_samples = 6
hdbscan_cluster_selection_method = 'eom'

hdbscan_backend = 'hdbscan-package'

from sklearn.cluster import HDBSCAN as SklearnHDBSCAN

hdbscan_backend = 'sklearn'
hdbscan_model = SklearnHDBSCAN( 
    min_cluster_size = hdbscan_min_cluster_size,
    min_samples = hdbscan_min_samples,
    cluster_selection_method = hdbscan_cluster_selection_method,
)

domain_input[ 'cluster_hdbscan' ] = hdbscan_model.fit_predict( X_scaled_domain )

hdbscan_n_clusters = int( len( set( domain_input[ 'cluster_hdbscan' ] ) - { -1 } ) )
hdbscan_noise_pct = float( ( domain_input[ 'cluster_hdbscan' ] == -1 ).mean( ) * 100 )
hdbscan_core_mask = domain_input[ 'cluster_hdbscan' ] != -1

if hdbscan_n_clusters >= 2 and hdbscan_core_mask.sum( ) >= 3:
    hdbscan_silhouette = float( 
        silhouette_score( 
            X_scaled_domain[ hdbscan_core_mask ],
            domain_input.loc[ hdbscan_core_mask, 'cluster_hdbscan' ],
        )
    )

else:
    hdbscan_silhouette = np.nan

comparison_phase_1_2c = pd.DataFrame( [ 
    { 
        'method': 'kmeans',
        'n_clusters': int( k_clusters ),
        'noise_pct': 0.0,
        'silhouette': float( domain_silhouette ),
    },
    { 
        'method': 'dbscan',
        'n_clusters': int( dbscan_n_clusters ),
        'noise_pct': float( dbscan_noise_pct ),
        'silhouette': float( dbscan_silhouette ) if pd.notna( dbscan_silhouette ) else np.nan,
    },
    { 
        'method': 'hdbscan',
        'n_clusters': hdbscan_n_clusters,
        'noise_pct': hdbscan_noise_pct,
        'silhouette': hdbscan_silhouette,
    },
] )

print( f'hdbscan backend: { hdbscan_backend }' )
print( f'hdbscan params: min_cluster_size = { hdbscan_min_cluster_size }, min_samples = { hdbscan_min_samples }, method = { hdbscan_cluster_selection_method }' )
print( f'hdbscan clusters: { hdbscan_n_clusters } | noise: { round( hdbscan_noise_pct, 2 ) }%' )
comparison_phase_1_2c


In [ ]:
# 1.2c - pca view with HDBSCAN labels ( noise = -1 )
hdbscan_plot = domain_plot[ [ 'region', 'station', 'pc1', 'pc2' ] ].copy( )
hdbscan_plot[ 'cluster_hdbscan' ] = domain_input[ 'cluster_hdbscan' ].to_numpy( )

hdbscan_clusters = sorted( cluster for cluster in hdbscan_plot[ 'cluster_hdbscan' ].unique( ) if cluster != -1 )
hdbscan_palette = { -1: '#9e9e9e' }
for idx, cluster_id in enumerate( hdbscan_clusters ):
    hdbscan_palette[ int( cluster_id ) ] = sns.color_palette( 'tab10', 10 )[ idx % 10 ]

plt.figure( figsize = ( 12, 8 ) )
sns.scatterplot( 
    data = hdbscan_plot,
    x = 'pc1',
    y = 'pc2',
    hue = 'cluster_hdbscan',
    palette = hdbscan_palette,
    s = 70,
    alpha = 0.90,
)
plt.title( 'HDBSCAN Clusters in PCA Space' )
plt.xlabel( f'PC1 ( { pc1_var_pct }% var )' )
plt.ylabel( f'PC2 ( { pc2_var_pct }% var )' )
plt.legend( title = 'hdbscan cluster', bbox_to_anchor = ( 1.03, 1 ), loc = 'upper left' )
plt.tight_layout( )
plt.show( )


In [ ]:
# 1.2c - quick interpretation from HDBSCAN core clusters
hdbscan_profile_raw = ( 
    domain_input
    .loc[ domain_input[ 'cluster_hdbscan' ] != -1 ]
    .groupby( 'cluster_hdbscan' )[ domain_feature_cols ]
    .mean( )
)

if hdbscan_profile_raw.empty:
    print( 'HDBSCAN interpretation: selected parameters produced only noise.' )

else:
    hdbscan_profile = hdbscan_profile_raw.round( 3 )
    hdbscan_profile_z = ( hdbscan_profile_raw - cluster_profile_mean ) / cluster_profile_std
    hdbscan_name_map = hdbscan_profile_z.apply( build_cluster_name, axis = 1 ).reset_index( )

    hdbscan_counts = ( 
        domain_input[ 'cluster_hdbscan' ]
        .value_counts( )
        .rename_axis( 'cluster_hdbscan' )
        .reset_index( name = 'n_stations' )
        .sort_values( 'cluster_hdbscan' )
    )

    hdbscan_interpretation = hdbscan_name_map.merge( hdbscan_counts, on = 'cluster_hdbscan', how = 'left' )
    print( 'HDBSCAN cluster interpretation ( core clusters only ):' )
    display( hdbscan_interpretation.sort_values( 'cluster_hdbscan' ) )

    plt.figure( figsize = ( 12, 5 ) )
    sns.heatmap( hdbscan_profile, cmap = 'YlGn', annot = True, fmt = '.2f' )
    plt.title( 'HDBSCAN Mean Station-Character Features ( Core Clusters )' )
    plt.xlabel( 'Features' )
    plt.ylabel( 'HDBSCAN Cluster' )
    plt.tight_layout( )
    plt.show( )


### 1.3 - Atmospheric/Forcing KMeans (Air Temp, Wind, Solar, Precip)

barebones compare pass: build baseline forcing signatures and cluster them


In [ ]:
# 1.3 - build forcing features from the same baseline years and run a slim KMeans pass

forcing_base = water.copy( )
forcing_base[ 'datetime' ] = pd.to_datetime( forcing_base[ 'datetime' ], errors = 'coerce' )
forcing_base = forcing_base.dropna( subset = [ 'datetime' ] )
forcing_base[ 'year' ] = forcing_base[ 'datetime' ].dt.year
forcing_base[ 'month' ] = forcing_base[ 'datetime' ].dt.month

forcing_base = forcing_base.merge( 
    baseline_years[ [ 'region', 'station', 'year' ] ],
    on = [ 'region', 'station', 'year' ],
    how = 'inner',
)
forcing_base = forcing_base.merge( eligible_stations, on = [ 'region', 'station' ], how = 'inner' )

# lets just see if the air above these stations also lend themselves to a separate classification?
forcing_annual = ( 
    forcing_base
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        air_temp_ymean = ( 'air_temp', 'mean' ),
        wind_speed_ymean = ( 'wind_speed', 'mean' ),
        solar_radiation_ymean = ( 'solar_radiation', 'mean' ),
        precipitation_ymean = ( 'precipitation', 'mean' ),
    )
)

forcing_annual_means = ( 
    forcing_annual
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        mean_annual_air_temp = ( 'air_temp_ymean', 'mean' ),
        mean_annual_wind_speed = ( 'wind_speed_ymean', 'mean' ),
        mean_annual_solar_radiation = ( 'solar_radiation_ymean', 'mean' ),
        mean_annual_precipitation = ( 'precipitation_ymean', 'mean' ),
    )
)

forcing_monthly = ( 
    forcing_base
    .groupby( [ 'region', 'station', 'month' ], as_index = False )
    .agg( 
        air_temp_mmean = ( 'air_temp', 'mean' ),
        wind_speed_mmean = ( 'wind_speed', 'mean' ),
        solar_radiation_mmean = ( 'solar_radiation', 'mean' ),
        precipitation_mmean = ( 'precipitation', 'mean' ),
    )
)

forcing_seasonal_amp = ( 
    forcing_monthly
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        seasonal_amp_air_temp = ( 'air_temp_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_wind_speed = ( 'wind_speed_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_solar_radiation = ( 'solar_radiation_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_precipitation = ( 'precipitation_mmean', lambda s: s.max( ) - s.min( ) ),
    )
)

forcing_feature_cols = [ 
    'mean_annual_air_temp',
    'seasonal_amp_air_temp',
    'mean_annual_wind_speed',
    'seasonal_amp_wind_speed',
    'mean_annual_solar_radiation',
    'seasonal_amp_solar_radiation',
    'mean_annual_precipitation',
    'seasonal_amp_precipitation',
]

forcing_input = forcing_annual_means.merge( 
    forcing_seasonal_amp,
    on = [ 'region', 'station' ],
    how = 'inner',
)

for col in forcing_feature_cols:
    forcing_input[ col ] = forcing_input.groupby( 'region' )[ col ].transform( lambda s: s.fillna( s.median( ) ) )
    forcing_input[ col ] = forcing_input[ col ].fillna( forcing_input[ col ].median( ) )

In [ ]:
forcing_input.describe( ).round( 3 ).T

In [ ]:
scaler_forcing = StandardScaler( )
X_scaled_forcing = scaler_forcing.fit_transform( forcing_input[ forcing_feature_cols ] )

k_scan_forcing_rows = [ ]
for k in range( 2, 11 ):
    km_scan = KMeans( n_clusters = k, random_state = 42, n_init = 20 )
    labels_scan = km_scan.fit_predict( X_scaled_forcing )
    k_scan_forcing_rows.append( { 
        'k': k,
        'inertia': float( km_scan.inertia_ ),
        'silhouette': float( silhouette_score( X_scaled_forcing, labels_scan ) ),
    } )

k_scan_forcing = pd.DataFrame( k_scan_forcing_rows )
k_scan_forcing[ 'inertia_drop' ] = k_scan_forcing[ 'inertia' ].shift( 1 ) - k_scan_forcing[ 'inertia' ]
k_scan_forcing[ 'inertia_drop_pct' ] = k_scan_forcing[ 'inertia_drop' ] / k_scan_forcing[ 'inertia' ].shift( 1 )

recommended_k_forcing = int( k_scan_forcing.loc[ k_scan_forcing[ 'silhouette' ].idxmax( ), 'k' ] )
k_min_interpretable_forcing = 3
k_max_interpretable_forcing = 6
k_scan_forcing_interpretable = k_scan_forcing.loc[ 
    k_scan_forcing[ 'k' ].between( k_min_interpretable_forcing, k_max_interpretable_forcing )
].copy( )

k_clusters_forcing = int( 
    k_scan_forcing_interpretable.loc[ k_scan_forcing_interpretable[ 'silhouette' ].idxmax( ), 'k' ]
)
kmeans_forcing = KMeans( n_clusters = k_clusters_forcing, random_state = 42, n_init = 20 )
forcing_input[ 'forcing_cluster' ] = kmeans_forcing.fit_predict( X_scaled_forcing )
forcing_silhouette = float( silhouette_score( X_scaled_forcing, forcing_input[ 'forcing_cluster' ] ) )

print( 'forcing features:', forcing_feature_cols )
print( '\nforcing-feature k scan:' )
print( k_scan_forcing.round( 4 ) )
print( f'unbounded silhouette max K: { recommended_k_forcing }' )
print( f'interpretable-window K ( { k_min_interpretable_forcing }-{ k_max_interpretable_forcing } ): { recommended_k_forcing }' )
print( f'forcing silhouette at chosen K: { round( forcing_silhouette, 4 ) }' )
print( '\nforcing cluster sizes:' )
print( forcing_input[ 'forcing_cluster' ].value_counts( ).sort_index( ) )

# side by side
fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )
axes[ 0 ].plot( 
    k_scan_forcing[ 'k' ],
    k_scan_forcing[ 'inertia' ],
    marker = 'o'
)
axes[ 0 ].set_title( 'Domain Features: K vs Inertia' )
axes[ 0 ].set_xlabel( 'K' )
axes[ 0 ].set_ylabel( 'Inertia' )

axes[ 1 ].plot( 
    k_scan_forcing[ 'k' ],
    k_scan_forcing[ 'silhouette' ],
    marker = 'o'
)
axes[ 1 ].set_title( 'Domain Features: K vs Silhouette Score' )
axes[ 1 ].set_xlabel( 'K' )
axes[ 1 ].set_ylabel( 'Silhouette Score' )

plt.tight_layout( )
plt.show( )


In [ ]:
# and again ... but for the air
# btw, forcing as in the external components driving change int he water
pca_forcing = PCA( n_components = 3 )
pcs_forcing = pca_forcing.fit_transform( X_scaled_forcing )

pca_loadings_forcing = pd.DataFrame( 
    pca_forcing.components_.T,
    index = forcing_feature_cols,
    columns = [ 'PC1', 'PC2', 'PC3' ],
)

forcing_feature_phrase_map = { 
    'mean_annual_air_temp': ( 'warmer air', 'cooler air' ),
    'seasonal_amp_air_temp': ( 'larger air-temp seasonality', 'smaller air-temp seasonality' ),
    'mean_annual_wind_speed': ( 'windier conditions', 'calmer conditions' ),
    'seasonal_amp_wind_speed': ( 'larger wind seasonality', 'smaller wind seasonality' ),
    'mean_annual_solar_radiation': ( 'higher solar load', 'lower solar load' ),
    'seasonal_amp_solar_radiation': ( 'larger solar seasonality', 'smaller solar seasonality' ),
    'mean_annual_precipitation': ( 'wetter conditions', 'drier conditions' ),
    'seasonal_amp_precipitation': ( 'larger precip seasonality', 'smaller precip seasonality' ),
}


def pc_axis_interpretation_forcing( component_name ):
    top_features = pca_loadings_forcing[ component_name ].abs( ).sort_values( ascending = False ).head( 2 ).index
    text_parts = [ ]

    for feat in top_features:
        pos_text, neg_text = forcing_feature_phrase_map[ feat ]
        loading = pca_loadings_forcing.loc[ feat, component_name ]
        text_parts.append( pos_text if loading >= 0 else neg_text )

    return ' + '.join( text_parts )


forcing_plot = forcing_input[ [ 'region', 'station', 'forcing_cluster' ] ].copy( )
forcing_plot[ 'pc1' ] = pcs_forcing[ :, 0 ]
forcing_plot[ 'pc2' ] = pcs_forcing[ :, 1 ]
forcing_plot[ 'pc3' ] = pcs_forcing[ :, 2 ]

pc1_forcing_var_pct = round( float( pca_forcing.explained_variance_ratio_[ 0 ] ) * 100, 1 )
pc2_forcing_var_pct = round( float( pca_forcing.explained_variance_ratio_[ 1 ] ) * 100, 1 )
pc3_forcing_var_pct = round( float( pca_forcing.explained_variance_ratio_[ 2 ] ) * 100, 1 )
forcing_pca_3d_var_pct = round( pc1_forcing_var_pct + pc2_forcing_var_pct + pc3_forcing_var_pct, 1 )

pc1_forcing_label_text = pc_axis_interpretation_forcing( 'PC1' )
pc2_forcing_label_text = pc_axis_interpretation_forcing( 'PC2' )
pc3_forcing_label_text = pc_axis_interpretation_forcing( 'PC3' )

print( f'forcing PCA variance: PC1={ pc1_forcing_var_pct }%, PC2={ pc2_forcing_var_pct }%, PC3={ pc3_forcing_var_pct }%, cumulative={ forcing_pca_3d_var_pct }%' )
print( f'PC1 interpretation: { pc1_forcing_label_text }' )
print( f'PC2 interpretation: { pc2_forcing_label_text }' )
print( f'PC3 interpretation: { pc3_forcing_label_text }' )

plt.figure( figsize = ( 12, 8 ) )
sns.scatterplot( 
    data = forcing_plot,
    x = 'pc1',
    y = 'pc2',
    hue = 'forcing_cluster',
    palette = 'tab10',
    s = 70,
    alpha = 0.88,
)
plt.title( f'Forcing KMeans Clusters in PCA Space (PC1/PC2, K = { k_clusters_forcing })' )
plt.xlabel( f'PC1 ( { pc1_forcing_var_pct }% var )' )
plt.ylabel( f'PC2 ( { pc2_forcing_var_pct }% var )' )
plt.legend( title = 'forcing cluster', bbox_to_anchor = ( 1.03, 1 ), loc = 'upper left' )
plt.tight_layout( )
plt.show( )


In [ ]:
# 1.3 - optional interactive 3d PCA view (plotly)
import plotly.express as px

forcing_plot_interactive = forcing_plot.copy( )
forcing_plot_interactive[ 'forcing_cluster' ] = forcing_plot_interactive[ 'forcing_cluster' ].astype( str )

fig_forcing_3d = px.scatter_3d( 
    forcing_plot_interactive,
    x = 'pc1',
    y = 'pc2',
    z = 'pc3',
    color = 'forcing_cluster',
    hover_data = [ 'region', 'station', 'forcing_cluster' ],
    title = f'Forcing KMeans 3D PCA (Interactive) | K = { k_clusters_forcing }',
    opacity = 0.85,
)
fig_forcing_3d.update_traces( marker = dict( size = 4 ) )
fig_forcing_3d.update_layout( 
    height = 700
)
fig_forcing_3d.show( )


In [ ]:
# 1.3 - forcing cluster profiles + auto-naming
forcing_cluster_profile_raw = forcing_input.groupby( 'forcing_cluster' )[ forcing_feature_cols ].mean( )
forcing_cluster_profile = forcing_cluster_profile_raw.round( 3 )

forcing_cluster_profile_mean = forcing_cluster_profile_raw.mean( )
forcing_cluster_profile_std = forcing_cluster_profile_raw.std( ddof = 0 ).replace( 0, np.nan )
forcing_cluster_profile_z = ( forcing_cluster_profile_raw - forcing_cluster_profile_mean ) / forcing_cluster_profile_std


def forcing_bucket_tag( value, low_label, mid_label, high_label, threshold = 0.35 ):
    if pd.isna( value ):
        return mid_label

    if value >= threshold:
        return high_label

    if value <= -threshold:
        return low_label

    return mid_label


def build_forcing_cluster_name( z_row ):
    temp_tag = forcing_bucket_tag( z_row[ 'mean_annual_air_temp' ], 'Cool Air', 'Mild Air', 'Warm Air' )
    rain_tag = forcing_bucket_tag( z_row[ 'mean_annual_precipitation' ], 'Dry', 'Mid Rain', 'Wet' )
    wind_tag = forcing_bucket_tag( z_row[ 'mean_annual_wind_speed' ], 'Calm', 'Mixed Wind', 'Windy' )

    solar_tag = forcing_bucket_tag( z_row[ 'mean_annual_solar_radiation' ], 'Low Sun', 'Mid Sun', 'High Sun' )
    season_tag = forcing_bucket_tag( z_row[ 'seasonal_amp_air_temp' ], 'Stable', 'Mixed', 'Seasonal' )

    name = f'{ temp_tag } / { rain_tag } / { wind_tag }'
    short_name = f'{ temp_tag }-{ rain_tag }'
    note = f'{ solar_tag }, { season_tag }'

    return pd.Series( { 'forcing_cluster_name': name, 'forcing_cluster_label': short_name, 'forcing_cluster_note': note } )


forcing_cluster_name_map = forcing_cluster_profile_z.apply( build_forcing_cluster_name, axis = 1 ).reset_index( )
forcing_input = forcing_input.merge( forcing_cluster_name_map, on = 'forcing_cluster', how = 'left' )
forcing_plot = forcing_plot.merge( forcing_cluster_name_map, on = 'forcing_cluster', how = 'left' )

forcing_cluster_preview = ( 
    forcing_input
    .merge( 
        station_baseline_display[ [ 'region', 'station', 'station_name' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'forcing_cluster', 'region', 'station' ] )
)

print( 'forcing cluster names:' )
display( forcing_cluster_name_map.sort_values( 'forcing_cluster' ) )

print( 'forcing cluster mean feature profiles:' )
forcing_cluster_profile

plt.figure( figsize = ( 12, 5 ) )
sns.heatmap( forcing_cluster_profile, cmap = 'YlGnBu', annot = True, fmt = '.2f' )
plt.title( 'Forcing Cluster Mean Features' )
plt.xlabel( 'Features' )
plt.ylabel( 'Forcing Cluster' )
plt.tight_layout( )
plt.show( )

forcing_cluster_preview[ [ 'region', 'station', 'station_name', 'forcing_cluster', 'forcing_cluster_name', 'forcing_cluster_label', 'forcing_cluster_note' ] ].sample( 20 )


### 1.3b - Barebones HDBSCAN on Forcing PCA Space

quick density-based pass on forcing PCA coordinates ( pc1 / pc2 )


In [ ]:
# 1.3b - run HDBSCAN directly in forcing PCA space
# this uses the pcs from 1.3 so we can compare shape directly
forcing_hdbscan_min_cluster_size = 4
forcing_hdbscan_min_samples = 3

forcing_hdbscan_backend = 'hdbscan-package'

forcing_hdbscan_backend = 'sklearn'
forcing_hdbscan_model = SklearnHDBSCAN( 
    min_cluster_size = forcing_hdbscan_min_cluster_size,
    min_samples = forcing_hdbscan_min_samples,
    cluster_selection_method = 'eom',
)

X_forcing_pca = forcing_plot[ [ 'pc1', 'pc2' ] ].to_numpy( )
forcing_input[ 'forcing_cluster_hdbscan_pca' ] = forcing_hdbscan_model.fit_predict( X_forcing_pca )

forcing_hdbscan_n_clusters = int( len( set( forcing_input[ 'forcing_cluster_hdbscan_pca' ] ) - { -1 } ) )
forcing_hdbscan_noise_pct = float( ( forcing_input[ 'forcing_cluster_hdbscan_pca' ] == -1 ).mean( ) * 100 )
forcing_hdbscan_core_mask = forcing_input[ 'forcing_cluster_hdbscan_pca' ] != -1

if forcing_hdbscan_n_clusters >= 2 and forcing_hdbscan_core_mask.sum( ) >= 3:
    forcing_hdbscan_silhouette = float( 
        silhouette_score( 
            X_forcing_pca[ forcing_hdbscan_core_mask ],
            forcing_input.loc[ forcing_hdbscan_core_mask, 'forcing_cluster_hdbscan_pca' ],
        )
    )

else:
    forcing_hdbscan_silhouette = np.nan

forcing_kmeans_silhouette_pca = float( silhouette_score( X_forcing_pca, forcing_input[ 'forcing_cluster' ] ) )

forcing_pca_cluster_compare = pd.DataFrame( [ 
    { 
        'method': 'kmeans_on_forcing_features',
        'n_clusters': int( k_clusters_forcing ),
        'noise_pct': 0.0,
        'silhouette_on_pca_space': forcing_kmeans_silhouette_pca,
    },
    { 
        'method': 'hdbscan_on_forcing_pca',
        'n_clusters': forcing_hdbscan_n_clusters,
        'noise_pct': forcing_hdbscan_noise_pct,
        'silhouette_on_pca_space': forcing_hdbscan_silhouette,
    },
] )

print( f'forcing hdbscan backend: { forcing_hdbscan_backend }' )
print( f'forcing hdbscan params: min_cluster_size = { forcing_hdbscan_min_cluster_size }, min_samples = { forcing_hdbscan_min_samples }' )
print( f'forcing hdbscan clusters: { forcing_hdbscan_n_clusters } | noise: { round( forcing_hdbscan_noise_pct, 2 ) }%' )
forcing_pca_cluster_compare


In [ ]:
# 1.3b - visualize HDBSCAN labels on the same forcing PCA coordinates
forcing_hdbscan_plot = forcing_plot[ [ 'region', 'station', 'pc1', 'pc2' ] ].copy( )
forcing_hdbscan_plot[ 'forcing_cluster_hdbscan_pca' ] = forcing_input[ 'forcing_cluster_hdbscan_pca' ].to_numpy( )

forcing_hdbscan_clusters = sorted( c for c in forcing_hdbscan_plot[ 'forcing_cluster_hdbscan_pca' ].unique( ) if c != -1 )
forcing_hdbscan_palette = { -1: '#9e9e9e' }
for idx, cluster_id in enumerate( forcing_hdbscan_clusters ):
    forcing_hdbscan_palette[ int( cluster_id ) ] = sns.color_palette( 'tab10', 10 )[ idx % 10 ]

plt.figure( figsize = ( 12, 8 ) )
sns.scatterplot( 
    data = forcing_hdbscan_plot,
    x = 'pc1',
    y = 'pc2',
    hue = 'forcing_cluster_hdbscan_pca',
    palette = forcing_hdbscan_palette,
    s = 70,
    alpha = 0.90,
)
plt.title( 'HDBSCAN on Forcing PCA Space' )
plt.xlabel( f'PC1 ( { pc1_forcing_var_pct }% var )' )
plt.ylabel( f'PC2 ( { pc2_forcing_var_pct }% var )' )
plt.legend( title = 'hdbscan forcing cluster', bbox_to_anchor = ( 1.03, 1 ), loc = 'upper left' )
plt.tight_layout( )
plt.show( )

forcing_hdbscan_plot[ [ 'region', 'station', 'forcing_cluster_hdbscan_pca' ] ].sample( 20 )

### 1.3c - HDBSCAN Region-Mix Diagnostic

check whether forcing-pca hdbscan clusters are mostly single-region groupings


In [ ]:
# 1.3c - map HDBSCAN forcing-cluster ids against regions
# if dominant_region_share is very high, clusters are basically regional groupings

hdbscan_region_counts = pd.crosstab( 
    forcing_hdbscan_plot[ 'forcing_cluster_hdbscan_pca' ],
    forcing_hdbscan_plot[ 'region' ],
).sort_index( )

hdbscan_region_share = hdbscan_region_counts.div( hdbscan_region_counts.sum( axis = 1 ), axis = 0 )

hdbscan_region_summary = pd.DataFrame( { 
    'dominant_region': hdbscan_region_share.idxmax( axis = 1 ),
    'dominant_region_share': hdbscan_region_share.max( axis = 1 ),
    'n_stations': hdbscan_region_counts.sum( axis = 1 ),
} )
hdbscan_region_summary[ 'dominant_region_share' ] = hdbscan_region_summary[ 'dominant_region_share' ].round( 3 )
hdbscan_region_summary[ 'roughly_region_specific_80pct' ] = hdbscan_region_summary[ 'dominant_region_share' ] >= 0.80

print( 'hdbscan forcing-pca cluster x region counts:' )
display( hdbscan_region_counts )

print( 'hdbscan forcing-pca dominant-region summary:' )
display( hdbscan_region_summary.sort_values( [ 'dominant_region_share', 'n_stations' ], ascending = [ False, False ] ) )

fig, axes = plt.subplots( 1, 2, figsize = ( 16, 6 ) )

sns.heatmap( 
    hdbscan_region_counts,
    annot = True,
    fmt = 'd',
    cmap = 'Blues',
    ax = axes[ 0 ],
)
axes[ 0 ].set_title( 'HDBSCAN Forcing-PCA: Cluster x Region Counts' )
axes[ 0 ].set_xlabel( 'Region' )
axes[ 0 ].set_ylabel( 'HDBSCAN Cluster' )

sns.heatmap( 
    hdbscan_region_share * 100,
    annot = True,
    fmt = '.1f',
    cmap = 'YlOrRd',
    vmin = 0,
    vmax = 100,
    ax = axes[ 1 ],
    cbar_kws = { "label": "% within cluster" },
)
axes[ 1 ].set_title( 'HDBSCAN Forcing-PCA: Region Share Within Each Cluster' )
axes[ 1 ].set_xlabel( 'Region' )
axes[ 1 ].set_ylabel( 'HDBSCAN Cluster' )

plt.tight_layout( )
plt.show( )


### 1.4 - Cluster Context and Distribution

prepare cluster-station context and inspect distribution by region


In [ ]:
# 1.4 - cluster context check with available station metadata
# keep a clean station-level table for context and distribution checks

cluster_station = station_baseline_display.copy( )

# fallback labels in case this cell is run before 1.2 naming is created
if 'cluster_name' not in cluster_station.columns:
    cluster_station[ 'cluster_name' ] = 'Cluster ' + cluster_station[ 'cluster' ].astype( str )

if 'cluster_note' not in cluster_station.columns:
    cluster_station[ 'cluster_note' ] = ''

cluster_station[ 'latitude' ] = pd.to_numeric( cluster_station[ 'latitude' ], errors = 'coerce' )
cluster_station[ 'longitude' ] = pd.to_numeric( cluster_station[ 'longitude' ], errors = 'coerce' )

cluster_distribution_region = pd.crosstab( 
    cluster_station[ 'region' ],
    cluster_station[ 'cluster' ],
)

print( 'cluster distribution by region:' )
cluster_distribution_region

plt.figure( figsize = ( 10, 6 ) )
sns.heatmap( cluster_distribution_region, annot = True, fmt = 'd', cmap = 'Blues' )
plt.title( 'Cluster Distribution by Region' )
plt.xlabel( 'Cluster' )
plt.ylabel( 'Region' )
plt.tight_layout( )
plt.show( )


### 1.5 - Assign Regime Labels to the Working Dataset (Slim Join)

add only the cluster id to the large working table, and keep descriptive labels in small lookup tables


In [ ]:
# 1.5 - slim cluster assignment to the large working table
# keep it lean: add water-regime cluster id and forcing-regime cluster id to water

station_cluster_lookup = ( 
    station_baseline[ [ 'region', 'station', 'cluster' ] ]
    .drop_duplicates( )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

station_forcing_cluster_lookup = ( 
    forcing_input[ [ 'region', 'station', 'forcing_cluster' ] ]
    .drop_duplicates( )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

cluster_label_lookup = ( 
    station_baseline[ [ 'cluster', 'cluster_name', 'cluster_label', 'cluster_note' ] ]
    .drop_duplicates( subset = [ 'cluster' ] )
    .sort_values( 'cluster' )
    .reset_index( drop = True )
)

# save the small lookup tables for reproducibility
station_cluster_lookup_path = '../data/reference/t4d.station_cluster.lookup.csv'
station_forcing_cluster_lookup_path = '../data/reference/t4d.station_forcing_cluster.lookup.csv'
cluster_label_lookup_path = '../data/reference/t4d.cluster_label.lookup.csv'

station_cluster_lookup.to_csv( station_cluster_lookup_path, index = False )
station_forcing_cluster_lookup.to_csv( station_forcing_cluster_lookup_path, index = False )
cluster_label_lookup.to_csv( cluster_label_lookup_path, index = False )

# add only cluster ids to the large water table
for col in [ 'cluster', 'forcing_cluster' ]:
    if col in water.columns:
        water = water.drop( columns = [ col ] )

water = water.merge( station_cluster_lookup, on = [ 'region', 'station' ], how = 'left' )
water = water.merge( station_forcing_cluster_lookup, on = [ 'region', 'station' ], how = 'left' )

n_rows = len( water )
n_clustered = int( water[ 'cluster' ].notna( ).sum( ) )
n_forcing_clustered = int( water[ 'forcing_cluster' ].notna( ).sum( ) )
cluster_coverage = round( 100 * n_clustered / n_rows, 2 ) if n_rows > 0 else np.nan
forcing_cluster_coverage = round( 100 * n_forcing_clustered / n_rows, 2 ) if n_rows > 0 else np.nan

water_memory_mb = round( water.memory_usage( deep = True ).sum( ) / 1024 / 1024, 2 )

print( f'water rows with water-regime cluster assigned: {n_clustered} / {n_rows} ( {cluster_coverage}% )' )
print( f'water rows with forcing-regime cluster assigned: {n_forcing_clustered} / {n_rows} ( {forcing_cluster_coverage}% )' )
print( f'current water table size (MB): {water_memory_mb}' )
print( f'saved: {station_cluster_lookup_path}' )
print( f'saved: {station_forcing_cluster_lookup_path}' )
print( f'saved: {cluster_label_lookup_path}' )

#print( )
#print( 'cluster label lookup:' )
#cluster_label_lookup


In [ ]:
water.sample( 5 ).T

### 1.6 - Reference Tables for Reporting

commit cluster summary and station tables to small reference objects/files


In [ ]:
# 1.6 - save reference tables for reporting
cluster_station[ 'latitude' ] = pd.to_numeric( cluster_station[ 'latitude' ], errors = 'coerce' )
cluster_station[ 'longitude' ] = pd.to_numeric( cluster_station[ 'longitude' ], errors = 'coerce' )

forcing_station_labels = ( 
    forcing_input[ [ 
        'region',
        'station',
        'forcing_cluster',
        'forcing_cluster_name',
        'forcing_cluster_label',
        'forcing_cluster_note',
    ] ]
    .drop_duplicates( )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

for col in [ 'forcing_cluster', 'forcing_cluster_name', 'forcing_cluster_label', 'forcing_cluster_note' ]:
    if col in cluster_station.columns:
        cluster_station = cluster_station.drop( columns = [ col ] )

cluster_station = cluster_station.merge( 
    forcing_station_labels,
    on = [ 'region', 'station' ],
    how = 'left',
)

cluster_specs = ( 
    cluster_station
    .groupby( 'cluster', as_index = False )
    .agg( # gather the following specs for each station cluster: (see kmeans earlier)
        cluster_name = ( 'cluster_name', 'first' ),
        cluster_label = ( 'cluster_label', 'first' ),
        cluster_note = ( 'cluster_note', 'first' ),
        n_stations = ( 'station', 'nunique' ),
        n_regions = ( 'region', 'nunique' ),
        mean_annual_water_temp = ( 'mean_annual_water_temp', 'mean' ),
        seasonal_amp_water_temp = ( 'seasonal_amp_water_temp', 'mean' ),
        mean_annual_salinity = ( 'mean_annual_salinity', 'mean' ),
        seasonal_amp_salinity = ( 'seasonal_amp_salinity', 'mean' ),
        mean_annual_oxygen = ( 'mean_annual_oxygen', 'mean' ),
        seasonal_amp_oxygen = ( 'seasonal_amp_oxygen', 'mean' ),
        mean_annual_depth = ( 'mean_annual_depth', 'mean' ),
        mean_latitude = ( 'latitude', 'mean' ),
        mean_longitude = ( 'longitude', 'mean' ),
    )
    .sort_values( 'cluster' )
    .reset_index( drop = True )
)

# a tad more readable ... 
for col in [ 
    'mean_annual_water_temp',
    'seasonal_amp_water_temp',
    'mean_annual_salinity',
    'seasonal_amp_salinity',
    'mean_annual_oxygen',
    'seasonal_amp_oxygen',
    'mean_annual_depth',
    'mean_latitude',
    'mean_longitude',
]:
    cluster_specs[ col ] = cluster_specs[ col ].round( 3 )

cluster_station_table = cluster_station[ [ 
    'region',
    'region_name',
    'station',
    'station_name',
    'latitude',
    'longitude',
    'cluster',
    'cluster_name',
    'cluster_label',
    'cluster_note',
    'forcing_cluster',
    'forcing_cluster_name',
    'forcing_cluster_label',
    'forcing_cluster_note',
] ].sort_values( [ 'cluster', 'region', 'station' ] )

# caching the cluster specifications as a reference guide later
# if we reconfigure anyhting about the k-means work, will have to delete these
cluster_specs_path = '../data/reference/t4d.domain.cluster.specs.csv'
cluster_station_table_path = '../data/reference/t4d.cluster.station.table.csv'

cluster_specs.to_csv( cluster_specs_path, index = False )
cluster_station_table.to_csv( cluster_station_table_path, index = False )

#print( f'saved: {cluster_specs_path}' )
#print( f'saved: {cluster_station_table_path}' )

print( 'station table sample:' )
cluster_station_table.sample( 6 ).round( 3 ).T


## Phase 2 — Temporal Diagnostics
Goal: characterize lag structure and trends before building predictive features

- 2.1 run STL decomposition on water temperature per station
  - see: https://www.youtube.com/watch?v=1NXryMoU7Ho
  - annual + diurnal cycles 
  - extract trend components
  - see trends analysis for example
- 2.2 compute cross-correlations between air temp and water temp across a range of lags (0–28 days)
  - identify lag-at-peak-correlation per station
- 2.3 repeat cross-correlation for wind/precip → salinity, air temp → DO
  - reading suggests DO may be longer lag times
- 2.4 summarize lag structure by regime
  - do cold estuaries respond more slowly than warm ones?
  - other oddities? 
  - stuff we'd report on in paper/poster
- 2.5 identify stations showing trend drift in the STL trend component
  - lag candidates for regime-transition analysis
  - feeds into phase 5, btw...

In [ ]:
# helper function to find best lag correlation for a given driver-response pair


def best_lag_table( daily_df, driver_col, response_col, lag_max = 28, min_pairs = 40 ):
    rows = [ ]

    for ( region, station ), g in daily_df.groupby( [ 'region', 'station' ] ):
        s = g.sort_values( 'date' ).set_index( 'date' )

        driver = s[ driver_col ]
        response = s[ response_col ]

        best_lag = np.nan
        best_corr = np.nan
        best_n = 0

        for lag in range( lag_max + 1 ):
            # lag means driver leads response by `lag` days
            pair = pd.concat( [ response, driver.shift( lag ) ], axis = 1 ).dropna( )

            if len( pair ) < min_pairs:
                continue

            # how strongly do they align?
            corr = pair.iloc[ :, 0 ].corr( pair.iloc[ :, 1 ] )

            if pd.isna( corr ):
                continue

            # hold on to the strong correlation
            if pd.isna( best_corr ) or abs( corr ) > abs( best_corr ):
                best_lag = lag
                best_corr = corr
                best_n = len( pair )

        # save it
        rows.append( { 
            'region': region,
            'station': station,
            'best_lag_days': best_lag,
            'peak_corr': best_corr,
            'n_pairs': best_n,
        } )

    out = pd.DataFrame( rows )

    out = out.merge( 
        daily_df[ [ 'region', 'station', 'cluster' ] ].drop_duplicates( ),
        on = [ 'region', 'station' ],
        how = 'left',
    )

    return out

# prolly a few things before this that could benefit from
# being function wrapped

In [ ]:
# 2.1 - temporal diagnostics (first working pass)
# simple outputs: daily_air diagnostics + STL trend summary + lag-at-peak-correlation tables
from pathlib import Path
from statsmodels.tsa.seasonal import STL

phase2_out_dir = '../data/reference/phase2'
Path( phase2_out_dir ).mkdir( parents = True, exist_ok = True )

# this is a bit of work so the results will be cached to file and checked
# for if more runs are done that don't need to redo this
# to redo it... delete the cache files
daily_cache_path = f'{phase2_out_dir}/t4d.phase2.daily.csv'
stl_summary_path = f'{phase2_out_dir}/t4d.phase2.stl.summary.csv'
lag_air_to_water_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_water.csv'
lag_wind_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.wind_to_salinity.csv'
lag_precip_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.precip_to_salinity.csv'
lag_air_to_oxygen_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_oxygen.csv'
lag_cluster_summary_path = f'{phase2_out_dir}/t4d.phase2.lag.cluster.summary.csv'
trend_drift_candidates_path = f'{phase2_out_dir}/t4d.phase2.trend.drift.candidates.csv'

core_cache_paths = [ 
    daily_cache_path,
    stl_summary_path,
    lag_air_to_water_path,
    lag_wind_to_salinity_path,
    lag_precip_to_salinity_path,
    lag_air_to_oxygen_path,
]

# basically.. it took about 10 min per run,
# and that's a long time to wait just to change a font in a chart later...
phase2_cache_loaded = all( [ Path( fp ).exists( ) for fp in core_cache_paths ] )
if phase2_cache_loaded:
    print( 'loading cached phase 2 tables before recompute...' )

    daily_air = pd.read_csv( daily_cache_path )
    daily_air[ 'date' ] = pd.to_datetime( daily_air[ 'date' ], errors = 'coerce' )

    stl_summary = pd.read_csv( stl_summary_path )
    lag_air_to_water = pd.read_csv( lag_air_to_water_path )
    lag_wind_to_salinity = pd.read_csv( lag_wind_to_salinity_path )
    lag_precip_to_salinity = pd.read_csv( lag_precip_to_salinity_path )
    lag_air_to_oxygen = pd.read_csv( lag_air_to_oxygen_path )

    if Path( lag_cluster_summary_path ).exists( ):
        lag_cluster_summary = pd.read_csv( lag_cluster_summary_path )

    if Path( trend_drift_candidates_path ).exists( ):
        trend_drift_candidates = pd.read_csv( trend_drift_candidates_path )

    print( 'cached row counts:' )
    print( 'daily rows:', len( daily_air ) )
    print( 'stl summary:', len( stl_summary ) )
    print( 'air -> water lag:', len( lag_air_to_water ) )
    print( 'wind -> salinity lag:', len( lag_wind_to_salinity ) )
    print( 'precip -> salinity lag:', len( lag_precip_to_salinity ) )
    print( 'air -> oxygen lag:', len( lag_air_to_oxygen ) )

else:
    print( 'phase 2 cache not complete; computing fresh outputs...' )

    phase2 = water.copy( )
    phase2[ 'datetime' ] = pd.to_datetime( phase2[ 'datetime' ], errors = 'coerce' )
    phase2 = phase2.dropna( subset = [ 'datetime' ] )

    # fix air temperature column naming from raw files if needed
    if 'air_temp' not in phase2.columns:
        if 'm_temp_c' in phase2.columns:
            phase2[ 'air_temp' ] = phase2[ 'm_temp_c' ]

    # make sure we have cluster on each row
    if 'cluster' not in phase2.columns:
        phase2 = phase2.merge( 
            station_baseline[ [ 'region', 'station', 'cluster' ] ].drop_duplicates( ),
            on = [ 'region', 'station' ],
            how = 'left',
        )

    phase2[ 'date' ] = phase2[ 'datetime' ].dt.floor( 'D' )

    daily_air = ( 
        phase2
        .groupby( [ 'region', 'station', 'cluster', 'date' ], as_index = False )
        .agg( 
            water_temp_daily = ( 'water_temp', 'mean' ),
            salinity_daily = ( 'salinity', 'mean' ),
            oxygen_daily = ( 'oxygen', 'mean' ),
            air_temp_daily = ( 'air_temp', 'mean' ),
            wind_speed_daily = ( 'wind_speed', 'mean' ),
            precip_daily = ( 'precipitation', 'sum' ),
        )
    )

    print( 'daily diagnostic rows:', len( daily_air ) )
    daily_air.to_csv( daily_cache_path, index = False )

    # 2.1 STL on daily water temperature (annual cycle)
    # see https://www.youtube.com/watch?v=1NXryMoU7Ho
    stl_rows = [ ]

    for ( region, station ), g in daily_air.groupby( [ 'region', 'station' ] ):
        s = g.sort_values( 'date' ).set_index( 'date' )[ 'water_temp_daily' ]

        # regular daily index so STL has a stable cadence
        full_idx = pd.date_range( s.index.min( ), s.index.max( ), freq = 'D' )
        s = s.reindex( full_idx )

        if s.notna( ).sum( ) < 365:
            continue

        s = s.interpolate( limit_direction = 'both' )

        if s.notna( ).sum( ) < 365:
            continue

        # reminder .. STL is seasonal trend decomposition based on LOESS smoothing
        # it's robust to outliers and can handle some missing data
        # but it does require a regular time index and enough data to identify the seasonal pattern
        stl = STL( s, period = 365, robust = True ).fit( )

        trend = stl.trend
        slope_per_day = np.polyfit( np.arange( len( trend ) ), trend, 1 )[ 0 ]
        slope_per_year = slope_per_day * 365

        stl_rows.append( { 
            'region': region,
            'station': station,
            'cluster': g[ 'cluster' ].dropna( ).iloc[ 0 ] if g[ 'cluster' ].notna( ).any( ) else np.nan,
            'n_days': int( len( s ) ),
            'stl_trend_slope_c_per_year': float( slope_per_year ),
            'stl_seasonal_amp_c': float( stl.seasonal.max( ) - stl.seasonal.min( ) ),
        } )

    stl_summary = pd.DataFrame( stl_rows )
    print( 'stl station count:', len( stl_summary ) )

### 2.2 and 2.3 - build lag tables

In [ ]:
# 2.2 and 2.3 lag tables (with cache)
# don't redo this work if it's done
# but if you WANT to, then you'll have to clean the files out physically and rerun
if phase2_cache_loaded:
    print( 'using lag tables already loaded from cache in 2.1' )

else:
    phase2_out_dir = '../data/reference/phase2'
    Path( phase2_out_dir ).mkdir( parents = True, exist_ok = True )

    lag_air_to_water_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_water.csv'
    lag_wind_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.wind_to_salinity.csv'
    lag_precip_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.precip_to_salinity.csv'
    lag_air_to_oxygen_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_oxygen.csv'

    lag_paths = [ 
        lag_air_to_water_path,
        lag_wind_to_salinity_path,
        lag_precip_to_salinity_path,
        lag_air_to_oxygen_path,
    ]

    if all( [ Path( fp ).exists( ) for fp in lag_paths ] ):
        print( 'loading cached lag tables from disk...' )

        lag_air_to_water = pd.read_csv( lag_air_to_water_path )
        lag_wind_to_salinity = pd.read_csv( lag_wind_to_salinity_path )
        lag_precip_to_salinity = pd.read_csv( lag_precip_to_salinity_path )
        lag_air_to_oxygen = pd.read_csv( lag_air_to_oxygen_path )

    else:
        print( 'lag cache not found; computing lag tables...' )

        # defined up above
        # rinse and repeat
        lag_air_to_water = best_lag_table( daily_air, 'air_temp_daily', 'water_temp_daily', lag_max = 28 )
        lag_wind_to_salinity = best_lag_table( daily_air, 'wind_speed_daily', 'salinity_daily', lag_max = 28 )
        lag_precip_to_salinity = best_lag_table( daily_air, 'precip_daily', 'salinity_daily', lag_max = 28 )
        lag_air_to_oxygen = best_lag_table( daily_air, 'air_temp_daily', 'oxygen_daily', lag_max = 28 )

        lag_air_to_water.to_csv( lag_air_to_water_path, index = False )
        lag_wind_to_salinity.to_csv( lag_wind_to_salinity_path, index = False )
        lag_precip_to_salinity.to_csv( lag_precip_to_salinity_path, index = False )
        lag_air_to_oxygen.to_csv( lag_air_to_oxygen_path, index = False )

#print( )
#print( 'lag table row counts:' )
#print( 'air -> water:', len( lag_air_to_water ) )
#print( 'wind -> salinity:', len( lag_wind_to_salinity ) )
#print( 'precip -> salinity:', len( lag_precip_to_salinity ) )
#print( 'air -> oxygen:', len( lag_air_to_oxygen ) )


### 2.3b - visualize lag table signals

In [ ]:
lag_tables = { 
    'air -> water': lag_air_to_water,
    'wind -> salinity': lag_wind_to_salinity,
    'precip -> salinity': lag_precip_to_salinity,
    'air -> oxygen': lag_air_to_oxygen,
}

lag_plot_rows = [ ]
for relation, lag_df in lag_tables.items( ):
    lag_tmp = lag_df.copy( )
    lag_tmp[ 'relation' ] = relation
    lag_plot_rows.append( lag_tmp )

lag_plot = pd.concat( lag_plot_rows, ignore_index = True )
lag_plot = lag_plot.dropna( subset = [ 'best_lag_days', 'peak_corr' ] )
lag_plot[ 'best_lag_days' ] = lag_plot[ 'best_lag_days' ].astype( int )

lag_signal_summary = ( 
    lag_plot
    .groupby( 'relation', as_index = False )
    .agg( 
        n_stations = ( 'station', 'nunique' ),
        median_best_lag_days = ( 'best_lag_days', 'median' ),
        mean_best_lag_days = ( 'best_lag_days', 'mean' ),
        mean_abs_peak_corr = ( 'peak_corr', lambda s: s.abs( ).mean( ) ),
    )
)

for col in [ 'median_best_lag_days', 'mean_best_lag_days', 'mean_abs_peak_corr' ]:
    lag_signal_summary[ col ] = lag_signal_summary[ col ].round( 3 )

print( 'lag signal summary:' )
display( lag_signal_summary )

In [ ]:
fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )

sns.histplot( 
    data = lag_plot,
    x = 'best_lag_days',
    hue = 'relation',
    multiple = 'dodge',
    discrete = True,
    shrink = 0.85,
    ax = axes[ 0 ],
)
axes[ 0 ].set_title( 'Best-Lag Distribution by Relationship' )
axes[ 0 ].set_xlabel( 'Best Lag (days)' )
axes[ 0 ].set_ylabel( 'Station Count' )

sns.boxplot( 
    data = lag_plot,
    x = 'relation',
    y = 'peak_corr',
    ax = axes[ 1 ],
)
axes[ 1 ].axhline( 0, color = 'black', linestyle = '--', linewidth = 1 )
axes[ 1 ].set_title( 'Peak Correlation Spread by Relationship' )
axes[ 1 ].set_xlabel( '' )
axes[ 1 ].set_ylabel( 'Peak Correlation' )
axes[ 1 ].tick_params( axis = 'x', rotation = 20 )

plt.tight_layout( )
plt.show( )

In [ ]:
# lets just see how the best lags are distributed across the relationships and clusters
lag_count = ( 
    lag_plot
    .groupby( [ 'relation', 'best_lag_days' ] )
    .size( )
    .reset_index( name = 'n_stations' )
)

lag_heat = lag_count.pivot( index = 'relation', columns = 'best_lag_days', values = 'n_stations' ).fillna( 0 )

plt.figure( figsize = ( 12, 4 ) )
sns.heatmap( lag_heat, annot = True, fmt = '.0f', cmap = 'Blues' )
plt.title( 'Station Counts by Best Lag Day and Relationship' )
plt.xlabel( 'Best Lag (days)' )
plt.ylabel( '' )
plt.tight_layout( )
plt.show( )

### 2.3c - reading the lag visuals (what / how / why)

**What this is:**
- compact view of lag behavior across station-level relationships:
  - air -> water temp
  - wind -> salinity
  - precip -> salinity
  - air -> oxygen
- shows (1) where best lags concentrate, and (2) how strong peak correlations are

**How it was built:**
- phase 2 lag tables already loaded/computed above
- stacks them into one tidy frame with a `relation` label
- keeps station-level `best_lag_days` and `peak_corr`.
- produces ...
  - summary table by relation
  - histogram of best lag day counts
  - boxplot of peak correlations
  - heatmap of station counts by lag day

**Why this matters:**
- confirms whether lag behavior is tight or broad by driver-response pair
- helps justify Phase 3 feature choices (fixed windows vs lag-specific features).
- gives interpretable evidence for writeup/poster about system response timing.
- flags whether some links are weaker/noisier (wider correlation spread), which is useful for model expectations in later phases

### 2.4 - summarize lag structure by cluster type

In [ ]:
lag_cluster_summary = ( 
    lag_air_to_water
    .groupby( 'cluster', as_index = False )
    .agg( 
        n_stations = ( 'station', 'nunique' ),
        mean_lag_days_air_to_water = ( 'best_lag_days', 'mean' ),
        median_lag_days_air_to_water = ( 'best_lag_days', 'median' ),
        mean_peak_corr_air_to_water = ( 'peak_corr', 'mean' ),
    )
)

for col in [ 'mean_lag_days_air_to_water', 'median_lag_days_air_to_water', 'mean_peak_corr_air_to_water' ]:
    lag_cluster_summary[ col ] = lag_cluster_summary[ col ].round( 3 )

### 2.5 - trend drift candidates?

In [ ]:
# simple threshold: |slope| >= 0.05 c per year
trend_drift_threshold = 0.05
trend_drift = stl_summary.copy( )
trend_drift[ 'abs_trend_slope' ] = trend_drift[ 'stl_trend_slope_c_per_year' ].abs( )
trend_drift_candidates = trend_drift.loc[ trend_drift[ 'abs_trend_slope' ] >= trend_drift_threshold ].copy( )
trend_drift_candidates = trend_drift_candidates.sort_values( 'abs_trend_slope', ascending = False )

# save phase 2 outputs
phase2_out_dir = '../data/reference/phase2'
Path( phase2_out_dir ).mkdir( parents = True, exist_ok = True )

daily_air.to_csv( f'{phase2_out_dir}/t4d.phase2.daily.csv', index = False )
stl_summary.to_csv( f'{phase2_out_dir}/t4d.phase2.stl.summary.csv', index = False )
lag_air_to_water.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.air_to_water.csv', index = False )
lag_wind_to_salinity.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.wind_to_salinity.csv', index = False )
lag_precip_to_salinity.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.precip_to_salinity.csv', index = False )
lag_air_to_oxygen.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.air_to_oxygen.csv', index = False )
lag_cluster_summary.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.cluster.summary.csv', index = False )
trend_drift_candidates.to_csv( f'{phase2_out_dir}/t4d.phase2.trend.drift.candidates.csv', index = False )

print( f'saved phase 2 tables under: {phase2_out_dir}' )


In [ ]:

lag_cluster_summary

In [ ]:
# some examples?
trend_drift_candidates.sample( 10 ).round( 3 )

In [ ]:
# let's visualize the lag relationships for the cluster summary
# via a boxplot of the station-level lag values by cluster
fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )
sns.boxplot( 
    data = trend_drift_candidates, 
    x = 'cluster', 
    y = 'stl_trend_slope_c_per_year',
    ax = axes[ 0 ]
)
axes[ 0 ].set_title( 'Annual Trend (in C)' )
axes[ 0 ].set_xlabel( 'Cluster' )
axes[ 0 ].set_ylabel( 'Trend' )

sns.boxplot( 
    data = trend_drift_candidates, 
    x = 'cluster', 
    y = 'stl_seasonal_amp_c',
    ax = axes[ 1 ]
)
axes[ 1 ].set_title( 'Seasonal Amplitide (in C)' )
axes[ 1 ].set_xlabel( 'Cluster' )
axes[ 1 ].set_ylabel( 'Amplitude' )

plt.tight_layout( )
plt.show( )

## Phase 3 — Feature Engineering
Goal: build the lagged, accumulated features your models will actually use

- 3.1 construct rolling-mean atmospheric features at multiple windows
  - 24hr, 72hr, 7-day, 14-day air temp averages
  - in retrospect, 1, 7 and 28 days
  - but note that these don't really match the lag analysis done in phase 2
  - so that needs decisions,...
- 3.2 construct rate-of-change features (first diffs) for air temp and wind speed
- 3.3 maybe measure time cyclically?
  - day-of-year as (sin, cos) pair
  - hour-of-day similarly if using 1hr data
  - this just so hours 0 and 23, or days 1 and 365 don't SEEM far apart when they're right next to each other
  - some libraries probably do this already, btw...
### NOTE -- Phase 3 feeds both 6 and 7.

### 3.1 - rolling air temps

In [ ]:
# to get rolling averages we dont want air temp NAs
# but we don't want to delete other data... 
# so here's a temp copy
phase3 = water.copy( )
phase3[ 'datetime' ] = pd.to_datetime( phase3[ 'datetime' ], errors = 'coerce' )
phase3 = phase3.dropna( subset = [ 'datetime' ] )
phase3[ 'date' ] = phase3[ 'datetime' ].dt.floor( 'D' )

# now, get the resolution down to daily means and we'll roll them avergages
daily_air = ( 
    phase3
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( 
        air_temp = ( 'air_temp', 'mean' ),
        wind_speed = ( 'wind_speed', 'mean' ),
        solar = ( 'solar_radiation', 'mean' ),
        precip = ( 'precipitation', 'sum' )
    )
    .sort_values( [ 'region', 'station', 'date' ] )
)

# over-engineer a bit and get a whole bunch of rolling averages for different windows and features
for s in [ 'air_temp', 'wind_speed', 'precip', 'solar' ]:
    for w in [ 1, 7, 28 ]: 
        daily_air[ f'{s}_r{w}d' ] = ( 
            daily_air
            .groupby( [ 'region', 'station' ] )[ s ]
            .transform( lambda s: s.shift( 1 ).rolling( window = w, min_periods = 1 ).mean( ) )
        )

# that shift 1 keeps the current day out of the average

In [ ]:
daily_air.describe( ).round( 3 ).T

Hmm... some of those mins and maxes look wild

### 3.3 - assessing the day of year as a circle rather than a line

This one will need some reading... it took a fair bit to get it, especially needing both sin and cos

To wit (not hoid):
- real stations peak at different times (phase shifts) of the year (diff climates)
- with both sin and cos, the model can form a(sin) + b(cos), which is a sine wave with any phase offset

In [ ]:
# 0.25 includes leap days
daily_air[ 'doy_sin' ] = np.sin( 2 * np.pi * daily_air[ 'date' ].dt.dayofyear / 365.25 )
daily_air[ 'doy_cos' ] = np.cos( 2 * np.pi * daily_air[ 'date' ].dt.dayofyear / 365.25 )

## Phase 4 — Water-Property Delta-From-Mean Features
Goal: construct station-relative anomaly features from baseline-period means

- 4.1 pick a baseline period and compute station baseline means
- 4.2 compute one delta feature example (salinity)

### NOTE -- Phase 4 primarily feeds Phase 7+ modeling.

In [ ]:
baseline_start = '1995-01-01'
baseline_end = '2000-12-31'

# daily water metrics used in later phases (including Phase 5 rolling means)
daily_water = ( 
    phase3
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( 
        water_temp = ( 'water_temp', 'mean' ),
        salinity = ( 'salinity', 'mean' ),
        oxygen = ( 'oxygen', 'mean' ),
        ph = ( 'ph', 'mean' ),
        depth = ( 'depth', 'mean' )
    )
    .sort_values( [ 'region', 'station', 'date' ] )
)



In [ ]:
properties_baseline = ( 
    daily_water
    .loc[ daily_water[ 'date' ].between( baseline_start, baseline_end ) ]
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        water_temp_baseline = ( 'water_temp', 'mean' ),
        salinity_baseline = ( 'salinity', 'mean' ),
        oxygen_baseline = ( 'oxygen', 'mean' ),
        ph_baseline = ( 'ph', 'mean' ),
        depth_baseline = ( 'depth', 'mean' )
    )
)

In [ ]:
daily_water = daily_water.merge( 
    properties_baseline,
    on = [ 'region', 'station' ],
    how = 'left',
)

daily_water[ 'delta_water_temp' ] = daily_water[ 'water_temp' ] - daily_water[ 'water_temp_baseline' ]
daily_water[ 'delta_salinity' ] = daily_water[ 'salinity' ] - daily_water[ 'salinity_baseline' ]
daily_water[ 'delta_oxygen' ] = daily_water[ 'oxygen' ] - daily_water[ 'oxygen_baseline' ]
daily_water[ 'delta_ph' ] = daily_water[ 'ph' ] - daily_water[ 'ph_baseline' ]
daily_water[ 'delta_depth' ] = daily_water[ 'depth' ] - daily_water[ 'depth_baseline' ]

In [ ]:
daily_water.describe( ).round( 3 ).T

## Phase 5 — Regime Transition Detection
Goal: identify natural validation cases and a forward-projection threshold

- 5.1 for each station, compute rolling 5-year means of key variables
  - slide across the full 30-year record
- 5.2 Compare rolling means to baseline regime centroids
  - flag stations whose recent window has crossed into an adjacent regime's territory
- 5.3 manually review flagged stations for plausibility
  - data gaps, instrument changes, or genuine trend?
- 5.4 designate confirmed transitioning stations as a held-out validation set
  - do not use in model training

### NOTE -- Phase 5 must complete before Phase 6

### 5.1 for each station, compute rolling 5-year means of key variables
  - slide across the full 30-year record

In [ ]:
# start from daily_water, then build monthly summaries for more gap-tolerant rolling windows
fiveyear_water = daily_water.copy( )
fiveyear_water[ 'date' ] = pd.to_datetime( fiveyear_water[ 'date' ], errors = 'coerce' )
fiveyear_water = fiveyear_water.dropna( subset = [ 'date' ] )
fiveyear_water = fiveyear_water.sort_values( [ 'region', 'station', 'date' ] )
fiveyear_water[ 'month' ] = fiveyear_water[ 'date' ].dt.to_period( 'M' ).dt.to_timestamp( )

# keep this explicit and readable for students
water_metric_candidates = [ 
    'water_temp',
    'salinity',
    'oxygen',
    'ph',
    'depth',
]

# only use metrics that are actually present in daily_water
fiveyear_water_metrics = [ m for m in water_metric_candidates if m in fiveyear_water.columns ]

monthly_water = ( 
    fiveyear_water
    .groupby( [ 'region', 'station', 'month' ], as_index = False )[ fiveyear_water_metrics ]
    .mean( )
    .sort_values( [ 'region', 'station', 'month' ] )
)

print( 'monthly metrics used for rolling windows:', fiveyear_water_metrics )

In [ ]:
# 5-year rolling means on monthly data (60 months)
# this is more tolerant to seasonal gaps than strict daily rolling
for metric in fiveyear_water_metrics:
    monthly_water[ f'{metric}_roll5y' ] = ( 
        monthly_water
        .groupby( [ 'region', 'station' ] )[ metric ]
        .transform( lambda values: values.rolling( window = 60, min_periods = 36 ).mean( ) )
    )

# rolling coverage over the same 5-year monthly window
monthly_water[ 'has_any_data' ] = monthly_water[ fiveyear_water_metrics ].notna( ).any( axis = 1 ).astype( int )
monthly_water[ 'months_with_any_data_5y' ] = ( 
    monthly_water
    .groupby( [ 'region', 'station' ] )[ 'has_any_data' ]
    .transform( lambda values: values.rolling( window = 60, min_periods = 1 ).sum( ) )
)
monthly_water[ 'coverage_ratio_5y' ] = monthly_water[ 'months_with_any_data_5y' ] / 60.0

# attach monthly rolling outputs back onto daily rows for plotting and event timing
roll5y_cols = [ f'{m}_roll5y' for m in fiveyear_water_metrics ]
monthly_roll_cols = [ 'region', 'station', 'month' ] + roll5y_cols + [ 'months_with_any_data_5y', 'coverage_ratio_5y' ]
fiveyear_water = fiveyear_water.merge( monthly_water[ monthly_roll_cols ], on = [ 'region', 'station', 'month' ], how = 'left' )

# keep a backward-compatible alias in case later cells use "fiveyear"
fiveyear = fiveyear_water.copy( )

In [ ]:
roll5y_cols = [ f'{m}_roll5y' for m in fiveyear_water_metrics ]

fiveyear_water[ [ 'region', 'station', 'date', 'month', 'coverage_ratio_5y' ] + roll5y_cols ].sample( 10 )

### 5.2 Compare rolling means to baseline regime centroids
  - flag stations whose recent window has crossed into an adjacent regime's territory

In [ ]:
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler

In [ ]:
# build a monthly Phase-5 working table for classification
required_roll_cols = [ 
    'water_temp_roll5y',
    'salinity_roll5y',
    'oxygen_roll5y',
    'depth_roll5y',
]

available_roll_cols = [ col for col in required_roll_cols if col in monthly_water.columns ]

p5_monthly = monthly_water[ [ 'region', 'station', 'month', 'coverage_ratio_5y' ] + available_roll_cols ].copy( )

p5_monthly = p5_monthly.rename( columns = { 
    'month': 'date',
    'water_temp_roll5y': 'mean_annual_water_temp',
    'salinity_roll5y': 'mean_annual_salinity',
    'oxygen_roll5y': 'mean_annual_oxygen',
    'depth_roll5y': 'mean_annual_depth',
} )

In [ ]:
# attach baseline cluster identity
baseline_cluster = cluster_station[ [ 'region', 'station', 'cluster' ] ].drop_duplicates( )
p5_monthly = p5_monthly.merge( baseline_cluster, on = [ 'region', 'station' ], how = 'left' )

In [ ]:
# compare against baseline centroids in standardized feature space
feature_cols = [ 
    'mean_annual_water_temp',
    'mean_annual_salinity',
    'mean_annual_oxygen',
    'mean_annual_depth',
]

# baseline station profiles from phase 1
ref = station_baseline[ [ 'cluster' ] + feature_cols ].dropna( ).copy( )
scaler_p5 = StandardScaler( ).fit( ref[ feature_cols ] )

ref_z = pd.DataFrame( 
    scaler_p5.transform( ref[ feature_cols ] ),
    columns = feature_cols,
    index = ref.index,
)
ref_z[ 'cluster' ] = ref[ 'cluster' ].values

centroids_z = ( 
    ref_z
    .groupby( 'cluster', as_index = True )[ feature_cols ]
    .mean( )
    .sort_index( )
)

In [ ]:
#ref_z
#centroids_z

In [ ]:
# monthly nearest-centroid assignment with coverage + partial-feature tolerance
coverage_threshold = 0.70
min_features_required = 3

feature_mean = pd.Series( scaler_p5.mean_, index = feature_cols )
feature_scale = pd.Series( scaler_p5.scale_, index = feature_cols )


def classify_month_row( row ):
    coverage = row.get( 'coverage_ratio_5y', np.nan )
    if pd.isna( coverage ) or coverage < coverage_threshold:
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': 0,
            'assignment_state': 'insufficient_coverage',
        } )

    available = [ col for col in feature_cols if pd.notna( row.get( col, np.nan ) ) ]
    if len( available ) < min_features_required:
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': len( available ),
            'assignment_state': 'insufficient_features',
        } )

    values = pd.to_numeric( row[ available ], errors = 'coerce' )
    scales = pd.to_numeric( feature_scale[ available ], errors = 'coerce' ).replace( 0, np.nan )
    means = pd.to_numeric( feature_mean[ available ], errors = 'coerce' )

    z_values = ( values - means ) / scales
    if z_values.isna( ).any( ):
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': len( available ),
            'assignment_state': 'scaling_issue',
        } )

    centroid_slice = centroids_z[ available ]

    try:
        centroid_array = np.asarray( centroid_slice.to_numpy( ), dtype = float )
        z_array = np.asarray( z_values.to_numpy( ), dtype = float )
        distances = np.sqrt( np.sum( ( centroid_array - z_array[ None, : ] ) ** 2, axis = 1 ) )

    except Exception:
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': len( available ),
            'assignment_state': 'distance_error',
        } )

    best_idx = int( np.argmin( distances ) )
    best_cluster = int( centroid_slice.index[ best_idx ] )
    best_dist = float( distances[ best_idx ] )

    if len( distances ) > 1:
        second_dist = float( np.partition( distances, 1 )[ 1 ] )
        margin = second_dist - best_dist

    else:
        second_dist = np.nan
        margin = np.nan

    state = 'classified_full' if len( available ) == len( feature_cols ) else 'classified_partial'

    return pd.Series( { 
        'implied_cluster': best_cluster,
        'dist_best': best_dist,
        'dist_second': second_dist,
        'margin': margin,
        'n_features_used': len( available ),
        'assignment_state': state,
    } )

In [ ]:
# apply classification row-by-row after coercing feature columns to numeric
for col in feature_cols:
    if col in p5_monthly.columns:
        p5_monthly[ col ] = pd.to_numeric( p5_monthly[ col ], errors = 'coerce' )

p5_monthly = pd.concat( [ p5_monthly, p5_monthly.apply( classify_month_row, axis = 1 ) ], axis = 1 )

p5_monthly[ 'assignment_state' ].value_counts( dropna = False )

In [ ]:
# confirm transitions on monthly assignments, then project onto daily rows
p5_monthly = p5_monthly.sort_values( [ 'region', 'station', 'date' ] ).copy( )

p5_monthly[ 'candidate_flip' ] = ( 
    p5_monthly[ 'implied_cluster' ].notna( )
    & p5_monthly[ 'cluster' ].notna( )
    & ( p5_monthly[ 'implied_cluster' ] != p5_monthly[ 'cluster' ] )
)

monthly_run_id = ( 
    p5_monthly
    .groupby( [ 'region', 'station' ] )[ 'implied_cluster' ]
    .transform( lambda values: values.ne( values.shift( ) ).cumsum( ) )
)

p5_monthly[ 'run_len_months' ] = ( 
    p5_monthly
    .groupby( [ 'region', 'station', monthly_run_id ] )[ 'implied_cluster' ]
    .transform( 'size' )
)

p5_monthly[ 'flip_confirmed_monthly' ] = ( 
    p5_monthly[ 'candidate_flip' ]
    & ( p5_monthly[ 'run_len_months' ] >= 6 )   # ~6 months persistence
    & ( p5_monthly[ 'margin' ] > 0.10 )
)

# bring monthly classification decisions down to each daily observation row
p5 = fiveyear_water[ [ 'region', 'station', 'date', 'month' ] ].copy( )

monthly_decision_cols = [ 
    'region',
    'station',
    'date',
    'cluster',
    'implied_cluster',
    'dist_best',
    'dist_second',
    'margin',
    'n_features_used',
    'assignment_state',
    'coverage_ratio_5y',
    'candidate_flip',
    'run_len_months',
    'flip_confirmed_monthly',
]

p5 = p5.merge( 
    p5_monthly[ monthly_decision_cols ].rename( columns = { 'date': 'month' } ),
    on = [ 'region', 'station', 'month' ],
    how = 'left',
)

p5[ 'flip_confirmed' ] = p5[ 'flip_confirmed_monthly' ].fillna( False )
p5[ 'run_len_days' ] = p5[ 'run_len_months' ] * 30.4375

p5 = p5.sort_values( [ 'region', 'station', 'date' ] ).copy( )

In [ ]:
p5[ [ 
    'region',
    'station',
    'date',
    'cluster',
    'implied_cluster',
    'assignment_state',
    'coverage_ratio_5y',
    'run_len_days',
    'margin',
    'flip_confirmed',
] ].sample( 10 )

In [ ]:
# anything?
first_event = ( 
    p5.loc[ p5[ 'flip_confirmed' ] ]
    .groupby( [ 'region', 'station' ], as_index = False )[ 'date' ]
    .min( )
    .rename( columns = { 'date': 'event_date' } )
)

first_event

In [ ]:
# may come to nothing, but... 
# let's do a quick survival analysis to see how long stations have gone without flipping, and if there are any patterns in which ones flip first
survival_df = ( 
    p5.groupby( [ 'region', 'station' ], as_index = False )[ 'date' ]
    .agg( start_date = 'min', end_date = 'max' )
    .merge( first_event, on = [ 'region', 'station' ], how = 'left' )
)

# common models like Kaplan-Meier or Cox Proportional Hazards require a duration and an event indicator
survival_df[ 'event' ] = survival_df[ 'event_date' ].notna( ).astype( int )
survival_df[ 'stop_date' ] = survival_df[ 'event_date' ].fillna( survival_df[ 'end_date' ] )
survival_df[ 'time_days' ] = ( survival_df[ 'stop_date' ] - survival_df[ 'start_date' ] ).dt.days

# not yet a plan for using this, but keeping it around in case we want to explore it more
#survival_df
survival_df.describe( ).round( 3 ).T

### 5.3 manually review flagged stations for plausibility
  - data gaps, instrument changes, or genuine trend?

In [ ]:

# top panel: implied regime over time
# middle panel: centroid-distance confidence over time
# bottom panel: rolling 5-year water metrics

cluster_label_map = ( 
    cluster_station[ [ 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'cluster' ] )
    .set_index( 'cluster' )[ 'cluster_label' ]
    .to_dict( )
)

flagged_stations = ( 
    p5.loc[ p5[ 'flip_confirmed' ], [ 'region', 'station', 'date' ] ]
    .groupby( [ 'region', 'station' ], as_index = False )[ 'date' ]
    .min( )
    .rename( columns = { 'date': 'first_flip_date' } )
    .sort_values( [ 'first_flip_date', 'region', 'station' ] )
)

print( f'stations with confirmed flips ({ len( flagged_stations ) }):' )
flagged_stations

In [ ]:
# let's look at one of them ...
station = 6  # <- pick your villain here (there were 24, so 0-23)
focus_region = flagged_stations.iloc[ station ][ 'region' ]
focus_station = flagged_stations.iloc[ station ][ 'station' ]

station_p5 = ( 
    p5.loc[ ( p5[ 'region' ] == focus_region ) & ( p5[ 'station' ] == focus_station ) ]
    .sort_values( 'date' )
    .copy( )
)

roll_plot_cols = [ 
    col
    for col in [ 'water_temp_roll5y', 'salinity_roll5y', 'oxygen_roll5y', 'ph_roll5y', 'depth_roll5y' ]
    if col in fiveyear_water.columns
]

station_roll = ( 
    fiveyear_water
    .loc[ 
        ( fiveyear_water[ 'region' ] == focus_region )
        & ( fiveyear_water[ 'station' ] == focus_station ),
        [ 'date' ] + roll_plot_cols,
    ]
    .sort_values( 'date' )
    .copy( )
)

baseline_cluster_val = np.nan
if len( station_p5 ) > 0 and station_p5[ 'cluster' ].notna( ).any( ):
    baseline_cluster_val = station_p5.loc[ station_p5[ 'cluster' ].notna( ), 'cluster' ].iloc[ 0 ]

fig, axes = plt.subplots( 3, 1, figsize = ( 14, 10 ), sharex = True )

axes[ 0 ].plot( 
    station_p5[ 'date' ],
    station_p5[ 'implied_cluster' ],
    color = 'tab:blue',
    linewidth = 1.6,
    label = 'Implied cluster',
)

if pd.notna( baseline_cluster_val ):
    axes[ 0 ].axhline( 
        baseline_cluster_val,
        color = 'tab:gray',
        linestyle = '--',
        linewidth = 1.2,
        label = 'Baseline cluster',
    )

flip_points = station_p5.loc[ station_p5[ 'flip_confirmed' ] ]
axes[ 0 ].scatter( 
    flip_points[ 'date' ],
    flip_points[ 'implied_cluster' ],
    color = 'tab:red',
    s = 20,
    label = 'Confirmed flip days',
)

cluster_ticks = station_p5[ 'implied_cluster' ].dropna( ).astype( int ).unique( ).tolist( )
if pd.notna( baseline_cluster_val ):
    cluster_ticks = cluster_ticks + [ int( baseline_cluster_val ) ]
cluster_ticks = sorted( set( cluster_ticks ) )

if len( cluster_ticks ) > 0:
    axes[ 0 ].set_yticks( cluster_ticks )
    axes[ 0 ].set_yticklabels( [ cluster_label_map.get( cid, f'Cluster {cid}' ) for cid in cluster_ticks ] )

axes[ 0 ].set_ylabel( 'Regime' )
axes[ 0 ].set_title( 'Phase 5.3 Review Panel: Regime Path + Confidence + Rolling Metrics' )
axes[ 0 ].legend( loc = 'best' )

axes[ 1 ].plot( 
    station_p5[ 'date' ],
    station_p5[ 'dist_best' ],
    color = 'tab:green',
    linewidth = 1.6,
    label = 'Distance to nearest centroid',
)
axes[ 1 ].plot( 
    station_p5[ 'date' ],
    station_p5[ 'dist_second' ],
    color = 'tab:orange',
    linewidth = 1.3,
    label = 'Distance to 2nd nearest centroid',
)
axes[ 1 ].plot( 
    station_p5[ 'date' ],
    station_p5[ 'margin' ],
    color = 'tab:purple',
    linewidth = 1.2,
    label = 'Margin (2nd - best)',
)
axes[ 1 ].axhline( 0.10, color = 'tab:red', linestyle = '--', linewidth = 1.0, label = 'Margin threshold (0.10)' )
axes[ 1 ].set_ylabel( 'Distance' )
axes[ 1 ].legend( loc = 'best' )

for col in roll_plot_cols:
    axes[ 2 ].plot( station_roll[ 'date' ], station_roll[ col ], linewidth = 1.1, label = col )

axes[ 2 ].set_ylabel( '5-year rolling value' )
axes[ 2 ].set_xlabel( 'Date' )
axes[ 2 ].legend( loc = 'best', ncol = 2 )

plt.tight_layout( )
plt.show( )

Okay, the above is confusing ... having manually looked at about 10, some seem to always be the same classification 

But are marked as transitory...

Gonna see about looking at them all at once:

In [ ]:
# 5.3 visualization #1b: full timeline for flagged stations
# one row per flagged station, one column per day
# white = no data, gray = data present but no implied cluster assignment
# left dot color = baseline regime for that station

from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

p5_vis = p5
fiveyear_vis = fiveyear_water

station_baseline_regime = ( 
    cluster_station[ [ 'region', 'station', 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'region', 'station' ] )
    .rename( columns = { 
        'cluster': 'baseline_cluster',
        'cluster_label': 'baseline_label',
    } )
)

cluster_label_map = ( 
    cluster_station[ [ 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'cluster' ] )
    .set_index( 'cluster' )[ 'cluster_label' ]
    .to_dict( )
)

flagged_from_p5 = ( 
    p5_vis.loc[ p5_vis[ 'flip_confirmed' ].fillna( False ), [ 'region', 'station', 'date' ] ]
    .groupby( [ 'region', 'station' ], as_index = False )[ 'date' ]
    .min( )
    .rename( columns = { 'date': 'first_flip_date' } )
)

offenders = flagged_from_p5.copy( )

offenders = offenders.merge( station_baseline_regime, on = [ 'region', 'station' ], how = 'left' )
offenders = offenders.sort_values( [ 'first_flip_date', 'region', 'station' ] ).reset_index( drop = True )
offenders[ 'row_label' ] = offenders[ 'region' ].astype( str ) + ' | ' + offenders[ 'station' ].astype( str )

print( f'flagged stations shown: {len( offenders )}' )

if len( offenders ) == 0:
    print( 'no flagged stations found in current cohort scope' )

else:
    timeline_start = pd.Timestamp( '1995-01-01' )
    timeline_end = pd.Timestamp( '2025-12-31' )
    all_dates = pd.date_range( timeline_start, timeline_end, freq = 'D' )

    station_grid = ( 
        offenders[ [ 'region', 'station', 'row_label' ] ]
        .assign( _tmp = 1 )
        .merge( pd.DataFrame( { 'date': all_dates, '_tmp': 1 } ), on = '_tmp', how = 'inner' )
        .drop( columns = [ '_tmp' ] )
    )

    raw_metric_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in fiveyear_vis.columns ]
    station_data_presence = ( 
        fiveyear_vis[ [ 'region', 'station', 'date' ] + raw_metric_cols ]
        .copy( )
        .assign( has_data = lambda frame: frame[ raw_metric_cols ].notna( ).any( axis = 1 ) if len( raw_metric_cols ) > 0 else True )
        .loc[ :, [ 'region', 'station', 'date', 'has_data' ] ]
    )

    p5_track = p5_vis[ [ 'region', 'station', 'date', 'implied_cluster' ] ].copy( )

    timeline = ( 
        station_grid
        .merge( station_data_presence, on = [ 'region', 'station', 'date' ], how = 'left' )
        .merge( p5_track, on = [ 'region', 'station', 'date' ], how = 'left' )
    )

    timeline[ 'has_data' ] = timeline[ 'has_data' ].fillna( False )

    timeline[ 'state' ] = -2
    timeline.loc[ timeline[ 'has_data' ], 'state' ] = -1
    timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'state' ] = timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'implied_cluster' ].astype( int )

    cluster_ids = sorted( cluster_station[ 'cluster' ].dropna( ).astype( int ).unique( ).tolist( ) )

    value_map = { -2: 0, -1: 1 }
    for idx, cid in enumerate( cluster_ids, start = 2 ):
        value_map[ int( cid ) ] = idx

    timeline[ 'cluster_plot' ] = timeline[ 'state' ].map( value_map )

    matrix = ( 
        timeline
        .pivot( index = 'row_label', columns = 'date', values = 'cluster_plot' )
        .reindex( offenders[ 'row_label' ] )
        .to_numpy( )
    )

    palette = [ '#ffffff', '#bdbdbd' ] + sns.color_palette( 'tab10', n_colors = len( cluster_ids ) ).as_hex( )
    cmap = ListedColormap( palette )
    norm = BoundaryNorm( np.arange( -0.5, len( palette ) + 0.5, 1 ), cmap.N )

    fig_height = max( 8, 0.45 * len( offenders ) + 2 )
    fig, ax = plt.subplots( figsize = ( 18, fig_height ) )
    ax.imshow( matrix, aspect = 'auto', interpolation = 'nearest', cmap = cmap, norm = norm )

    ax.set_yticks( np.arange( len( offenders ) ) )
    ax.set_yticklabels( offenders[ 'row_label' ] )

    baseline_values = offenders[ 'baseline_cluster' ].apply( lambda val: value_map.get( int( val ), 0 ) if pd.notna( val ) else 0 )
    baseline_colors = [ palette[ int( val ) ] for val in baseline_values ]

    ax.scatter( 
        np.full( len( offenders ), -4.0 ),
        np.arange( len( offenders ) ),
        s = 110,
        c = baseline_colors,
        edgecolors = 'black',
        linewidths = 0.7,
        clip_on = False,
        zorder = 5,
    )
    ax.set_xlim( -8, matrix.shape[ 1 ] - 0.5 )

    tick_dates = pd.date_range( timeline_start, timeline_end, freq = '2YS' )
    tick_positions = [ np.searchsorted( all_dates.values, np.datetime64( dt ) ) for dt in tick_dates ]
    ax.set_xticks( tick_positions )
    ax.set_xticklabels( [ str( dt.year ) for dt in tick_dates ], rotation = 45, ha = 'right' )

    ax.set_xlabel( 'Date' )
    ax.set_ylabel( 'Flagged Station ( Region | Station )' )
    ax.set_title( 'Flagged Station Regime Timeline ( Daily, 1995-2025 )' )

    legend_handles = [ 
        mpatches.Patch( color = palette[ 0 ], label = 'No data' ),
        mpatches.Patch( color = palette[ 1 ], label = 'Data, no classification' ),
    ]

    for cid in cluster_ids:
        label_text = cluster_label_map.get( cid, f'Cluster {cid}' )
        legend_handles.append( mpatches.Patch( color = palette[ value_map[ cid ] ], label = label_text ) )

    legend_handles.append( mpatches.Patch( facecolor = 'white', edgecolor = 'black', label = 'Left dot = baseline regime' ) )

    ax.legend( handles = legend_handles, loc = 'upper left', bbox_to_anchor = ( 1.01, 1.0 ), frameon = True, title = 'Timeline State' )

    plt.tight_layout( )
    plt.show( )

In [ ]:
# 5.3 visualization #1c: full timeline for unflagged stations
# one row per unflagged station, one column per day
# white = no data, gray = data present but no implied cluster assignment
# left dot color = baseline regime for that station

from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

p5_vis = p5
fiveyear_vis = fiveyear_water

station_baseline_regime = ( 
    cluster_station[ [ 'region', 'station', 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'region', 'station' ] )
    .rename( columns = { 
        'cluster': 'baseline_cluster',
        'cluster_label': 'baseline_label',
    } )
)

cluster_label_map = ( 
    cluster_station[ [ 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'cluster' ] )
    .set_index( 'cluster' )[ 'cluster_label' ]
    .to_dict( )
)

all_stations = station_baseline_regime[ [ 'region', 'station', 'baseline_cluster', 'baseline_label' ] ].drop_duplicates( )

flagged_keys = p5_vis.loc[ p5_vis[ 'flip_confirmed' ].fillna( False ), [ 'region', 'station' ] ].drop_duplicates( )

unflagged_stations = ( 
    all_stations
    .merge( flagged_keys.assign( is_flagged = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ 'is_flagged' ].isna( ), [ 'region', 'station', 'baseline_cluster', 'baseline_label' ] ]
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

print( f'total stations in current scope: {len( all_stations )}' )
print( f'flagged stations in current scope: {len( flagged_keys )}' )
print( f'unflagged stations in current scope: {len( unflagged_stations )}' )

comparison_rows = unflagged_stations.head( 100 ).copy( )
comparison_rows

if len( comparison_rows ) == 0:
    print( 'no unflagged stations available after current filtering' )

else:
    timeline_start = pd.Timestamp( '1995-01-01' )
    timeline_end = pd.Timestamp( '2025-12-31' )
    all_dates = pd.date_range( timeline_start, timeline_end, freq = 'D' )

    comparison_rows[ 'row_label' ] = comparison_rows[ 'region' ].astype( str ) + ' | ' + comparison_rows[ 'station' ].astype( str )

    station_grid = ( 
        comparison_rows[ [ 'region', 'station', 'row_label' ] ]
        .assign( _tmp = 1 )
        .merge( pd.DataFrame( { 'date': all_dates, '_tmp': 1 } ), on = '_tmp', how = 'inner' )
        .drop( columns = [ '_tmp' ] )
    )

    raw_metric_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in fiveyear_vis.columns ]
    station_data_presence = ( 
        fiveyear_vis[ [ 'region', 'station', 'date' ] + raw_metric_cols ]
        .copy( )
        .assign( has_data = lambda frame: frame[ raw_metric_cols ].notna( ).any( axis = 1 ) if len( raw_metric_cols ) > 0 else True )
        .loc[ :, [ 'region', 'station', 'date', 'has_data' ] ]
    )

    p5_track = p5_vis[ [ 'region', 'station', 'date', 'implied_cluster' ] ].copy( )

    timeline = ( 
        station_grid
        .merge( station_data_presence, on = [ 'region', 'station', 'date' ], how = 'left' )
        .merge( p5_track, on = [ 'region', 'station', 'date' ], how = 'left' )
    )

    timeline[ 'has_data' ] = timeline[ 'has_data' ].fillna( False )

    timeline[ 'state' ] = -2
    timeline.loc[ timeline[ 'has_data' ], 'state' ] = -1
    timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'state' ] = timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'implied_cluster' ].astype( int )

    cluster_ids = sorted( cluster_station[ 'cluster' ].dropna( ).astype( int ).unique( ).tolist( ) )

    value_map = { -2: 0, -1: 1 }
    for idx, cid in enumerate( cluster_ids, start = 2 ):
        value_map[ int( cid ) ] = idx

    timeline[ 'cluster_plot' ] = timeline[ 'state' ].map( value_map )

    matrix = ( 
        timeline
        .pivot( index = 'row_label', columns = 'date', values = 'cluster_plot' )
        .reindex( comparison_rows[ 'row_label' ] )
        .to_numpy( )
    )

    palette = [ '#ffffff', '#bdbdbd' ] + sns.color_palette( 'tab10', n_colors = len( cluster_ids ) ).as_hex( )
    cmap = ListedColormap( palette )
    norm = BoundaryNorm( np.arange( -0.5, len( palette ) + 0.5, 1 ), cmap.N )

    fig_height = max( 8, 0.45 * len( comparison_rows ) + 2 )
    fig, ax = plt.subplots( figsize = ( 18, fig_height ) )
    ax.imshow( matrix, aspect = 'auto', interpolation = 'nearest', cmap = cmap, norm = norm )

    ax.set_yticks( np.arange( len( comparison_rows ) ) )
    ax.set_yticklabels( comparison_rows[ 'row_label' ] )

    baseline_values = comparison_rows[ 'baseline_cluster' ].apply( lambda val: value_map.get( int( val ), 0 ) if pd.notna( val ) else 0 )
    baseline_colors = [ palette[ int( val ) ] for val in baseline_values ]

    ax.scatter( 
        np.full( len( comparison_rows ), -4.0 ),
        np.arange( len( comparison_rows ) ),
        s = 110,
        c = baseline_colors,
        edgecolors = 'black',
        linewidths = 0.7,
        clip_on = False,
        zorder = 5,
    )
    ax.set_xlim( -8, matrix.shape[ 1 ] - 0.5 )

    tick_dates = pd.date_range( timeline_start, timeline_end, freq = '2YS' )
    tick_positions = [ np.searchsorted( all_dates.values, np.datetime64( dt ) ) for dt in tick_dates ]
    ax.set_xticks( tick_positions )
    ax.set_xticklabels( [ str( dt.year ) for dt in tick_dates ], rotation = 45, ha = 'right' )

    ax.set_xlabel( 'Date' )
    ax.set_ylabel( 'Unflagged Station ( Region | Station )' )
    ax.set_title( 'Unflagged Station Regime Timeline ( Daily, 1995-2025 )' )

    legend_handles = [ 
        mpatches.Patch( color = palette[ 0 ], label = 'No data' ),
        mpatches.Patch( color = palette[ 1 ], label = 'Data, no classification' ),
    ]

    for cid in cluster_ids:
        label_text = cluster_label_map.get( cid, f'Cluster {cid}' )
        legend_handles.append( mpatches.Patch( color = palette[ value_map[ cid ] ], label = label_text ) )

    legend_handles.append( mpatches.Patch( facecolor = 'white', edgecolor = 'black', label = 'Left dot = baseline regime' ) )

    ax.legend( handles = legend_handles, loc = 'upper left', bbox_to_anchor = ( 1.01, 1.0 ), frameon = True, title = 'Timeline State' )

    plt.tight_layout( )
    plt.show( )


Okay, but why is job+jb assigned a regime (red), but has no data showing?

In [ ]:
key = ( 'job', 'jb' )

print( 'in baseline:',
      ( ( cluster_station[ 'region' ] == key[ 0 ] ) & ( cluster_station[ 'station' ] == key[ 1 ] ) ).any( ) )

fw = fiveyear_water.loc[ 
    ( fiveyear_water[ 'region' ] == key[ 0 ] ) &
    ( fiveyear_water[ 'station' ] == key[ 1 ] )
]
print( 'fiveyear_water rows:', len( fw ) )

In [ ]:
# 5.3 diagnostic near 5.4: raw water-table metrics for job | jb
raw_region = 'job'
raw_station = 'jb'

raw_metric_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in water.columns ]

# baseline assignment (phase 1)
base_match = cluster_station.loc[ 
    ( cluster_station[ 'region' ] == raw_region )
    & ( cluster_station[ 'station' ] == raw_station ),
    [ 'cluster', 'cluster_label' ],
].drop_duplicates( )

if len( base_match ) == 0:
    print( 'baseline assignment: none found in cluster_station' )

else:
    print( 'baseline assignment:' )
    display( base_match )

# implied assignment coverage (phase 5)
p5_match = p5.loc[ 
    ( p5[ 'region' ] == raw_region )
    & ( p5[ 'station' ] == raw_station )
].copy( )
print( f'p5 rows for station: {len( p5_match )}' )
if len( p5_match ) > 0:
    print( 'implied-cluster non-null rows:', int( p5_match[ 'implied_cluster' ].notna( ).sum( ) ) )
    print( 'flip_confirmed rows:', int( p5_match[ 'flip_confirmed' ].fillna( False ).sum( ) ) )

job_raw = water.loc[ 
    ( water[ 'region' ] == raw_region )
    & ( water[ 'station' ] == raw_station )
].copy( )

print( f'raw rows in water before timestamp parsing: {len( job_raw )}' )

if len( job_raw ) == 0:
    print( 'no rows found in water for this station key' )

else:
    # choose the timestamp column that actually parses best for THIS station
    time_candidates = [ col for col in [ 'datetime', 'date' ] if col in job_raw.columns ]
    parsed_counts = { }
    parsed_series = { }

    for col in time_candidates:
        parsed = pd.to_datetime( job_raw[ col ], errors = 'coerce' )
        parsed_series[ col ] = parsed
        parsed_counts[ col ] = int( parsed.notna( ).sum( ) )

    if len( parsed_counts ) == 0:
        print( 'no datetime/date column found in water' )

    else:
        print( 'timestamp parse counts by column:', parsed_counts )
        time_col = max( parsed_counts, key = parsed_counts.get )
        print( f'time column used: {time_col}' )

        job_raw[ 'timestamp' ] = parsed_series[ time_col ]
        job_raw = job_raw.dropna( subset = [ 'timestamp' ] ).sort_values( 'timestamp' )

        window_start = pd.Timestamp( '1995-01-01' )
        window_end = pd.Timestamp( '2025-12-31 23:59:59' )
        job_raw = job_raw.loc[ job_raw[ 'timestamp' ].between( window_start, window_end ) ]

        print( f'rows in plotting window: {len( job_raw )}' )

        if len( job_raw ) == 0:
            print( 'no valid timestamped rows in 1995-2025 for this station' )

        else:
            non_null_counts = job_raw[ raw_metric_cols ].notna( ).sum( )
            print( 'non-null counts by metric:' )
            display( non_null_counts.to_frame( 'non_null_count' ) )

            if int( non_null_counts.sum( ) ) == 0:
                print( 'metrics are all NaN for this station in this window (chart will be blank)' )

            else:
                fig, ax = plt.subplots( figsize = ( 16, 5 ) )
                palette = sns.color_palette( 'tab10', n_colors = len( raw_metric_cols ) )

                for metric, color in zip( raw_metric_cols, palette ):
                    sub = job_raw.loc[ job_raw[ metric ].notna( ), [ 'timestamp', metric ] ]
                    ax.scatter( 
                        sub[ 'timestamp' ],
                        sub[ metric ],
                        s = 3,
                        alpha = 0.5,
                        color = color,
                        label = metric,
                    )

                ax.set_xlim( window_start, window_end )
                ax.set_xlabel( 'Date' )
                ax.set_ylabel( 'Raw Observed Value' )
                ax.set_title( f'Raw Water Time Series by Metric: {raw_region} | {raw_station} (1995-2025)' )
                ax.legend( loc = 'upper left', ncol = min( 3, len( raw_metric_cols ) ), frameon = True )
                plt.tight_layout( )
                plt.show( )


### 5.4 finalize modeling cohorts (data quality + holdout split)
- remove stations with no usable water signal (all-NaN across core water metrics)
- designate confirmed transitioning stations as a held-out validation set
- keep remaining stations as the primary training cohort


In [ ]:
# 5.4 - remove all-NaN stations and create train/holdout station cohorts

core_water_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in daily_water.columns ]

station_raw_quality = ( 
    daily_water
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( n_daily_rows = ( 'date', 'size' ) )
)

if len( core_water_cols ) > 0:
    raw_non_null_counts = ( 
        daily_water
        .groupby( [ 'region', 'station' ] )[ core_water_cols ]
        .apply( lambda frame: int( frame.notna( ).sum( ).sum( ) ) )
        .rename( 'n_non_null_core_water' )
        .reset_index( )
    )

else:
    raw_non_null_counts = station_raw_quality[ [ 'region', 'station' ] ].copy( )
    raw_non_null_counts[ 'n_non_null_core_water' ] = 0

station_raw_quality = station_raw_quality.merge( raw_non_null_counts, on = [ 'region', 'station' ], how = 'left' )
station_raw_quality[ 'has_raw_signal' ] = station_raw_quality[ 'n_non_null_core_water' ] > 0

valid_station_keys = station_raw_quality.loc[ station_raw_quality[ 'has_raw_signal' ], [ 'region', 'station' ] ].drop_duplicates( )
dropped_nan_station_keys = station_raw_quality.loc[ ~station_raw_quality[ 'has_raw_signal' ], [ 'region', 'station' ] ].drop_duplicates( )

print( f'total stations considered: {station_raw_quality.shape[ 0 ]}' )
print( f'stations with usable raw signal: {valid_station_keys.shape[ 0 ]}' )
print( f'stations dropped (all-NaN core water metrics): {dropped_nan_station_keys.shape[ 0 ]}' )

if dropped_nan_station_keys.shape[ 0 ] > 0:
    display( dropped_nan_station_keys.head( 20 ) )


def keep_valid_stations( frame ):
    return frame.merge( valid_station_keys, on = [ 'region', 'station' ], how = 'inner' )


daily_air_model = keep_valid_stations( daily_air )
daily_water_model = keep_valid_stations( daily_water )
fiveyear_water_model = keep_valid_stations( fiveyear_water )
p5_model = keep_valid_stations( p5 )
p5_monthly_model = keep_valid_stations( p5_monthly )

transition_station_keys = ( 
    p5_model
    .loc[ p5_model[ 'flip_confirmed' ].fillna( False ), [ 'region', 'station' ] ]
    .drop_duplicates( )
)

train_station_keys = ( 
    valid_station_keys
    .merge( transition_station_keys.assign( is_transition = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ 'is_transition' ].isna( ), [ 'region', 'station' ] ]
)

holdout_station_keys = transition_station_keys.copy( )


def subset_by_station_keys( frame, station_keys ):
    return frame.merge( station_keys, on = [ 'region', 'station' ], how = 'inner' )


daily_air_train = subset_by_station_keys( daily_air_model, train_station_keys )
daily_water_train = subset_by_station_keys( daily_water_model, train_station_keys )
fiveyear_water_train = subset_by_station_keys( fiveyear_water_model, train_station_keys )
p5_train = subset_by_station_keys( p5_model, train_station_keys )
p5_monthly_train = subset_by_station_keys( p5_monthly_model, train_station_keys )

daily_air_holdout = subset_by_station_keys( daily_air_model, holdout_station_keys )
daily_water_holdout = subset_by_station_keys( daily_water_model, holdout_station_keys )
fiveyear_water_holdout = subset_by_station_keys( fiveyear_water_model, holdout_station_keys )
p5_holdout = subset_by_station_keys( p5_model, holdout_station_keys )
p5_monthly_holdout = subset_by_station_keys( p5_monthly_model, holdout_station_keys )

print( f'train stations: {train_station_keys.shape[ 0 ]}' )
print( f'holdout (transition) stations: {holdout_station_keys.shape[ 0 ]}' )
print( f'training rows (daily_air, daily_water, p5): {daily_air_train.shape[ 0 ]}, {daily_water_train.shape[ 0 ]}, {p5_train.shape[ 0 ]}' )
print( f'holdout rows (daily_air, daily_water, p5): {daily_air_holdout.shape[ 0 ]}, {daily_water_holdout.shape[ 0 ]}, {p5_holdout.shape[ 0 ]}' )



In [ ]:
# 5.4 progress report: post-filter flagged vs unflagged timelines
# shows both cohorts under current 5.4 filtering in one run

from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

core_water_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in daily_water.columns ]

station_raw_quality = ( 
    daily_water
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( n_daily_rows = ( 'date', 'size' ) )
)

if len( core_water_cols ) > 0:
    raw_non_null_counts = ( 
        daily_water
        .groupby( [ 'region', 'station' ] )[ core_water_cols ]
        .apply( lambda frame: int( frame.notna( ).sum( ).sum( ) ) )
        .rename( 'n_non_null_core_water' )
        .reset_index( )
    )

else:
    raw_non_null_counts = station_raw_quality[ [ 'region', 'station' ] ].copy( )
    raw_non_null_counts[ 'n_non_null_core_water' ] = 0

station_raw_quality = station_raw_quality.merge( raw_non_null_counts, on = [ 'region', 'station' ], how = 'left' )
station_raw_quality[ 'has_raw_signal' ] = station_raw_quality[ 'n_non_null_core_water' ] > 0
valid_station_keys_vis = station_raw_quality.loc[ station_raw_quality[ 'has_raw_signal' ], [ 'region', 'station' ] ].drop_duplicates( )


def keep_valid_for_vis( frame ):
    return frame.merge( valid_station_keys_vis, on = [ 'region', 'station' ], how = 'inner' )

p5_vis = keep_valid_for_vis( p5 )
fiveyear_vis = keep_valid_for_vis( fiveyear_water )

station_baseline_regime = ( 
    cluster_station[ [ 'region', 'station', 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'region', 'station' ] )
    .rename( columns = { 
        'cluster': 'baseline_cluster',
        'cluster_label': 'baseline_label',
    } )
)

cluster_label_map = ( 
    cluster_station[ [ 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'cluster' ] )
    .set_index( 'cluster' )[ 'cluster_label' ]
    .to_dict( )
)

all_stations = valid_station_keys_vis.merge( station_baseline_regime, on = [ 'region', 'station' ], how = 'left' )
flagged_keys = p5_vis.loc[ p5_vis[ 'flip_confirmed' ].fillna( False ), [ 'region', 'station' ] ].drop_duplicates( )

unflagged_keys = ( 
    all_stations[ [ 'region', 'station', 'baseline_cluster', 'baseline_label' ] ]
    .merge( flagged_keys.assign( is_flagged = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ 'is_flagged' ].isna( ), [ 'region', 'station', 'baseline_cluster', 'baseline_label' ] ]
)

print( f'total stations in scope: {len( all_stations )}' )
print( f'flagged stations in scope: {len( flagged_keys )}' )
print( f'unflagged stations in scope: {len( unflagged_keys )}' )

raw_metric_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in fiveyear_vis.columns ]
cluster_ids = sorted( cluster_station[ 'cluster' ].dropna( ).astype( int ).unique( ).tolist( ) )
value_map = { -2: 0, -1: 1 }
for idx, cid in enumerate( cluster_ids, start = 2 ):
    value_map[ int( cid ) ] = idx

palette = [ '#ffffff', '#bdbdbd' ] + sns.color_palette( 'tab10', n_colors = len( cluster_ids ) ).as_hex( )
cmap = ListedColormap( palette )
norm = BoundaryNorm( np.arange( -0.5, len( palette ) + 0.5, 1 ), cmap.N )


def render_timeline( station_df, title, y_label, max_rows = 60 ):
    station_df = station_df.copy( )
    station_df = station_df.head( max_rows ).reset_index( drop = True )

    if len( station_df ) == 0:
        print( f'{title}: no stations to show' )
        return

    station_df[ 'row_label' ] = station_df[ 'region' ].astype( str ) + ' | ' + station_df[ 'station' ].astype( str )

    timeline_start = pd.Timestamp( '1995-01-01' )
    timeline_end = pd.Timestamp( '2025-12-31' )
    all_dates = pd.date_range( timeline_start, timeline_end, freq = 'D' )

    station_grid = ( 
        station_df[ [ 'region', 'station', 'row_label' ] ]
        .assign( _tmp = 1 )
        .merge( pd.DataFrame( { 'date': all_dates, '_tmp': 1 } ), on = '_tmp', how = 'inner' )
        .drop( columns = [ '_tmp' ] )
    )

    station_data_presence = ( 
        fiveyear_vis[ [ 'region', 'station', 'date' ] + raw_metric_cols ]
        .copy( )
        .assign( has_data = lambda frame: frame[ raw_metric_cols ].notna( ).any( axis = 1 ) if len( raw_metric_cols ) > 0 else True )
        .loc[ :, [ 'region', 'station', 'date', 'has_data' ] ]
    )

    p5_track = p5_vis[ [ 'region', 'station', 'date', 'implied_cluster' ] ].copy( )

    timeline = ( 
        station_grid
        .merge( station_data_presence, on = [ 'region', 'station', 'date' ], how = 'left' )
        .merge( p5_track, on = [ 'region', 'station', 'date' ], how = 'left' )
    )

    timeline[ 'has_data' ] = timeline[ 'has_data' ].fillna( False )
    timeline[ 'state' ] = -2
    timeline.loc[ timeline[ 'has_data' ], 'state' ] = -1
    timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'state' ] = timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'implied_cluster' ].astype( int )
    timeline[ 'cluster_plot' ] = timeline[ 'state' ].map( value_map )

    matrix = ( 
        timeline
        .pivot( index = 'row_label', columns = 'date', values = 'cluster_plot' )
        .reindex( station_df[ 'row_label' ] )
        .to_numpy( )
    )

    fig_height = max( 6, 0.42 * len( station_df ) + 2 )
    fig, ax = plt.subplots( figsize = ( 18, fig_height ) )
    ax.imshow( matrix, aspect = 'auto', interpolation = 'nearest', cmap = cmap, norm = norm )

    ax.set_yticks( np.arange( len( station_df ) ) )
    ax.set_yticklabels( station_df[ 'row_label' ] )

    baseline_values = station_df[ 'baseline_cluster' ].apply( lambda val: value_map.get( int( val ), 0 ) if pd.notna( val ) else 0 )
    baseline_colors = [ palette[ int( val ) ] for val in baseline_values ]
    ax.scatter( 
        np.full( len( station_df ), -4.0 ),
        np.arange( len( station_df ) ),
        s = 105,
        c = baseline_colors,
        edgecolors = 'black',
        linewidths = 0.7,
        clip_on = False,
        zorder = 5,
    )
    ax.set_xlim( -8, matrix.shape[ 1 ] - 0.5 )

    tick_dates = pd.date_range( timeline_start, timeline_end, freq = '2YS' )
    tick_positions = [ np.searchsorted( all_dates.values, np.datetime64( dt ) ) for dt in tick_dates ]
    ax.set_xticks( tick_positions )
    ax.set_xticklabels( [ str( dt.year ) for dt in tick_dates ], rotation = 45, ha = 'right' )

    ax.set_xlabel( 'Date' )
    ax.set_ylabel( y_label )
    ax.set_title( title )

    legend_handles = [ 
        mpatches.Patch( color = palette[ 0 ], label = 'No data' ),
        mpatches.Patch( color = palette[ 1 ], label = 'Data, no classification' ),
    ]
    for cid in cluster_ids:
        legend_handles.append( mpatches.Patch( color = palette[ value_map[ cid ] ], label = cluster_label_map.get( cid, f'Cluster {cid}' ) ) )
    legend_handles.append( mpatches.Patch( facecolor = 'white', edgecolor = 'black', label = 'Left dot = baseline regime' ) )

    ax.legend( handles = legend_handles, loc = 'upper left', bbox_to_anchor = ( 1.01, 1.0 ), frameon = True, title = 'Timeline State' )
    plt.tight_layout( )
    plt.show( )


flagged_rows = flagged_keys.merge( station_baseline_regime, on = [ 'region', 'station' ], how = 'left' )
render_timeline( flagged_rows, 'Progress Report: Flagged Stations ( Post-5.4 )', 'Flagged Station ( Region | Station )', max_rows = 60 )
render_timeline( unflagged_keys, 'Progress Report: Unflagged Stations ( Post-5.4 )', 'Unflagged Station ( Region | Station )', max_rows = 100 )



In [ ]:
# 5.5 - drop never-classified stations and rebuild final flagged/unflagged cohorts

p5_scope = p5_model
daily_air_scope = daily_air_model
daily_water_scope = daily_water_model
fiveyear_water_scope = fiveyear_water_model
valid_station_scope = valid_station_keys.copy( )

station_classification_quality = ( 
    p5_scope
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_rows = ( 'date', 'size' ),
        n_classified_days = ( 'implied_cluster', lambda values: int( values.notna( ).sum( ) ) ),
        n_flip_days = ( 'flip_confirmed', lambda values: int( values.fillna( False ).sum( ) ) ),
    )
)

classifiable_station_keys = station_classification_quality.loc[ 
    station_classification_quality[ 'n_classified_days' ] > 0,
    [ 'region', 'station' ],
].drop_duplicates( )

dropped_never_classified_station_keys = ( 
    valid_station_scope
    .merge( classifiable_station_keys.assign( is_classifiable = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ 'is_classifiable' ].isna( ), [ 'region', 'station' ] ]
)

print( f'valid stations entering 5.5: {len( valid_station_scope )}' )
print( f'classifiable stations kept: {len( classifiable_station_keys )}' )
print( f'dropped never-classified stations: {len( dropped_never_classified_station_keys )}' )

if len( dropped_never_classified_station_keys ) > 0:
    display( dropped_never_classified_station_keys.head( 20 ) )

flagged_station_keys_final = holdout_station_keys.merge( classifiable_station_keys, on = [ 'region', 'station' ], how = 'inner' )

unflagged_station_keys_final = ( 
    classifiable_station_keys
    .merge( flagged_station_keys_final.assign( is_flagged = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ 'is_flagged' ].isna( ), [ 'region', 'station' ] ]
)


def subset_by_station_keys( frame, station_keys ):
    return frame.merge( station_keys, on = [ 'region', 'station' ], how = 'inner' )


# this is the final scope of stations for all datasets, including both train and holdouts, after all 5.5 filtering
# used in phase 6
daily_air_final = subset_by_station_keys( daily_air_scope, classifiable_station_keys )
daily_water_final = subset_by_station_keys( daily_water_scope, classifiable_station_keys )
fiveyear_water_final = subset_by_station_keys( fiveyear_water_scope, classifiable_station_keys )
p5_final = subset_by_station_keys( p5_scope, classifiable_station_keys )
p5_monthly_final = subset_by_station_keys( p5_monthly_model, classifiable_station_keys )

daily_air_train_final = subset_by_station_keys( daily_air_scope, unflagged_station_keys_final )
daily_water_train_final = subset_by_station_keys( daily_water_scope, unflagged_station_keys_final )
fiveyear_water_train_final = subset_by_station_keys( fiveyear_water_scope, unflagged_station_keys_final )
p5_train_final = subset_by_station_keys( p5_scope, unflagged_station_keys_final )
p5_monthly_train_final = subset_by_station_keys( p5_monthly_model, unflagged_station_keys_final )

daily_air_holdout_final = subset_by_station_keys( daily_air_scope, flagged_station_keys_final )
daily_water_holdout_final = subset_by_station_keys( daily_water_scope, flagged_station_keys_final )
fiveyear_water_holdout_final = subset_by_station_keys( fiveyear_water_scope, flagged_station_keys_final )
p5_holdout_final = subset_by_station_keys( p5_scope, flagged_station_keys_final )
p5_monthly_holdout_final = subset_by_station_keys( p5_monthly_model, flagged_station_keys_final )

print( f'final unflagged/train stations: {len( unflagged_station_keys_final )}' )
print( f'final flagged/holdout stations: {len( flagged_station_keys_final )}' )
print( f'final training rows (daily_air, daily_water, p5): {len( daily_air_train_final )}, {len( daily_water_train_final )}, {len( p5_train_final )}' )
print( f'final holdout rows (daily_air, daily_water, p5): {len( daily_air_holdout_final )}, {len( daily_water_holdout_final )}, {len( p5_holdout_final )}' )


## Phase 6 — Model Development: Air → Water Temperature
Goal: predict ΔT_water given atmospheric forcing history

# also stage 1

- 6.1 establish naive baselines
  - persistence forecast and climatological mean by day-of-year
  - record RMSE and MAE
- 6.2 train ridge/lasso regression using lagged atmospheric features
  - interpret coefficients as a sanity check
- 6.3 train gradient boosted model (XGBoost or LightGBM) on same features
  - compare to linear baseline
- 6.4 use walk-forward temporal cross-validation
  - train on years 1–N, test on N+1
  - since reading suggests standard k-fold doesn't work well with this kind of job
- 6.5 evaluate on held-out transitioning stations from phase 5
  - does the model generalize across space?
- 6.6 select best model, document feature importances
  - lag windows that dominate are a reportable finding

## Note

- for “unknown estuary” inference, use 3 modes:
  - air-only (weakest, fallback).
  - air + estuary context (recommended): recent observed water means/variability, depth, season.
  - air + context + short calibration (best): fit a simple per-estuary intercept correction on first window.

In [ ]:
# 6.0 - define inference contract + build a lightweight estuary-context table
# assumes phase 5 final already exist in the notebook session

context_source_water = daily_water_final.copy( )
context_source_water[ 'date' ] = pd.to_datetime( context_source_water[ 'date' ], errors = 'coerce' )
context_source_water = context_source_water.dropna( subset = [ 'date' ] )

context_metric_candidates = [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ]
context_metrics = [ col for col in context_metric_candidates if col in context_source_water.columns ]

# for the inference contract, we want to provide recent historical context for each station-date
# in the form of simple rolling window aggregate of water 'state'
recent_window_days = 60
recent_context_rows = ( 
    context_source_water
    .sort_values( [ 'region', 'station', 'date' ] )
    .groupby( [ 'region', 'station' ], group_keys = False )
    .tail( recent_window_days )
    .copy( )
)

context_agg = { }
for col in context_metrics:
    context_agg[ f'ctx_{col}_mean' ] = ( col, 'mean' )
    context_agg[ f'ctx_{col}_std' ] = ( col, 'std' )

phase6_context_table = ( 
    recent_context_rows
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( **context_agg )
)

phase6_air_feature_candidates = [ 
    'air_temp_r1d',
    'air_temp_r7d',
    'air_temp_r28d',
    'wind_speed_r1d',
    'wind_speed_r7d',
    'wind_speed_r28d',
    'precip_r1d',
    'precip_r7d',
    'precip_r28d',
    'solar_r1d',
    'solar_r7d',
    'solar_r28d',
    'air_temp',
    'wind_speed',
    'precip',
    'solar',
    'doy_sin',
    'doy_cos',
]

phase6_context_feature_candidates = [ 
    'water_temp_baseline',
    'salinity_baseline',
    'oxygen_baseline',
    'ph_baseline',
    'depth_baseline',
] + [ col for col in phase6_context_table.columns if col.startswith( 'ctx_' ) ]

phase6_inference_contract = { 
    'required_keys': [ 'region', 'station', 'date' ],
    'required_air_features': phase6_air_feature_candidates,
    'optional_estuary_context': phase6_context_feature_candidates,
    'target': 'delta_water_temp',
    'absolute_reconstruction': 'water_temp_baseline + predicted_delta_water_temp',
}

print( f"phase6 context rows: {len( phase6_context_table )}" )
print( 'phase6 inference contract:' )
for key, value in phase6_inference_contract.items( ):
    print( f'  - {key}: {value}' )

display( phase6_context_table.sample( 10 ).T )


In [ ]:
# 6.1 to 6.3 - small first pass for delta-water-temp modeling
# assumes phase 5 final tables are present and this notebook is re-run top to bottom

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

air_train_source = daily_air_train_final.copy( )
water_train_source = daily_water_train_final.copy( )
air_holdout_source = daily_air_holdout_final.copy( )
water_holdout_source = daily_water_holdout_final.copy( )

base_join_cols = [ 'region', 'station', 'date' ]
water_keep_cols = [ 
    'delta_water_temp',
    'water_temp_baseline',
    'salinity_baseline',
    'oxygen_baseline',
    'ph_baseline',
    'depth_baseline',
]

phase6_train = air_train_source.merge( 
    water_train_source[ base_join_cols + water_keep_cols ],
    on = base_join_cols,
    how = 'inner',
)
phase6_holdout = air_holdout_source.merge( 
    water_holdout_source[ base_join_cols + water_keep_cols ],
    on = base_join_cols,
    how = 'inner',
)

# get the context features into the train and holdout tables via left join
phase6_train = phase6_train.merge( phase6_context_table, on = [ 'region', 'station' ], how = 'left' )
phase6_holdout = phase6_holdout.merge( phase6_context_table, on = [ 'region', 'station' ], how = 'left' )

phase6_train[ 'date' ] = pd.to_datetime( phase6_train[ 'date' ], errors = 'coerce' )
phase6_holdout[ 'date' ] = pd.to_datetime( phase6_holdout[ 'date' ], errors = 'coerce' )
phase6_train[ 'doy' ] = phase6_train[ 'date' ].dt.dayofyear
phase6_holdout[ 'doy' ] = phase6_holdout[ 'date' ].dt.dayofyear

target_col = 'delta_water_temp'
phase6_train_model = phase6_train.dropna( subset = [ target_col ] ).copy( )
phase6_holdout_model = phase6_holdout.dropna( subset = [ target_col ] ).copy( )

air_features = [ col for col in phase6_air_feature_candidates if col in phase6_train_model.columns ]
context_features = [ col for col in phase6_context_feature_candidates if col in phase6_train_model.columns ]
joint_features = list( dict.fromkeys( air_features + context_features ) )

print( f'train rows: {len( phase6_train_model )}' )
print( f'holdout rows: {len( phase6_holdout_model )}' )
print( f'air features ({len( air_features )}): {air_features}' )
print( f'context features ({len( context_features )}): {context_features}' )

# naive baseline 1: persistence by station (previous observed day)
phase6_holdout_model = phase6_holdout_model.sort_values( [ 'region', 'station', 'date' ] )
phase6_holdout_model[ 'pred_persistence' ] = phase6_holdout_model.groupby( [ 'region', 'station' ] )[ target_col ].shift( 1 )

# naive baseline 2: climatology by day-of-year from train set
clim_by_doy = ( 
    phase6_train_model
    .groupby( 'doy', as_index = False )[ target_col ]
    .median( )
    .rename( columns = { target_col: 'pred_climatology' } )
)

phase6_holdout_model[ 'pred_climatology' ] = phase6_holdout_model[ 'doy' ].map( clim_by_doy.set_index( 'doy' )[ 'pred_climatology' ] )
global_target_median = float( phase6_train_model[ target_col ].median( ) )
phase6_holdout_model[ 'pred_persistence' ] = phase6_holdout_model[ 'pred_persistence' ].fillna( phase6_holdout_model[ 'pred_climatology' ] )
phase6_holdout_model[ 'pred_persistence' ] = phase6_holdout_model[ 'pred_persistence' ].fillna( global_target_median )
phase6_holdout_model[ 'pred_climatology' ] = phase6_holdout_model[ 'pred_climatology' ].fillna( global_target_median )

phase6_scores_rows = [ ]

y_holdout = phase6_holdout_model[ target_col ]
pred = phase6_holdout_model[ 'pred_persistence' ]
phase6_scores_rows.append( { 
    'model': 'naive_persistence',
    'mae': float( mean_absolute_error( y_holdout, pred ) ),
    'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
    'r2': float( r2_score( y_holdout, pred ) ),
} )

pred = phase6_holdout_model[ 'pred_climatology' ]
phase6_scores_rows.append( { 
    'model': 'naive_climatology_doy',
    'mae': float( mean_absolute_error( y_holdout, pred ) ),
    'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
    'r2': float( r2_score( y_holdout, pred ) ),
} )

# model 1: ridge on air-only features
X_train_air = phase6_train_model[ air_features ].copy( )
X_holdout_air = phase6_holdout_model[ air_features ].copy( )

air_fill_values = X_train_air.median( numeric_only = True )
X_train_air = X_train_air.fillna( air_fill_values )
X_holdout_air = X_holdout_air.fillna( air_fill_values )

y_train = phase6_train_model[ target_col ]
ridge_air_model = Ridge( alpha = 1.0 )
ridge_air_model.fit( X_train_air, y_train )
phase6_holdout_model[ 'pred_ridge_air_only' ] = ridge_air_model.predict( X_holdout_air )

pred = phase6_holdout_model[ 'pred_ridge_air_only' ]
phase6_scores_rows.append( { 
    'model': 'ridge_air_only',
    'mae': float( mean_absolute_error( y_holdout, pred ) ),
    'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
    'r2': float( r2_score( y_holdout, pred ) ),
} )

# model 2: ridge on air + estuary context
X_train_joint = phase6_train_model[ joint_features ].copy( )
X_holdout_joint = phase6_holdout_model[ joint_features ].copy( )

joint_fill_values = X_train_joint.median( numeric_only = True )
X_train_joint = X_train_joint.fillna( joint_fill_values )
X_holdout_joint = X_holdout_joint.fillna( joint_fill_values )

ridge_air_context_model = Ridge( alpha = 1.0 )
ridge_air_context_model.fit( X_train_joint, y_train )
phase6_holdout_model[ 'pred_ridge_air_context' ] = ridge_air_context_model.predict( X_holdout_joint )

pred = phase6_holdout_model[ 'pred_ridge_air_context' ]
phase6_scores_rows.append( { 
    'model': 'ridge_air_context',
    'mae': float( mean_absolute_error( y_holdout, pred ) ),
    'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
    'r2': float( r2_score( y_holdout, pred ) ),
} )

# model 3: small gradient boosting pass on same joint features
hgb_air_context_model = HistGradientBoostingRegressor( 
    learning_rate = 0.05,
    max_depth = 6,
    max_iter = 250,
    random_state = 42,
)
hgb_air_context_model.fit( X_train_joint, y_train )
phase6_holdout_model[ 'pred_hgb_air_context' ] = hgb_air_context_model.predict( X_holdout_joint )

pred = phase6_holdout_model[ 'pred_hgb_air_context' ]
phase6_scores_rows.append( { 
    'model': 'hgb_air_context',
    'mae': float( mean_absolute_error( y_holdout, pred ) ),
    'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
    'r2': float( r2_score( y_holdout, pred ) ),
} )

phase6_scores = pd.DataFrame( phase6_scores_rows ).sort_values( 'rmse', ascending = True ).reset_index( drop = True )
phase6_scores_operational = phase6_scores.copy( )
phase6_scores_scenario = phase6_scores.loc[ ~phase6_scores[ 'model' ].isin( [ 'naive_persistence' ] ) ].reset_index( drop = True )

print( '\nphase 6 leaderboard (operational, includes persistence):' )
display( phase6_scores_operational )
print( 'phase 6 leaderboard (scenario, excludes persistence):' )
display( phase6_scores_scenario )

phase6_selected_model = hgb_air_context_model
phase6_selected_features = joint_features.copy( )
phase6_selected_name = 'hgb_air_context'

print( f'selected phase6 model for scenario demos: {phase6_selected_name}' )
print( f'selected feature count: {len( phase6_selected_features )}' )

# tiny unseen-estuary style demo: warm one row by +2 C in air-temp features
demo_cols = [ 'region', 'station', 'date', target_col ] + phase6_selected_features
phase6_demo_row = phase6_holdout_model[ demo_cols ].dropna( subset = phase6_selected_features ).head( 1 ).copy( )

phase6_demo_warm = phase6_demo_row.copy( )
warm_cols = [ col for col in phase6_selected_features if col.startswith( 'air_temp' ) ]
for col in warm_cols:
    phase6_demo_warm[ col ] = phase6_demo_warm[ col ] + 2.0

baseline_pred = float( phase6_selected_model.predict( phase6_demo_row[ phase6_selected_features ] )[ 0 ] )
warm_pred = float( phase6_selected_model.predict( phase6_demo_warm[ phase6_selected_features ] )[ 0 ] )

print( '\nunknown-estuary style +2 C quick demo:' )
print( phase6_demo_row[ [ 'region', 'station', 'date', target_col ] ] )
print( f'baseline predicted delta_water_temp: {baseline_pred:.3f}' )
print( f'+2 C predicted delta_water_temp: {warm_pred:.3f}' )
print( f'implied delta shift from warming: {warm_pred - baseline_pred:.3f}' )


In [ ]:
# naive baseline 1: persistence by station (previous observed day)
phase6_holdout_model = phase6_holdout_model.sort_values( [ 'region', 'station', 'date' ] )
phase6_holdout_model[ 'pred_persistence' ] = phase6_holdout_model.groupby( [ 'region', 'station' ] )[ target_col ].shift( 1 )

# naive baseline 2: climatology by day-of-year from train set
clim_by_doy = ( 
    phase6_train_model
    .groupby( 'doy', as_index = False )[ target_col ]
    .median( )
    .rename( columns = { target_col: 'pred_climatology' } )
)

phase6_holdout_model[ 'pred_climatology' ] = phase6_holdout_model[ 'doy' ].map( clim_by_doy.set_index( 'doy' )[ 'pred_climatology' ] )
global_target_median = float( phase6_train_model[ target_col ].median( ) )
phase6_holdout_model[ 'pred_persistence' ] = phase6_holdout_model[ 'pred_persistence' ].fillna( phase6_holdout_model[ 'pred_climatology' ] )
phase6_holdout_model[ 'pred_persistence' ] = phase6_holdout_model[ 'pred_persistence' ].fillna( global_target_median )
phase6_holdout_model[ 'pred_climatology' ] = phase6_holdout_model[ 'pred_climatology' ].fillna( global_target_median )

phase6_scores_rows = [ ]

y_holdout = phase6_holdout_model[ target_col ]
pred = phase6_holdout_model[ 'pred_persistence' ]
phase6_scores_rows.append( { 
    'model': 'naive_persistence',
    'mae': float( mean_absolute_error( y_holdout, pred ) ),
    'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
    'r2': float( r2_score( y_holdout, pred ) ),
} )

pred = phase6_holdout_model[ 'pred_climatology' ]
phase6_scores_rows.append( { 
    'model': 'naive_climatology_doy',
    'mae': float( mean_absolute_error( y_holdout, pred ) ),
    'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
    'r2': float( r2_score( y_holdout, pred ) ),
} )

# model 1: ridge on air-only features
X_train_air = phase6_train_model[ air_features ].copy( )
X_holdout_air = phase6_holdout_model[ air_features ].copy( )

air_fill_values = X_train_air.median( numeric_only = True )
X_train_air = X_train_air.fillna( air_fill_values )
X_holdout_air = X_holdout_air.fillna( air_fill_values )

y_train = phase6_train_model[ target_col ]
ridge_air_model = Ridge( alpha = 1.0 )
ridge_air_model.fit( X_train_air, y_train )
phase6_holdout_model[ 'pred_ridge_air_only' ] = ridge_air_model.predict( X_holdout_air )

pred = phase6_holdout_model[ 'pred_ridge_air_only' ]
phase6_scores_rows.append( { 
    'model': 'ridge_air_only',
    'mae': float( mean_absolute_error( y_holdout, pred ) ),
    'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
    'r2': float( r2_score( y_holdout, pred ) ),
} )

phase6_selected_model = ridge_air_model
phase6_selected_features = air_features.copy( )
phase6_selected_name = 'ridge_air_only'

# model 2: ridge on air + estuary context (if context exists)
if len( joint_features ) > len( air_features ):
    X_train_joint = phase6_train_model[ joint_features ].copy( )
    X_holdout_joint = phase6_holdout_model[ joint_features ].copy( )

    joint_fill_values = X_train_joint.median( numeric_only = True )
    X_train_joint = X_train_joint.fillna( joint_fill_values )
    X_holdout_joint = X_holdout_joint.fillna( joint_fill_values )

    ridge_air_context_model = Ridge( alpha = 1.0 )
    ridge_air_context_model.fit( X_train_joint, y_train )
    phase6_holdout_model[ 'pred_ridge_air_context' ] = ridge_air_context_model.predict( X_holdout_joint )

    pred = phase6_holdout_model[ 'pred_ridge_air_context' ]
    phase6_scores_rows.append( { 
        'model': 'ridge_air_context',
        'mae': float( mean_absolute_error( y_holdout, pred ) ),
        'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
        'r2': float( r2_score( y_holdout, pred ) ),
    } )

    # model 3: small gradient boosting pass on same joint features
    hgb_air_context_model = HistGradientBoostingRegressor( 
        learning_rate = 0.05,
        max_depth = 6,
        max_iter = 250,
        random_state = 42,
    )
    hgb_air_context_model.fit( X_train_joint, y_train )
    phase6_holdout_model[ 'pred_hgb_air_context' ] = hgb_air_context_model.predict( X_holdout_joint )

    pred = phase6_holdout_model[ 'pred_hgb_air_context' ]
    phase6_scores_rows.append( { 
        'model': 'hgb_air_context',
        'mae': float( mean_absolute_error( y_holdout, pred ) ),
        'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
        'r2': float( r2_score( y_holdout, pred ) ),
    } )

    phase6_selected_model = hgb_air_context_model
    phase6_selected_features = joint_features.copy( )
    phase6_selected_name = 'hgb_air_context'

phase6_scores = pd.DataFrame( phase6_scores_rows ).sort_values( 'rmse', ascending = True ).reset_index( drop = True )
phase6_scores_operational = phase6_scores.copy( )
phase6_scores_scenario = phase6_scores.loc[ ~phase6_scores[ 'model' ].isin( [ 'naive_persistence' ] ) ].reset_index( drop = True )

print( '\nphase 6 leaderboard (operational, includes persistence):' )
display( phase6_scores_operational )
print( 'phase 6 leaderboard (scenario, excludes persistence):' )
display( phase6_scores_scenario )

print( f'selected phase6 model for scenario demos: {phase6_selected_name}' )
print( f'selected feature count: {len( phase6_selected_features )}' )


In [ ]:
# tiny unseen-estuary style demo: warm one row by +2 C in air-temp features
demo_cols = [ 'region', 'station', 'date', target_col ] + phase6_selected_features
phase6_demo_row = phase6_holdout_model[ demo_cols ].dropna( subset = phase6_selected_features ).head( 1 ).copy( )

phase6_demo_warm = phase6_demo_row.copy( )
warm_cols = [ col for col in phase6_selected_features if col.startswith( 'air_temp' ) ]
for col in warm_cols:
    phase6_demo_warm[ col ] = phase6_demo_warm[ col ] + 2.0

baseline_pred = float( phase6_selected_model.predict( phase6_demo_row[ phase6_selected_features ] )[ 0 ] )
warm_pred = float( phase6_selected_model.predict( phase6_demo_warm[ phase6_selected_features ] )[ 0 ] )

print( '\nunknown-estuary style +2 C quick demo:' )
print( phase6_demo_row[ [ 'region', 'station', 'date', target_col ] ] )
print( f'baseline predicted delta_water_temp: {baseline_pred:.3f}' )
print( f'+2 C predicted delta_water_temp: {warm_pred:.3f}' )
print( f'implied delta shift from warming: {warm_pred - baseline_pred:.3f}' )

In [ ]:
# 6.4 diagnostic: typical air-water spread baseline vs selected model spread
# spread is defined here as ( water_temp - air_temp )
# water_temp is reconstructed from baseline + predicted/observed delta

selected_pred_col_map = { 
    'ridge_air_only': 'pred_ridge_air_only',
    'ridge_air_context': 'pred_ridge_air_context',
    'hgb_air_context': 'pred_hgb_air_context',
}
selected_pred_col = selected_pred_col_map.get( phase6_selected_name )

# for this diagnostic, we want to see how the selected model's predicted spread 
# compares to the observed spread, and how both compare to a simple typical spread baseline
spread_train = phase6_train_model[ [ 
    'region',
    'station',
    'doy',
    'air_temp',
    'water_temp_baseline',
    target_col,
] ].copy( )
spread_train = spread_train.dropna( subset = [ 'air_temp', 'water_temp_baseline', target_col ] )
spread_train[ 'water_temp_obs' ] = spread_train[ 'water_temp_baseline' ] + spread_train[ target_col ]
spread_train[ 'spread_obs' ] = spread_train[ 'water_temp_obs' ] - spread_train[ 'air_temp' ]

# we can define typical spread baselines at multiple levels of granularity 
#  (station-doy, station, region-doy, doy, global)
# which will be useful for understanding where the model adds value
typical_spread_station_doy = ( 
    spread_train
    .groupby( [ 'region', 'station', 'doy' ], as_index = False )[ 'spread_obs' ]
    .median( )
    .rename( columns = { 'spread_obs': 'typical_spread_station_doy' } )
)

# station-level typical spread (no doy breakdown) ... e.g., day of year
typical_spread_station = ( 
    spread_train
    .groupby( [ 'region', 'station' ], as_index = False )[ 'spread_obs' ]
    .median( )
    .rename( columns = { 'spread_obs': 'typical_spread_station' } )
)

# region-doy level typical spread
typical_spread_region_doy = ( 
    spread_train
    .groupby( [ 'region', 'doy' ], as_index = False )[ 'spread_obs' ]
    .median( )
    .rename( columns = { 'spread_obs': 'typical_spread_region_doy' } )
)

# doy-level typical spread (no region breakdown) ... re: seasonal cycle
typical_spread_doy = ( 
    spread_train
    .groupby( 'doy', as_index = False )[ 'spread_obs' ]
    .median( )
    .rename( columns = { 'spread_obs': 'typical_spread_doy' } )
)

typical_spread_global = float( spread_train[ 'spread_obs' ].median( ) )

# for this diagnostic, scope is the phase-6 holdout table ( flagged transition stations from phase 5 )
# if this looks like fewer stations than expected, check the printed station-count trace below
spread_holdout_pre = phase6_holdout_model[ [ 
    'region',
    'station',
    'date',
    'doy',
    'air_temp',
    'water_temp_baseline',
    target_col,
    selected_pred_col,
] ].copy( )

holdout_station_keys_pre = spread_holdout_pre[ [ 'region', 'station' ] ].drop_duplicates( )
spread_holdout = spread_holdout_pre.dropna( subset = [ 'air_temp', 'water_temp_baseline', target_col, selected_pred_col ] )
holdout_station_keys_post = spread_holdout[ [ 'region', 'station' ] ].drop_duplicates( )

dropped_spread_station_keys = ( 
    holdout_station_keys_pre
    .merge( holdout_station_keys_post.assign( _kept = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ '_kept' ].isna( ), [ 'region', 'station' ] ]
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

# reconstruct observed and predicted water temp, then spread
spread_holdout[ 'water_temp_obs' ] = spread_holdout[ 'water_temp_baseline' ] + spread_holdout[ target_col ]
spread_holdout[ 'water_temp_pred' ] = spread_holdout[ 'water_temp_baseline' ] + spread_holdout[ selected_pred_col ]
spread_holdout[ 'spread_obs' ] = spread_holdout[ 'water_temp_obs' ] - spread_holdout[ 'air_temp' ]
spread_holdout[ 'spread_pred_model' ] = spread_holdout[ 'water_temp_pred' ] - spread_holdout[ 'air_temp' ]

# now merge on the various typical spread baselines,
spread_holdout = spread_holdout.merge( typical_spread_station_doy, on = [ 'region', 'station', 'doy' ], how = 'left' )
spread_holdout = spread_holdout.merge( typical_spread_station, on = [ 'region', 'station' ], how = 'left' )
spread_holdout = spread_holdout.merge( typical_spread_region_doy, on = [ 'region', 'doy' ], how = 'left' )
spread_holdout = spread_holdout.merge( typical_spread_doy, on = [ 'doy' ], how = 'left' )

# define a final typical spread prediction column that uses the most specific available baseline
spread_holdout[ 'spread_pred_typical' ] = spread_holdout[ 'typical_spread_station_doy' ]
spread_holdout[ 'typical_source' ] = pd.Series( index = spread_holdout.index, dtype = 'object' )
spread_holdout.loc[ spread_holdout[ 'spread_pred_typical' ].notna( ), 'typical_source' ] = 'station_doy'

# fill in missing typical spread predictions with less specific baselines
station_fill_mask = spread_holdout[ 'spread_pred_typical' ].isna( ) & spread_holdout[ 'typical_spread_station' ].notna( )
spread_holdout.loc[ station_fill_mask, 'spread_pred_typical' ] = spread_holdout.loc[ station_fill_mask, 'typical_spread_station' ]
spread_holdout.loc[ station_fill_mask, 'typical_source' ] = 'station' 

# fill in missing typical spread predictions with even less specific baselines
region_doy_fill_mask = spread_holdout[ 'spread_pred_typical' ].isna( ) & spread_holdout[ 'typical_spread_region_doy' ].notna( )
spread_holdout.loc[ region_doy_fill_mask, 'spread_pred_typical' ] = spread_holdout.loc[ region_doy_fill_mask, 'typical_spread_region_doy' ]
spread_holdout.loc[ region_doy_fill_mask, 'typical_source' ] = 'region_doy'

# fill in missing typical spread predictions with least specific baseline (doy-level)
doy_fill_mask = spread_holdout[ 'spread_pred_typical' ].isna( ) & spread_holdout[ 'typical_spread_doy' ].notna( )
spread_holdout.loc[ doy_fill_mask, 'spread_pred_typical' ] = spread_holdout.loc[ doy_fill_mask, 'typical_spread_doy' ]
spread_holdout.loc[ doy_fill_mask, 'typical_source' ] = 'doy'

# fill in any remaining missing typical spread predictions with global median spread
global_fill_mask = spread_holdout[ 'spread_pred_typical' ].isna( )
spread_holdout.loc[ global_fill_mask, 'spread_pred_typical' ] = typical_spread_global
spread_holdout.loc[ global_fill_mask, 'typical_source' ] = 'global'

# now we have the holdout table with observed spread, model-predicted spread
# and typical spread baseline all together for comparison
spread_holdout[ 'model_spread_error' ] = spread_holdout[ 'spread_pred_model' ] - spread_holdout[ 'spread_obs' ]
spread_holdout[ 'typical_spread_error' ] = spread_holdout[ 'spread_pred_typical' ] - spread_holdout[ 'spread_obs' ]
spread_holdout[ 'month' ] = spread_holdout[ 'date' ].dt.month

# prior to this, the standard was a single value ... 

# for the diagnostic, we want to see how the selected model's predicted spread
# compares to the observed spread, and how both compare to a simple typical spread baseline
spread_score_rows = [ ]
for model_name, pred_col in [ 
    ( 'selected_model_spread', 'spread_pred_model' ),
    ( 'typical_spread_baseline', 'spread_pred_typical' ),
]:
    y_true = spread_holdout[ 'spread_obs' ]
    y_pred = spread_holdout[ pred_col ]
    spread_score_rows.append( { 
        'model': model_name,
        'mae': float( mean_absolute_error( y_true, y_pred ) ),
        'rmse': float( mean_squared_error( y_true, y_pred ) ** 0.5 ),
        'r2': float( r2_score( y_true, y_pred ) ),
    } )

spread_scores = pd.DataFrame( spread_score_rows ).sort_values( 'rmse', ascending = True ).reset_index( drop = True )

print( f'selected scenario model: {phase6_selected_name} ({selected_pred_col})' )
print( 'spread diagnostic scope: phase6_holdout_model ( flagged transition holdout stations )' )
print( f'train rows for spread baseline: {len( spread_train )}' )
print( f'holdout rows in spread diagnostic: {len( spread_holdout )}' )
print( f'holdout stations before spread dropna: {len( holdout_station_keys_pre )}' )
print( f'holdout stations after spread dropna: {len( holdout_station_keys_post )}' )
print( f'holdout stations fully dropped by spread non-null filter: {len( dropped_spread_station_keys )}' )
if len( dropped_spread_station_keys ) > 0:
    display( dropped_spread_station_keys )
print( f'global median typical spread ( water - air ): {typical_spread_global:.3f} C' )
typical_source_counts = spread_holdout[ 'typical_source' ].value_counts( dropna = False )
print( 'typical baseline source usage:' )
display( typical_source_counts )
print( '\nspread forecast skill on holdout:' )
display( spread_scores )

# let's also look at the spread distribution and how it varies over time in the holdout,
spread_distribution = ( 
    spread_holdout[ [ 'spread_obs', 'spread_pred_model', 'spread_pred_typical' ] ]
    .describe( percentiles = [ 0.10, 0.25, 0.50, 0.75, 0.90 ] )
    .T
    .round( 3 )
)
print( 'spread distribution (C):' )
display( spread_distribution )

# and how the spread and errors vary by month to see if there are any seasonal patterns
spread_monthly = ( 
    spread_holdout
    .groupby( 'month', as_index = False )
    .agg( 
        mean_spread_obs = ( 'spread_obs', 'mean' ),
        std_spread_obs = ( 'spread_obs', 'std' ),
        mae_model = ( 'model_spread_error', lambda values: float( values.abs( ).mean( ) ) ),
        mae_typical = ( 'typical_spread_error', lambda values: float( values.abs( ).mean( ) ) ),
        n_rows = ( 'spread_obs', 'size' ),
    )
    .sort_values( 'month' )
)
print( 'monthly spread profile + error:' )
display( spread_monthly.round( 3 ) )


In [ ]:
# finally, let's visualize the spread distribution and an example timeline for a single station
fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )

spread_bin_edges = np.linspace( -15, 20, 61 )
sns.histplot( 
    spread_holdout[ 'spread_obs' ],
    bins = spread_bin_edges,
    stat = 'count',
    element = 'step',
    fill = False,
    linewidth = 2.2,
    color = 'black',
    alpha = 1.0,
    ax = axes[ 0 ],
    label = 'Observed spread',
)
sns.histplot( 
    spread_holdout[ 'spread_pred_typical' ],
    bins = spread_bin_edges,
    stat = 'count',
    element = 'step',
    fill = False,
    linewidth = 1.8,
    color = 'tab:blue',
    alpha = 1.0,
    ax = axes[ 0 ],
    label = 'Typical spread baseline',
)
sns.histplot( 
    spread_holdout[ 'spread_pred_model' ],
    bins = spread_bin_edges,
    stat = 'count',
    element = 'step',
    fill = False,
    linewidth = 1.8,
    color = 'tab:orange',
    alpha = 1.0,
    ax = axes[ 0 ],
    label = 'Selected model spread',
)
axes[ 0 ].set_title( 'Holdout Distribution: Air-Water Spread ( C )' )
axes[ 0 ].set_xlabel( 'water_temp - air_temp ( C )' )
axes[ 0 ].set_ylabel( 'Count' )
axes[ 0 ].set_xlim( -15, 20 )
axes[ 0 ].set_ylim( 0, 30000 )
axes[ 0 ].legend( loc = 'best' )

top_station = ( 
    spread_holdout
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( n_rows = ( 'date', 'size' ) )
    .sort_values( 'n_rows', ascending = False )
    .head( 1 )
)

station_region = top_station.iloc[ 0 ][ 'region' ]
station_name = top_station.iloc[ 0 ][ 'station' ]
station_slice = spread_holdout.loc[ 
    ( spread_holdout[ 'region' ] == station_region )
    & ( spread_holdout[ 'station' ] == station_name )
].sort_values( 'date' )

axes[ 1 ].plot( 
    station_slice[ 'date' ], station_slice[ 'spread_obs' ], 
    linewidth = 1.0, 
    color = 'black', 
    label = 'Observed spread'
)
axes[ 1 ].plot( 
    station_slice[ 'date' ], station_slice[ 'spread_pred_typical' ], 
    linewidth = 1.0, 
    color = 'tab:blue', 
    alpha = 0.9, 
    label = 'Typical spread baseline'
)
axes[ 1 ].plot( 
    station_slice[ 'date' ], station_slice[ 'spread_pred_model' ], 
    linewidth = 1.0, 
    color = 'tab:orange', 
    alpha = 0.9, 
    label = f'Selected model ({phase6_selected_name})'
)
axes[ 1 ].set_title( f'Spread Timeline Example: {station_region} | {station_name}' )
axes[ 1 ].set_xlabel( 'Date' )
axes[ 1 ].set_ylabel( 'water_temp - air_temp ( C )' )
axes[ 1 ].legend( loc = 'best' )

plt.tight_layout( )
plt.show( )

### 6.4b - station-level spread residual offsets
- build per-station prediction-offset dataset ( model spread minus observed spread )


In [ ]:
# 6.4b - station-level offsets from selected model spread vs observed spread
# residual sign: positive means model spread is too warm vs observed spread

spread_holdout_resid = spread_holdout.copy( )
spread_holdout_resid[ 'spread_residual_model_minus_obs' ] = ( 
    spread_holdout_resid[ 'spread_pred_model' ] - spread_holdout_resid[ 'spread_obs' ]
)
spread_holdout_resid[ 'spread_abs_residual_model_minus_obs' ] = spread_holdout_resid[ 'spread_residual_model_minus_obs' ].abs( )

spread_station_offset = ( 
    spread_holdout_resid
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_rows = ( 'date', 'size' ),
        mean_offset_c = ( 'spread_residual_model_minus_obs', 'mean' ),
        median_offset_c = ( 'spread_residual_model_minus_obs', 'median' ),
        mae_offset_c = ( 'spread_abs_residual_model_minus_obs', 'mean' ),
        rmse_offset_c = ( 'spread_residual_model_minus_obs', lambda values: float( np.sqrt( np.mean( np.square( values ) ) ) ) ),
    )
    .sort_values( 'mean_offset_c' )
    .reset_index( drop = True )
)

spread_station_offset[ 'station_key' ] = spread_station_offset[ 'region' ].astype( str ) + ' | ' + spread_station_offset[ 'station' ].astype( str )

for col in [ 
    'mean_offset_c',
    'median_offset_c',
    'mae_offset_c',
    'rmse_offset_c',
]:
    spread_station_offset[ col ] = spread_station_offset[ col ].round( 4 )

print( f'station rows in offset table: { len( spread_station_offset ) }' )
display( spread_station_offset.sort_values( 'mean_offset_c' ).reset_index( drop = True ) )

plot_df = spread_station_offset.sort_values( 'mean_offset_c' ).reset_index( drop = True )
plot_df[ 'station_key' ] = pd.Categorical( plot_df[ 'station_key' ], categories = plot_df[ 'station_key' ], ordered = True )

region_order = sorted( plot_df[ 'region' ].dropna( ).unique( ).tolist( ) )
region_palette = dict( zip( region_order, sns.color_palette( 'tab20', n_colors = max( len( region_order ), 1 ) ) ) )

fig_height = max( 8, 0.25 * len( plot_df ) + 2 )
fig, axes = plt.subplots( 1, 2, figsize = ( 18, fig_height ), sharey = True )

for region in region_order:
    sub = plot_df.loc[ plot_df[ 'region' ] == region ]
    axes[ 0 ].scatter( 
        sub[ 'mean_offset_c' ],
        sub[ 'station_key' ],
        s = 200,
        alpha = 0.85,
        color = region_palette[ region ],
        label = region,
    )

axes[ 0 ].axvline( 0, color = 'black', linewidth = 1.1, linestyle = '--' )
axes[ 0 ].set_title( 'Station Mean Offset ( Model Spread - Observed Spread )' )
axes[ 0 ].set_xlabel( 'Mean Offset ( C )' )
axes[ 0 ].set_ylabel( 'Station ( Region | Station )' )
axes[ 0 ].legend( title = 'Region', bbox_to_anchor = ( 1.01, 1 ), loc = 'upper left' )

axes[ 1 ].barh( 
    plot_df[ 'station_key' ],
    plot_df[ 'mae_offset_c' ],
    color = [ region_palette[ r ] for r in plot_df[ 'region' ] ],
    alpha = 0.85,
)
axes[ 1 ].set_title( 'Station MAE of Spread Residual' )
axes[ 1 ].set_xlabel( 'MAE ( C )' )

plt.tight_layout( )
plt.show( )


### 6.4c - station bias classes and model-vs-typical MAE delta
- classify station spread bias and compare selected model MAE vs typical spread MAE


In [ ]:
# 6.4c - compare selected model spread error vs typical spread error by station
# negative delta_mae_model_minus_typical_c means selected model beat the typical-spread baseline

spread_holdout_resid[ 'typical_abs_error_c' ] = spread_holdout_resid[ 'typical_spread_error' ].abs( )

spread_station_compare = ( 
    spread_holdout_resid
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_rows = ( 'date', 'size' ),
        mean_model_offset_c = ( 'spread_residual_model_minus_obs', 'mean' ),
        median_model_offset_c = ( 'spread_residual_model_minus_obs', 'median' ),
        mae_model_c = ( 'spread_abs_residual_model_minus_obs', 'mean' ),
        mae_typical_c = ( 'typical_abs_error_c', 'mean' ),
    )
)

spread_station_compare[ 'delta_mae_model_minus_typical_c' ] = ( 
    spread_station_compare[ 'mae_model_c' ] - spread_station_compare[ 'mae_typical_c' ]
)
spread_station_compare[ 'station_key' ] = spread_station_compare[ 'region' ].astype( str ) + ' | ' + spread_station_compare[ 'station' ].astype( str )

bias_cutoff_c = 0.5
spread_station_compare[ 'bias_class' ] = 'near_zero'
spread_station_compare.loc[ spread_station_compare[ 'mean_model_offset_c' ] <= -bias_cutoff_c, 'bias_class' ] = 'under'
spread_station_compare.loc[ spread_station_compare[ 'mean_model_offset_c' ] >= bias_cutoff_c, 'bias_class' ] = 'over'
spread_station_compare[ 'model_beats_typical' ] = spread_station_compare[ 'delta_mae_model_minus_typical_c' ] < 0

for col in [ 
    'mean_model_offset_c',
    'median_model_offset_c',
    'mae_model_c',
    'mae_typical_c',
    'delta_mae_model_minus_typical_c',
]:
    spread_station_compare[ col ] = spread_station_compare[ col ].round( 4 )

spread_station_compare = spread_station_compare.sort_values( 'mean_model_offset_c' ).reset_index( drop = True )

print( '' )
print( 'stations where selected model beat typical spread baseline ( lower MAE ):' )
print( int( spread_station_compare[ 'model_beats_typical' ].sum( ) ), '/', len( spread_station_compare ) )

print( f'station rows in compare table: { len( spread_station_compare ) }' )
display( spread_station_compare.sort_values( 'mean_model_offset_c' ).reset_index( drop = True ) )

region_bias_summary = ( 
    spread_station_compare
    .groupby( 'region', as_index = False )
    .agg( 
        n_stations = ( 'station', 'nunique' ),
        mean_station_offset_c = ( 'mean_model_offset_c', 'mean' ),
        mean_station_delta_mae_c = ( 'delta_mae_model_minus_typical_c', 'mean' ),
        frac_model_beats_typical = ( 'model_beats_typical', 'mean' ),
    )
    .sort_values( 'mean_station_offset_c' )
    .reset_index( drop = True )
)

for col in [ 
    'mean_station_offset_c',
    'mean_station_delta_mae_c',
    'frac_model_beats_typical',
]:
    region_bias_summary[ col ] = region_bias_summary[ col ].round( 4 )

display( region_bias_summary )

fig, ax = plt.subplots( figsize = ( 10, 5 ) )
sns.histplot( 
    spread_station_compare[ 'mean_model_offset_c' ],
    bins = 16,
    color = 'tab:purple',
    alpha = 0.75,
    edgecolor = 'white',
    ax = ax,
)
ax.axvline( 0, color = 'black', linewidth = 1.2, linestyle = '--', label = 'Zero bias' )
ax.axvline( -bias_cutoff_c, color = 'tab:gray', linewidth = 1.0, linestyle = ':' )
ax.axvline( bias_cutoff_c, color = 'tab:gray', linewidth = 1.0, linestyle = ':', label = 'Bias class cutoff' )
ax.set_title( 'Histogram: Station Mean Spread Bias ( Model - Observed )' )
ax.set_xlabel( 'Mean Spread Bias ( C )' )
ax.set_ylabel( 'Station Count' )
ax.legend( loc = 'best' )
plt.tight_layout( )
plt.show( )


In [ ]:
# confusion-matrix style view for regression (holdout)
y_true = phase6_holdout_model[ target_col ]
y_pred = phase6_holdout_model[ selected_pred_col ]

# choose meaningful bins for delta_water_temp (C)
bins = [ -np.inf, -2.0, -1.0, -0.5, 0.5, 1.0, 2.0, np.inf ]
labels = [ '<-2', '-2:-1', '-1:-0.5', '-0.5:0.5', '0.5:1', '1:2', '>2' ]

obs_bin = pd.cut( y_true, bins = bins, labels = labels )
pred_bin = pd.cut( y_pred, bins = bins, labels = labels )

cm = pd.crosstab( 
    obs_bin,
    pred_bin,
    rownames = [ 'Observed ΔT bin' ],
    colnames = [ 'Predicted ΔT bin' ],
    normalize = 'index',
)

#display( cm.round( 3 ) )

plt.figure( figsize = ( 8, 6 ) )
sns.heatmap( cm, annot = True, fmt = '.2f', cmap = 'Blues' )
plt.title( 'Holdout: Binned Observed vs Predicted ΔT (row-normalized)' )
plt.tight_layout( )
plt.show( )

# optional: tolerance-hit rates
abs_err = ( y_pred - y_true ).abs( )
for tol in [ 0.5, 1.0, 2.0 ]:
    print( f'within ±{tol:.1f} C: {( abs_err <= tol ).mean():.3f}' )


### 6.4d - interpretation of binned observed vs predicted ΔT matrix
- rows are observed bins; columns are predicted bins; each row sums to ~1.0.
- strong performance at the extremes:
  - observed `<-2`: 0.938 on-diagonal
  - observed `>2`: 0.891 on-diagonal
- mid-range bins are much less stable:
  - observed `-2:-1`: only 0.191 on-diagonal, with 0.555 pushed into `<-2`
  - observed `-1:-0.5`: only 0.124 on-diagonal, with large spill into colder bins
  - observed `-0.5:0.5`: only 0.215 on-diagonal, broad spread across both sides
  - observed `0.5:1`: only 0.105 on-diagonal
  - observed `1:2`: only 0.172 on-diagonal, with 0.383 pushed into `>2`
- model recognizes strong cooling/warming events fairly well, but moderate ΔT regimes are often mis-binned and pulled toward neighboring or extreme bins.

### 6.5 to 6.7 - seasonal sampling-gap diagnostics (no pipeline changes yet)
Goal: test whether changing winter/summer coverage could bias annual means and apparent warming signals.


In [ ]:
# 6.5 - build station-year sampling diagnostics from daily water observations
coverage_diag = daily_water[ [ 'region', 'station', 'date', 'water_temp', 'delta_water_temp' ] ].copy( )
coverage_diag[ 'date' ] = pd.to_datetime( coverage_diag[ 'date' ], errors = 'coerce' )
coverage_diag = coverage_diag.dropna( subset = [ 'date' ] )
coverage_diag[ 'year' ] = coverage_diag[ 'date' ].dt.year
coverage_diag[ 'month' ] = coverage_diag[ 'date' ].dt.month
coverage_diag[ 'has_temp' ] = coverage_diag[ 'water_temp' ].notna( ).astype( int )

monthly_presence = ( 
    coverage_diag
    .groupby( [ 'region', 'station', 'year', 'month' ], as_index = False )[ 'has_temp' ]
    .max( )
)

monthly_presence[ 'winter_obs' ] = monthly_presence[ 'has_temp' ] * monthly_presence[ 'month' ].isin( [ 12, 1, 2 ] ).astype( int )
monthly_presence[ 'summer_obs' ] = monthly_presence[ 'has_temp' ] * monthly_presence[ 'month' ].isin( [ 6, 7, 8 ] ).astype( int )

station_year_sampling = ( 
    monthly_presence
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        n_months_obs = ( 'has_temp', 'sum' ),
        n_winter_months = ( 'winter_obs', 'sum' ),
        n_summer_months = ( 'summer_obs', 'sum' ),
    )
)

annual_temp_by_station = ( 
    coverage_diag
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        annual_mean_water_temp = ( 'water_temp', 'mean' ),
        annual_mean_delta_water_temp = ( 'delta_water_temp', 'mean' ),
        n_daily_rows = ( 'water_temp', 'size' ),
    )
)

station_year_sampling = station_year_sampling.merge( 
    annual_temp_by_station,
    on = [ 'region', 'station', 'year' ],
    how = 'left',
)

station_year_sampling[ 'has_winter_gap' ] = station_year_sampling[ 'n_winter_months' ] == 0
station_year_sampling[ 'has_summer_gap' ] = station_year_sampling[ 'n_summer_months' ] == 0
station_year_sampling[ 'seasonally_balanced' ] = ( 
    ( station_year_sampling[ 'n_months_obs' ] >= 9 )
    & ( station_year_sampling[ 'n_winter_months' ] >= 2 )
    & ( station_year_sampling[ 'n_summer_months' ] >= 2 )
)

station_year_sampling[ 'season_balance_ratio' ] = ( 
    np.minimum( station_year_sampling[ 'n_winter_months' ], station_year_sampling[ 'n_summer_months' ] )
    / np.maximum( station_year_sampling[ 'n_winter_months' ], station_year_sampling[ 'n_summer_months' ] ).replace( 0, np.nan )
)

print( f'station-year rows: {len( station_year_sampling )}' )
print( f'pct with zero winter months: {100.0 * station_year_sampling[ "has_winter_gap" ].mean( ):.2f}%' )
print( f'pct with zero summer months: {100.0 * station_year_sampling[ "has_summer_gap" ].mean( ):.2f}%' )
print( f'pct seasonally balanced (>=9 months + >=2 winter + >=2 summer): {100.0 * station_year_sampling[ "seasonally_balanced" ].mean( ):.2f}%' )

display( 
    station_year_sampling[ [ 
        'n_months_obs',
        'n_winter_months',
        'n_summer_months',
        'season_balance_ratio',
        'annual_mean_delta_water_temp',
    ] ]
    .describe( percentiles = [ 0.10, 0.25, 0.50, 0.75, 0.90 ] )
    .round( 3 )
)


In [ ]:
# 6.6 - visualize sampling coverage shifts over time
yearly_sampling = ( 
    station_year_sampling
    .groupby( 'year', as_index = False )
    .agg( 
        n_station_years = ( 'station', 'size' ),
        mean_months_obs = ( 'n_months_obs', 'mean' ),
        pct_with_winter_gap = ( 'has_winter_gap', lambda values: 100.0 * float( values.mean( ) ) ),
        pct_with_summer_gap = ( 'has_summer_gap', lambda values: 100.0 * float( values.mean( ) ) ),
        pct_seasonally_balanced = ( 'seasonally_balanced', lambda values: 100.0 * float( values.mean( ) ) ),
        mean_annual_delta_all = ( 'annual_mean_delta_water_temp', 'mean' ),
    )
)

yearly_balanced = ( 
    station_year_sampling
    .loc[ station_year_sampling[ 'seasonally_balanced' ] ]
    .groupby( 'year', as_index = False )
    .agg( 
        mean_annual_delta_balanced = ( 'annual_mean_delta_water_temp', 'mean' ),
        n_station_years_balanced = ( 'station', 'size' ),
    )
)

yearly_sampling = yearly_sampling.merge( yearly_balanced, on = 'year', how = 'left' )

fig, axes = plt.subplots( 1, 2, figsize = ( 16, 5 ) )

axes[ 0 ].plot( yearly_sampling[ 'year' ], yearly_sampling[ 'pct_with_winter_gap' ], label = '% station-years with winter gap', linewidth = 1.6 )
axes[ 0 ].plot( yearly_sampling[ 'year' ], yearly_sampling[ 'pct_with_summer_gap' ], label = '% station-years with summer gap', linewidth = 1.6 )
axes[ 0 ].plot( yearly_sampling[ 'year' ], yearly_sampling[ 'pct_seasonally_balanced' ], label = '% seasonally balanced', linewidth = 1.8 )
axes[ 0 ].set_title( 'Seasonal Sampling Coverage Over Time' )
axes[ 0 ].set_xlabel( 'Year' )
axes[ 0 ].set_ylabel( 'Percent of Station-Years' )
axes[ 0 ].set_ylim( 0, 100 )
axes[ 0 ].legend( loc = 'best' )

axes[ 1 ].plot( yearly_sampling[ 'year' ], yearly_sampling[ 'mean_annual_delta_all' ], label = 'Mean annual delta (all station-years)', linewidth = 1.8, color = 'black' )
axes[ 1 ].plot( yearly_sampling[ 'year' ], yearly_sampling[ 'mean_annual_delta_balanced' ], label = 'Mean annual delta (seasonally balanced only)', linewidth = 1.8, color = 'tab:orange' )
axes[ 1 ].axhline( 0.0, color = 'gray', linestyle = '--', linewidth = 1.0 )
axes[ 1 ].set_title( 'Annual Delta Water Temp: All vs Balanced Coverage' )
axes[ 1 ].set_xlabel( 'Year' )
axes[ 1 ].set_ylabel( 'Mean annual delta_water_temp ( C )' )
axes[ 1 ].legend( loc = 'best' )

plt.tight_layout( )
plt.show( )

display( yearly_sampling.tail( 15 ).round( 3 ) )


In [ ]:
# 6.7 - quick sensitivity tests: trend slope and gap-vs-no-gap contrast


def slope_c_per_year( frame, y_col ):
    fit = frame[ [ 'year', y_col ] ].dropna( ).copy( )
    if len( fit ) < 2:
        return np.nan

    return float( np.polyfit( fit[ 'year' ].astype( float ), fit[ y_col ].astype( float ), 1 )[ 0 ] )


slope_all = slope_c_per_year( yearly_sampling, 'mean_annual_delta_all' )
slope_balanced = slope_c_per_year( yearly_sampling, 'mean_annual_delta_balanced' )

gap_effect_winter = ( 
    station_year_sampling
    .groupby( 'has_winter_gap', as_index = False )
    .agg( 
        mean_delta = ( 'annual_mean_delta_water_temp', 'mean' ),
        median_delta = ( 'annual_mean_delta_water_temp', 'median' ),
        n_rows = ( 'annual_mean_delta_water_temp', 'size' ),
    )
)

gap_effect_summer = ( 
    station_year_sampling
    .groupby( 'has_summer_gap', as_index = False )
    .agg( 
        mean_delta = ( 'annual_mean_delta_water_temp', 'mean' ),
        median_delta = ( 'annual_mean_delta_water_temp', 'median' ),
        n_rows = ( 'annual_mean_delta_water_temp', 'size' ),
    )
)

print( f'slope all station-years (C/year): {slope_all:.4f}' )
print( f'slope seasonally balanced only (C/year): {slope_balanced:.4f}' )
print( f'slope difference (balanced - all): {slope_balanced - slope_all:.4f}' )

print( '' )
print( 'winter-gap contrast (annual_mean_delta_water_temp):' )
display( gap_effect_winter.round( 3 ) )

print( 'summer-gap contrast (annual_mean_delta_water_temp):' )
display( gap_effect_summer.round( 3 ) )

sns.boxplot( 
    data = station_year_sampling.assign( winter_gap_label = station_year_sampling[ 'has_winter_gap' ].map( { False: 'winter present', True: 'winter missing' } ) ),
    x = 'winter_gap_label',
    y = 'annual_mean_delta_water_temp',
    showfliers = False,
)
plt.title( 'Annual Delta Water Temp by Winter Coverage Status' )
plt.xlabel( 'Coverage Group' )
plt.ylabel( 'annual_mean_delta_water_temp ( C )' )
plt.tight_layout( )
plt.show( )



## Phase 7 — Model Development: Water Temp → Water Properties
Goal: predict Δsalinity, ΔDO, ΔpH given ΔT_water and atmospheric context

# also stage 2

- 7.1 for each target variable, repeat baseline 
  - linear → gradient boosted sequence from phase 6
- 7.2 use predicted Δ air temp for water as input
  - keeps the pipeline honest and propagates uncertainty
- 7.3 compare response sensitivity by regime
  - does a +2°C warming suppress dissolved oxy more in warm estuaries than cold ones?
  - prolly a yes ... quantifying it is the contribution
- 7.4 document responses as simple lookup relationships
  - e.g., per-degree sensitivities per regime 
  - these are our core scientific deliverables

### 7.0 - build annual climate-response tables (non-daily)
- purpose: climate-change potential, not next-day forecasting
- aggregate station-year means, then model annual non-nutrient deltas
- keep phase-6 link by using phase-6 predicted `delta_water_temp` as a phase-7 driver


In [ ]:
# 7.0 - scenario-oriented phase 7 prep (annual aggregates)
# rolling atmospheric features already exist from phase 3 ( *_r1d, *_r7d, *_r28d )

air_source_7 = daily_air_final.copy( )
water_source_7 = daily_water_final.copy( )

air_source_7[ 'date' ] = pd.to_datetime( air_source_7[ 'date' ], errors = 'coerce' )
water_source_7[ 'date' ] = pd.to_datetime( water_source_7[ 'date' ], errors = 'coerce' )

# depth parked for now per current draft focus
phase7_target_candidates = [ 'delta_salinity', 'delta_oxygen', 'delta_ph' ]
phase7_targets = [ col for col in phase7_target_candidates if col in water_source_7.columns ]

water_keep_cols_7 = [ 'region', 'station', 'date' ] + phase7_targets
for col in [ 'water_temp_baseline', 'salinity_baseline', 'oxygen_baseline', 'ph_baseline', 'depth_baseline' ]:
    if col in water_source_7.columns:
        water_keep_cols_7.append( col )
water_keep_cols_7 = list( dict.fromkeys( water_keep_cols_7 ) )

phase7_daily = air_source_7.merge( 
    water_source_7[ water_keep_cols_7 ],
    on = [ 'region', 'station', 'date' ],
    how = 'inner',
)
phase7_daily = phase7_daily.merge( phase6_context_table, on = [ 'region', 'station' ], how = 'left' )

phase7_p6_feature_cols = [ col for col in phase6_selected_features if col in phase7_daily.columns ]
phase7_p6_fill_values = phase6_train_model[ phase7_p6_feature_cols ].median( numeric_only = True )

X_phase7_for_p6 = phase7_daily[ phase7_p6_feature_cols ].copy( ).fillna( phase7_p6_fill_values )
phase7_daily[ 'delta_water_temp_pred_p6' ] = phase6_selected_model.predict( X_phase7_for_p6 )

phase7_daily = phase7_daily.dropna( subset = [ 'date' ] )
phase7_daily[ 'year' ] = phase7_daily[ 'date' ].dt.year

agg_cols_7 = [ ]
for col in [ 'delta_water_temp_pred_p6', 'air_temp', 'precip', 'wind_speed', 'solar' ]:
    if col in phase7_daily.columns:
        agg_cols_7.append( col )

phase7_agg = { 
    'n_days': ( 'date', 'size' ),
}

for col in agg_cols_7:
    phase7_agg[ f'{ col }_mean' ] = ( col, 'mean' )
    phase7_agg[ f'{ col }_std' ] = ( col, 'std' )

for target in phase7_targets:
    phase7_agg[ f'{ target }_annual_mean' ] = ( target, 'mean' )

phase7_station_year = ( 
    phase7_daily
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( **phase7_agg )
)

phase7_station_year_train = phase7_station_year.merge( 
    unflagged_station_keys_final[ [ 'region', 'station' ] ].drop_duplicates( ),
    on = [ 'region', 'station' ],
    how = 'inner',
)
phase7_station_year_holdout = phase7_station_year.merge( 
    flagged_station_keys_final[ [ 'region', 'station' ] ].drop_duplicates( ),
    on = [ 'region', 'station' ],
    how = 'inner',
)

phase7_feature_candidates = [ 
    'delta_water_temp_pred_p6_mean',
    'air_temp_mean',
    'precip_mean',
    'wind_speed_mean',
    'solar_mean',
    'delta_water_temp_pred_p6_std',
    'air_temp_std',
    'precip_std',
    'wind_speed_std',
    'solar_std',
]
phase7_features = [ col for col in phase7_feature_candidates if col in phase7_station_year_train.columns ]

print( f'phase7 targets: { phase7_targets }' )
print( f'phase7 daily rows: { len( phase7_daily ) }' )
print( f'phase7 station-year rows (all): { len( phase7_station_year ) }' )
print( f'phase7 station-year rows (train): { len( phase7_station_year_train ) }' )
print( f'phase7 station-year rows (holdout): { len( phase7_station_year_holdout ) }' )
print( f'phase7 features ({ len( phase7_features ) }): { phase7_features }' )


### 7.1 - fit annual response models (no persistence)
- compare `naive_global_median`, `ridge_phase7_climate`, `hgb_phase7_climate`


In [ ]:
# 7.1 - annual response model stack per non-nutrient target
# no persistence baseline (intentional: climate-response framing)

phase7_scores_rows = [ ]
phase7_best_rows = [ ]
phase7_model_store = { }
phase7_feature_store = { }
phase7_fill_store = { }

for target in phase7_targets:
    target_col = f'{ target }_annual_mean'

    train_t = phase7_station_year_train.dropna( subset = [ target_col ] ).copy( )
    holdout_t = phase7_station_year_holdout.dropna( subset = [ target_col ] ).copy( )

    feature_cols_t = [ col for col in phase7_features if col in train_t.columns ]

    if len( train_t ) == 0 or len( holdout_t ) == 0 or len( feature_cols_t ) == 0:
        continue

    y_train_t = train_t[ target_col ]
    y_holdout_t = holdout_t[ target_col ]

    X_train_t = train_t[ feature_cols_t ].copy( )
    X_holdout_t = holdout_t[ feature_cols_t ].copy( )

    fill_values_t = X_train_t.median( numeric_only = True )
    X_train_t = X_train_t.fillna( fill_values_t )
    X_holdout_t = X_holdout_t.fillna( fill_values_t )

    pred_naive = np.full( len( y_holdout_t ), float( y_train_t.median( ) ) )
    phase7_scores_rows.append( { 
        'target': target,
        'model': 'naive_global_median',
        'mae': float( mean_absolute_error( y_holdout_t, pred_naive ) ),
        'rmse': float( mean_squared_error( y_holdout_t, pred_naive ) ** 0.5 ),
        'r2': float( r2_score( y_holdout_t, pred_naive ) ),
        'n_train': int( len( train_t ) ),
        'n_holdout': int( len( holdout_t ) ),
        'n_features': int( len( feature_cols_t ) ),
    } )

    ridge_t = Ridge( alpha = 1.0 )
    ridge_t.fit( X_train_t, y_train_t )
    pred_ridge = ridge_t.predict( X_holdout_t )

    phase7_scores_rows.append( { 
        'target': target,
        'model': 'ridge_phase7_climate',
        'mae': float( mean_absolute_error( y_holdout_t, pred_ridge ) ),
        'rmse': float( mean_squared_error( y_holdout_t, pred_ridge ) ** 0.5 ),
        'r2': float( r2_score( y_holdout_t, pred_ridge ) ),
        'n_train': int( len( train_t ) ),
        'n_holdout': int( len( holdout_t ) ),
        'n_features': int( len( feature_cols_t ) ),
    } )

    hgb_t = HistGradientBoostingRegressor( 
        learning_rate = 0.05,
        max_depth = 4,
        max_iter = 300,
        min_samples_leaf = 20,
        random_state = 42,
    )
    hgb_t.fit( X_train_t, y_train_t )
    pred_hgb = hgb_t.predict( X_holdout_t )

    phase7_scores_rows.append( { 
        'target': target,
        'model': 'hgb_phase7_climate',
        'mae': float( mean_absolute_error( y_holdout_t, pred_hgb ) ),
        'rmse': float( mean_squared_error( y_holdout_t, pred_hgb ) ** 0.5 ),
        'r2': float( r2_score( y_holdout_t, pred_hgb ) ),
        'n_train': int( len( train_t ) ),
        'n_holdout': int( len( holdout_t ) ),
        'n_features': int( len( feature_cols_t ) ),
    } )

    target_scores = pd.DataFrame( phase7_scores_rows )
    target_scores = target_scores.loc[ target_scores[ 'target' ] == target ].sort_values( 'rmse', ascending = True )

    target_scores_non_naive = ( 
        target_scores
        .loc[ target_scores[ 'model' ] != 'naive_global_median' ]
        .sort_values( 'rmse', ascending = True )
    )

    if len( target_scores_non_naive ) > 0:
        best_row_for_scenarios = target_scores_non_naive.iloc[ 0 ]

    else:
        best_row_for_scenarios = target_scores.iloc[ 0 ]

    best_model_name = str( best_row_for_scenarios[ 'model' ] )

    if best_model_name == 'hgb_phase7_climate':
        phase7_model_store[ target ] = hgb_t

    elif best_model_name == 'ridge_phase7_climate':
        phase7_model_store[ target ] = ridge_t

    else:
        phase7_model_store[ target ] = ridge_t
        best_model_name = 'ridge_phase7_climate'
        best_row_for_scenarios = target_scores.loc[ target_scores[ 'model' ] == 'ridge_phase7_climate' ].iloc[ 0 ]

    phase7_feature_store[ target ] = feature_cols_t
    phase7_fill_store[ target ] = fill_values_t

    phase7_best_rows.append( { 
        'target': target,
        'best_model_any': str( target_scores.iloc[ 0 ][ 'model' ] ),
        'best_model_for_scenarios': best_model_name,
        'best_rmse_for_scenarios': float( best_row_for_scenarios[ 'rmse' ] ),
        'best_mae_for_scenarios': float( best_row_for_scenarios[ 'mae' ] ),
        'best_r2_for_scenarios': float( best_row_for_scenarios[ 'r2' ] ),
        'n_train': int( len( train_t ) ),
        'n_holdout': int( len( holdout_t ) ),
    } )

phase7_scores = pd.DataFrame( phase7_scores_rows )
phase7_scores = phase7_scores.sort_values( [ 'target', 'rmse' ], ascending = [ True, True ] ).reset_index( drop = True )

phase7_best = pd.DataFrame( phase7_best_rows )
phase7_best = phase7_best.sort_values( 'best_rmse_for_scenarios' ).reset_index( drop = True )

print( 'phase7 annual-response leaderboard (holdout):' )
display( phase7_scores.round( 4 ) )

print( 'phase7 best model per target:' )
display( phase7_best.round( 4 ) )


### 7.1b - holdout model success/failure diagnostics
- evaluate best phase-7 scenario model on holdout station-year rows
- plot MAE/RMSE, residual spread, and observed-vs-predicted parity


In [ ]:
# 7.1b - holdout diagnostics for phase 7 scenario models

phase7_eval_rows = [ ]

for target in phase7_targets:
    target_col = f'{ target }_annual_mean'
    model_t = phase7_model_store.get( target )
    feature_cols_t = phase7_feature_store.get( target, [ ] )
    fill_values_t = phase7_fill_store.get( target, pd.Series( dtype = 'float64' ) )

    holdout_t = phase7_station_year_holdout.dropna( subset = [ target_col ] ).copy( )

    if model_t is None or len( feature_cols_t ) == 0 or len( holdout_t ) == 0:
        continue

    X_holdout_t = holdout_t[ feature_cols_t ].copy( ).fillna( fill_values_t )
    pred_t = model_t.predict( X_holdout_t )

    eval_t = holdout_t[ [ 'region', 'station', 'year' ] ].copy( )
    eval_t[ 'target' ] = target
    eval_t[ 'y_true' ] = holdout_t[ target_col ].values
    eval_t[ 'y_pred' ] = pred_t
    eval_t[ 'residual' ] = eval_t[ 'y_pred' ] - eval_t[ 'y_true' ]
    eval_t[ 'abs_error' ] = eval_t[ 'residual' ].abs( )

    phase7_eval_rows.append( eval_t )

if len( phase7_eval_rows ) == 0:
    print( 'phase7_eval_rows is empty; no diagnostics generated' )

else:
    phase7_eval_long = pd.concat( phase7_eval_rows, ignore_index = True )

    phase7_eval_summary = ( 
        phase7_eval_long
        .groupby( 'target', as_index = False )
        .agg( 
            n_holdout = ( 'y_true', 'size' ),
            mae = ( 'abs_error', 'mean' ),
            rmse = ( 'residual', lambda values: float( np.sqrt( np.mean( np.square( values ) ) ) ) ),
            r2 = ( 'y_true', lambda values: np.nan ),
            bias = ( 'residual', 'mean' ),
            median_abs_error = ( 'abs_error', 'median' ),
        )
    )

    for idx, row in phase7_eval_summary.iterrows( ):
        t = row[ 'target' ]
        sub = phase7_eval_long.loc[ phase7_eval_long[ 'target' ] == t ]
        phase7_eval_summary.loc[ idx, 'r2' ] = float( r2_score( sub[ 'y_true' ], sub[ 'y_pred' ] ) )

    phase7_eval_summary = phase7_eval_summary.sort_values( 'rmse' ).reset_index( drop = True )

    print( 'phase7 holdout diagnostics summary:' )
    display( phase7_eval_summary.round( 4 ) )

    score_plot_df = phase7_eval_summary.melt( 
        id_vars = [ 'target' ],
        value_vars = [ 'mae', 'rmse', 'median_abs_error' ],
        var_name = 'metric',
        value_name = 'value',
    )

    fig, axes = plt.subplots( 1, 2, figsize = ( 16, 5 ) )

    sns.barplot( 
        data = score_plot_df,
        x = 'target',
        y = 'value',
        hue = 'metric',
        ax = axes[ 0 ],
    )
    axes[ 0 ].set_title( 'Phase 7 Holdout Error Metrics by Target' )
    axes[ 0 ].set_xlabel( 'Target' )
    axes[ 0 ].set_ylabel( 'Error ( annual delta units )' )
    axes[ 0 ].tick_params( axis = 'x', rotation = 20 )

    sns.boxplot( 
        data = phase7_eval_long,
        x = 'target',
        y = 'residual',
        ax = axes[ 1 ],
    )
    axes[ 1 ].axhline( 0, color = 'black', linewidth = 1.0, linestyle = '--' )
    axes[ 1 ].set_title( 'Phase 7 Holdout Residual Spread ( Pred - Obs )' )
    axes[ 1 ].set_xlabel( 'Target' )
    axes[ 1 ].set_ylabel( 'Residual ( annual delta units )' )
    axes[ 1 ].tick_params( axis = 'x', rotation = 20 )

    plt.tight_layout( )
    plt.show( )

    targets_plot = phase7_eval_summary[ 'target' ].tolist( )
    n_targets = len( targets_plot )
    fig, axes = plt.subplots( 1, n_targets, figsize = ( 6 * n_targets, 5 ) )
    if n_targets == 1:
        axes = [ axes ]

    for ax, t in zip( axes, targets_plot ):
        sub = phase7_eval_long.loc[ phase7_eval_long[ 'target' ] == t ]
        sns.scatterplot( data = sub, x = 'y_true', y = 'y_pred', s = 35, alpha = 0.65, ax = ax )
        lo = float( min( sub[ 'y_true' ].min( ), sub[ 'y_pred' ].min( ) ) )
        hi = float( max( sub[ 'y_true' ].max( ), sub[ 'y_pred' ].max( ) ) )
        ax.plot( [ lo, hi ], [ lo, hi ], color = 'black', linestyle = '--', linewidth = 1.0 )
        ax.set_title( f'{ t }: Observed vs Predicted' )
        ax.set_xlabel( 'Observed annual mean delta' )
        ax.set_ylabel( 'Predicted annual mean delta' )

    plt.tight_layout( )
    plt.show( )


### 7.1c - confusion-matrix style bins for regression holdout
- binned observed vs predicted annual deltas (row-normalized)
- diagonal concentration = stronger categorical agreement


In [ ]:
# 7.1c - regression confusion-matrix style heatmaps

if 'phase7_eval_long' not in globals( ):
    print( 'Run 7.1b first (phase7_eval_long missing).' )

else:
    phase7_cm_store = { }
    targets_cm = sorted( phase7_eval_long[ 'target' ].dropna( ).unique( ).tolist( ) )

    for target in targets_cm:
        sub = phase7_eval_long.loc[ phase7_eval_long[ 'target' ] == target ].copy( )

        if len( sub ) < 20 or sub[ 'y_true' ].nunique( ) < 4:
            continue

        n_bins = min( 6, int( sub[ 'y_true' ].nunique( ) ) )

        try:
            _, bins = pd.qcut( sub[ 'y_true' ], q = n_bins, retbins = True, duplicates = 'drop' )

        except ValueError:
            continue

        bins = np.unique( bins )
        if len( bins ) < 3:
            continue

        labels = [ f'B{ i + 1 }' for i in range( len( bins ) - 1 ) ]

        obs_bin = pd.cut( sub[ 'y_true' ], bins = bins, labels = labels, include_lowest = True )
        pred_bin = pd.cut( sub[ 'y_pred' ], bins = bins, labels = labels, include_lowest = True )

        cm = pd.crosstab( 
            obs_bin,
            pred_bin,
            rownames = [ 'Observed bin' ],
            colnames = [ 'Predicted bin' ],
            normalize = 'index',
        )

        phase7_cm_store[ target ] = cm

    print( f'confusion-style targets generated: { sorted( phase7_cm_store.keys( ) ) }' )

    if len( phase7_cm_store ) == 0:
        print( 'No confusion-style matrices generated (insufficient variation/rows).' )

    else:
        n = len( phase7_cm_store )
        fig, axes = plt.subplots( 1, n, figsize = ( 6 * n, 5 ) )
        if n == 1:
            axes = [ axes ]

        for ax, ( target, cm ) in zip( axes, phase7_cm_store.items( ) ):
            sns.heatmap( 
                cm,
                annot = True,
                fmt = '.2f',
                cmap = 'Blues',
                vmin = 0,
                vmax = 1,
                ax = ax,
                cbar_kws = { 'label': 'Row share' },
            )
            ax.set_title( f'{ target } binned Obs vs Pred' )
            ax.set_xlabel( 'Predicted bin' )
            ax.set_ylabel( 'Observed bin' )

        plt.tight_layout( )
        plt.show( )


### 7.2 - climate-change benchmark scenarios
- scenario examples: `+1.5C` with `precip x0.9 / x1.0 / x1.1`
- output: projected annual delta responses overall and by region


In [ ]:
# 7.2 - scenario projections from annual response models

# estimate a simple phase6 air->water transfer coefficient (per +1C air shift)
phase7_phase6_air_cols = [ col for col in phase7_p6_feature_cols if col.startswith( 'air_temp' ) ]

if len( phase7_p6_feature_cols ) > 0 and len( phase7_phase6_air_cols ) > 0:
    X_p6_holdout_base = phase6_holdout_model[ phase7_p6_feature_cols ].copy( )
    X_p6_holdout_base = X_p6_holdout_base.fillna( phase7_p6_fill_values )

    X_p6_holdout_warm = X_p6_holdout_base.copy( )
    for col in phase7_phase6_air_cols:
        X_p6_holdout_warm[ col ] = X_p6_holdout_warm[ col ] + 1.0

    pred_base = phase6_selected_model.predict( X_p6_holdout_base )
    pred_warm = phase6_selected_model.predict( X_p6_holdout_warm )

    phase7_water_temp_transfer_per_c = float( np.mean( pred_warm - pred_base ) )

else:
    phase7_water_temp_transfer_per_c = 1.0

print( f'phase7 inferred phase6 transfer: +1C air -> { phase7_water_temp_transfer_per_c:.4f}C in predicted delta_water_temp' )

phase7_scenarios = pd.DataFrame( [ 
    { 'scenario': 'baseline_ref', 'air_temp_add_c': 0.0, 'precip_mult': 1.00 },
    { 'scenario': 'plus1p5C_precip_minus10pct', 'air_temp_add_c': 1.5, 'precip_mult': 0.90 },
    { 'scenario': 'plus1p5C_precip_same', 'air_temp_add_c': 1.5, 'precip_mult': 1.00 },
    { 'scenario': 'plus1p5C_precip_plus10pct', 'air_temp_add_c': 1.5, 'precip_mult': 1.10 },
] )

projection_rows = [ ]
projection_region_rows = [ ]
phase7_projected_targets = [ ]
phase7_skipped_targets = [ ]

projection_base = phase7_station_year.copy( )

for target in phase7_targets:
    model_t = phase7_model_store.get( target )
    feature_cols_t = phase7_feature_store.get( target, [ ] )
    fill_values_t = phase7_fill_store.get( target, pd.Series( dtype = 'float64' ) )

    if model_t is None or len( feature_cols_t ) == 0:
        phase7_skipped_targets.append( target )
        continue

    phase7_projected_targets.append( target )

    X_base = projection_base[ feature_cols_t ].copy( ).fillna( fill_values_t )

    target_projection_df = projection_base[ [ 'region', 'station', 'year' ] ].copy( )

    for _, scen in phase7_scenarios.iterrows( ):
        X_scen = X_base.copy( )

        if 'air_temp_mean' in X_scen.columns:
            X_scen[ 'air_temp_mean' ] = X_scen[ 'air_temp_mean' ] + float( scen[ 'air_temp_add_c' ] )

        if 'precip_mean' in X_scen.columns:
            X_scen[ 'precip_mean' ] = X_scen[ 'precip_mean' ] * float( scen[ 'precip_mult' ] )

        if 'delta_water_temp_pred_p6_mean' in X_scen.columns:
            X_scen[ 'delta_water_temp_pred_p6_mean' ] = ( 
                X_scen[ 'delta_water_temp_pred_p6_mean' ]
                + float( scen[ 'air_temp_add_c' ] ) * phase7_water_temp_transfer_per_c
            )

        pred_scen = model_t.predict( X_scen )
        scen_name = str( scen[ 'scenario' ] )

        target_projection_df[ scen_name ] = pred_scen

        projection_rows.append( { 
            'target': target,
            'scenario': scen_name,
            'mean_pred': float( np.mean( pred_scen ) ),
            'median_pred': float( np.median( pred_scen ) ),
            'n_station_years': int( len( pred_scen ) ),
        } )

        region_tmp = target_projection_df.groupby( 'region', as_index = False )[ scen_name ].mean( )
        for _, r in region_tmp.iterrows( ):
            projection_region_rows.append( { 
                'target': target,
                'scenario': scen_name,
                'region': r[ 'region' ],
                'mean_pred': float( r[ scen_name ] ),
            } )

phase7_projection_summary = pd.DataFrame( projection_rows )

baseline_lookup = ( 
    phase7_projection_summary
    .loc[ phase7_projection_summary[ 'scenario' ] == 'baseline_ref', [ 'target', 'mean_pred' ] ]
    .rename( columns = { 'mean_pred': 'baseline_mean_pred' } )
)

phase7_projection_summary = phase7_projection_summary.merge( baseline_lookup, on = 'target', how = 'left' )
phase7_projection_summary[ 'delta_vs_baseline' ] = phase7_projection_summary[ 'mean_pred' ] - phase7_projection_summary[ 'baseline_mean_pred' ]
phase7_projection_summary = phase7_projection_summary.sort_values( [ 'target', 'scenario' ] ).reset_index( drop = True )

phase7_projection_by_region = pd.DataFrame( projection_region_rows )
baseline_region_lookup = ( 
    phase7_projection_by_region
    .loc[ phase7_projection_by_region[ 'scenario' ] == 'baseline_ref', [ 'target', 'region', 'mean_pred' ] ]
    .rename( columns = { 'mean_pred': 'baseline_mean_pred' } )
)
phase7_projection_by_region = phase7_projection_by_region.merge( baseline_region_lookup, on = [ 'target', 'region' ], how = 'left' )
phase7_projection_by_region[ 'delta_vs_baseline' ] = phase7_projection_by_region[ 'mean_pred' ] - phase7_projection_by_region[ 'baseline_mean_pred' ]
phase7_projection_by_region = phase7_projection_by_region.sort_values( [ 'target', 'region', 'scenario' ] ).reset_index( drop = True )

print( 'phase7 climate benchmark summary (all station-years):' )
display( phase7_projection_summary.round( 4 ) )

print( 'phase7 climate benchmark summary by region:' )
display( phase7_projection_by_region.round( 4 ) )



print( f'phase7 projected targets: { sorted( set( phase7_projected_targets ) ) }' )
if len( phase7_skipped_targets ) > 0:
    print( f'phase7 skipped targets (no scenario model available): { sorted( set( phase7_skipped_targets ) ) }' )

scenario_delta_plot = phase7_projection_summary.loc[ phase7_projection_summary[ 'scenario' ] != 'baseline_ref' ].copy( )

if len( scenario_delta_plot ) > 0:
    fig, axes = plt.subplots( 1, 2, figsize = ( 18, 6 ) )

    sns.barplot( 
        data = scenario_delta_plot,
        x = 'target',
        y = 'delta_vs_baseline',
        hue = 'scenario',
        ax = axes[ 0 ],
    )
    axes[ 0 ].axhline( 0, color = 'black', linewidth = 1.0, linestyle = '--' )
    axes[ 0 ].set_title( 'Phase 7 Scenario Response by Target' )
    axes[ 0 ].set_xlabel( 'Target' )
    axes[ 0 ].set_ylabel( 'Delta vs Baseline ( annual mean delta units )' )
    axes[ 0 ].tick_params( axis = 'x', rotation = 25 )

    region_focus = phase7_projection_by_region.loc[ 
        phase7_projection_by_region[ 'scenario' ] == 'plus1p5C_precip_same'
    ].copy( )

    if len( region_focus ) > 0:
        region_heat = region_focus.pivot( index = 'region', columns = 'target', values = 'delta_vs_baseline' )
        sns.heatmap( 
            region_heat,
            annot = True,
            fmt = '.3f',
            cmap = 'coolwarm',
            center = 0,
            ax = axes[ 1 ],
            cbar_kws = { 'label': 'Delta vs Baseline' },
        )
        axes[ 1 ].set_title( 'Regional Response Heatmap ( +1.5C, precip same )' )
        axes[ 1 ].set_xlabel( 'Target' )
        axes[ 1 ].set_ylabel( 'Region' )

    else:
        axes[ 1 ].axis( 'off' )

    plt.tight_layout( )
    plt.show( )

oxygen_region_plot = phase7_projection_by_region.loc[ 
    ( phase7_projection_by_region[ 'target' ] == 'delta_oxygen' )
    & ( phase7_projection_by_region[ 'scenario' ] != 'baseline_ref' )
].copy( )

if len( oxygen_region_plot ) > 0:
    plt.figure( figsize = ( 12, 5 ) )
    sns.barplot( 
        data = oxygen_region_plot,
        x = 'region',
        y = 'delta_vs_baseline',
        hue = 'scenario',
    )
    plt.axhline( 0, color = 'black', linewidth = 1.0, linestyle = '--' )
    plt.title( 'Delta Oxygen Scenario Shift by Region' )
    plt.xlabel( 'Region' )
    plt.ylabel( 'Delta vs Baseline' )
    plt.xticks( rotation = 35 )
    plt.tight_layout( )
    plt.show( )


### 7.2b - interpretation guide for climate benchmark tables
- use `delta_vs_baseline` as the scenario signal.
- positive `delta_vs_baseline` means the target annual delta is projected higher than baseline.
- negative `delta_vs_baseline` means projected suppression relative to baseline.
- this is a coarse annual-response emulator (first-draft, scenario-facing), not a daily forecast model.


## Phase 8 — Nutrient Models
Goal: predict changes in phosphates, nitrates, chlorophyll given water state

# also stage 3 (if time permits)

- 8.1 use the nutrient-inclusive dataset 
  - is okay that coverage is sparser and document that explicitly
- 8.2 same pipeline/process as phase 7, but input features now include water property predictions from earlier
- 8.3 report model skill honestly 
  - nutrient prediction will be noisier; even identifying dominant drivers is a valid result
  - so far, chlorophyll (despite being most obvious to people as blooms) seemed the weakest response?

In [ ]:
# code

## Phase 9 — Forward Projection
Goal: apply response functions to CMIP6 scenarios

- 9.1 obtain CMIP6 projected Δ air temp for your estuary regions under SSP2-4.5 and SSP5-8.5
  - ESGF portal or pre-downscaled products like NASA NEX-GDDP
  - also ote that there's some evidence that CMIP6 has under-estimated climate damage (by overestimating plant growth)
  - and the new estimate thinks it might be about 11% worse than predicted
  - might factor that in (with a *note?)
- 9.2 feed projected Δ air temp into stage 1 model 
  - get projected Δ water temp by decade
- 9.3 Feed projected Δ water temp into stage 2 response functions 
  - get projected Δ salinity, Δ oxygen, Δ pH
- 9.4 Identify projected regime-crossing dates per station under each scenario 
  - *"station X transitions from cool to warm regime between 2045–2060 under SSP5-8.5"* maybe?
- 9.5 propagate and report uncertainty 
  - at minimum, show scenario spread (SSP2 vs SSP5)
  - if time permits, model confidence intervals

In [ ]:
# code, though...